# Origins-of-life vent campaign

_Investigation `origins-of-life-vent` — coder reproduction notebook._

**Question.** Could the first cells have originated at alkaline hydrothermal vents? Can a physically plausible early-Earth vent–ocean interface generate and sustain, in order, the free energy, reduced carbon, self-amplifying chemistry, concentration, and compartments a protocell needs — without inserting metabolism, membranes, or heredity by assumption?

A staged, falsifiable Vivarium campaign testing the alkaline-vent theory for the origin of cells — treating a cellular origin as coupled transitions from geochemistry to bioenergetic autonomy, grounded in serpentinization-driven alkaline hydrothermal vents, with explicit competing scenarios.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


**Competing frameworks**

- **Classical alkaline-vent (Russell/Martin/Lane/Sojo)**
- **2026 unfavorable reassessment (Tutolo; Jackson)**
- **Shallow-vent (Barge & Price)**
- **Terrestrial geothermal (Mulkidjanian)**
- **Cyanosulfidic/photochemical (Sutherland)**


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-genesis/viva-genesis').is_dir():
    REPO = Path('/home/runner/work/viva-genesis/viva-genesis')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_genesis.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: The Serpentinizing Vent (`og-00-vent-reactor-baseline`)

**Question.** Can a Lost City-like, serpentinization-fed vent-ocean interface reproduce the physical and chemical environment in which the later origin-of-life steps are supposed to occur?

**Objective.** Build a reduced vent–ocean reactor and validate it against Lost City field ranges to establish a geochemical/spatial testbed.

**Hypothesis.** A 1D/2D porous-wall reactor with prescribed boundary fluxes reproduces published Lost City ranges (T, pH, H2, DIC) and qualitative precipitation trends while conserving mass and charge.

**Claim.** Across the swept olivine-rate x flow-velocity grid, about 80% of the operating region falls inside the combined Lost City window (40-91 C, pH 9.0-10.8, positive H2 flux), and the baseline settles comfortably inside it at 50.6 C / pH 9.37 with sustained H2, exact charge balance, and spatial-flux divergence within tolerance. This establishes a bounded geochemical habitat, a region rather than a single tuned point, and makes no origin-of-life claim by itself.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- A: serpentinization as boundary flux
- B: flow/mixing/precipitation, evolving porosity

**Proposed models / methods**

- PHREEQC or Reaktoro (speciation + mineral kinetics)
- FiPy finite-volume advection–diffusion–reaction
- prescribed vent source term

**Key assumptions**

- Hadean ocean chemistry parameterized separately — never reuse modern seawater as the ancient prior.
- Partial equilibrium: a prescribed olivine dissolution rate (flow/T-driven) feeds PHREEQC equilibrium speciation each step — fast aqueous/mineral equilibrium vs a rate-limited rock reaction, standard for reactive-transport.
- The heat-partition fraction (fraction of serpentinization enthalpy retained in the fluid parcel) is CALIBRATED to the observed 40-91 C vent temperature; the chemistry (H2, pH, pe, charge) is emergent from PHREEQC, not tuned.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Vent reactor baseline` | `viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor`**

A reusable alkaline-vent reactor module. Module A = genuine PHREEQC serpentinization (phreeqpython + vendored llnl.dat): olivine dissolution + serpentine/brucite/magnetite equilibrium gives dissolved H2 (reducing power), alkalinity, heat, pe, and a charge-balanced fluid. Module B = genuine FiPy finite-volume advection-diffusion-reaction column transporting those sources across the vent-ocean interface and setting outlet T/pH/porosity. PHREEQC conserves mass+charge and FiPy's FVM conserves flux, satisfying the og-00 contract invariants; the coupled model is driven by a real IPhreeqc speciation (needs phreeqpython) and a real FiPy FVM solve (needs fipy).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor = {
    'flow_velocity': 0.001,   # Boundary water flux through the reaction zone (m/s)
    'olivine_rate' : 0.15,   # Olivine dissolution rate (mol per flow-saturated reaction step)
    'nx'           : 40,   # FiPy finite-volume cells across the interface column
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `pressure` | float | pascal | [100000.0, 50000000.0] |
| `flow_velocity` | float | meter/second | >= 0.0 |
| `H2` | float | molar | >= 0.0 |
| `DIC` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `H2_flux` | float | mole/second | >= 0.0 |
| `alkalinity_flux` | float | mole/second |  |
| `heat_flux` | float | watt |  |
| `porosity` | float | dimensionless | [0.0, 1.0] |

**Conditions**

- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`
- `charge_neutrality` (invariant): `abs(net_charge) < tol`
- `spatial_flux_conservation` (invariant): `divergence(flux) == source_minus_sink`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-00-vent-reactor-baseline ===
STUDY = 'og-00-vent-reactor-baseline'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Vent_reactor_baseline = 100
INTERVAL_Vent_reactor_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor if 'ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor = materialize_composite('viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor['state']}, core=core)
            _comp.run(STEPS_Vent_reactor_baseline)
        print('ran', 'Vent reactor baseline', f'(viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor) for', STEPS_Vent_reactor_baseline, 'steps')
    except Exception as _e:
        print(f"Vent reactor baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**vent-cross-section**


In [ ]:
# vent-cross-section
show_viz(_render_one('local:VentCrossSection', {'composite': 'viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor', 'n_steps': 60, 'title': 'Real 2-D vent cross-section (FiPy)', 'max_frames': 20}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor if 'ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor' in dir() else None)))

**vent-reactor-timeseries**


In [ ]:
# vent-reactor-timeseries
show_viz(_render_one('local:VentReactorTimeseries', {'composite': 'viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor', 'n_steps': 120, 'title': 'Real vent reactor (PHREEQC + FiPy)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor if 'ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor' in dir() else None)))

**vent-serpentinization-engine**


In [ ]:
# vent-serpentinization-engine
show_viz(_render_one('local:VentSerpentinizationEngine', {'n_steps': 120, 'nx': 40, 'title': 'The serpentinization engine (Mechanism view)'}, RUNS_DB, STUDY_YAML))

**vent-habitability-map**


In [ ]:
# vent-habitability-map
show_viz(_render_one('local:VentHabitabilityMap', {'grid_json': 'workspace/studies/og-00-vent-reactor-baseline/artifacts/vent_habitability_grid.json', 'title': 'The Lost City habitability window (Decision view)'}, RUNS_DB, STUDY_YAML))

**vent-wall-3d**


In [ ]:
# vent-wall-3d
show_viz(_render_one('local:VentChimney3D', {'title': 'The vent wall in 3-D — the across-wall thermal gradient (World view)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| vent-temperature-in-lostcity-range _(primary)_ | kind=derived path=settled_temperature_c detail=last-quartile mean outlet temperature, degrees C | op range low 40.0 high 91.0 detail settled T in [40, 91] C |
| h2-flux-sustained _(primary)_ | kind=derived path=min_sustained_H2_flux detail=minimum H2 source flux over the back half of the run | op > value 1e-09 detail min sustained H2 flux > 0 |
| charge-conserved _(primary)_ | kind=derived path=max_abs_charge_imbalance detail=worst-case |charge imbalance| over the run (eq/kgw) | op <= value 1e-06 detail |charge imbalance| < 1e-6 throughout |
| spatial-flux-conserved _(primary)_ | kind=derived path=max_flux_divergence_residual detail=worst-case normalized FVM flux-divergence residual over the run | op <= value 0.001 detail max |flux divergence residual| < 1e-3 (direct LU solve ~ machine eps) |
| vent-pH-in-lostcity-range _(primary)_ | kind=derived path=settled_pH detail=last-quartile mean outlet pH | op range low 9.0 high 10.8 detail settled pH in [9.0, 10.8] |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Lost City ~40-91 C, pH ~9-10.8, >=30 ky activity | outlet T,pH; run duration | Kelley 2001 / Fruh-Green 2003 / Ludwig 2006 | Met by the real coupled model: settled T ~51 C and pH ~9.4 are both in the Lost City window, H2 reducing power is sustained, and mass/charge/flux conservation hold. pH was closed by reacting peridotite (olivine + pyroxene) so the Ca-OH endmember buffers the fluid to field pH — a real PHREEQC result. |


## Study: The First Carbon (`og-01-geology-to-reduced-carbon`)

**Question.** Can geological H2 drive CO2 reduction on vent mineral catalysts strongly enough to sustain a source of reduced organic carbon?

**Objective.** Add H2-driven CO2 reduction, reproduce the three benchmark experiments, and map the parameter region of sustained net organic-carbon flux.

**Hypothesis.** Over Fe/Ni/S surfaces and across the pH gradient, H2-driven CO2 reduction yields formate/acetate/pyruvate at lab-calibrated rates within a bounded (T,pH,H2,pressure) region; the Tutolo-2026 in-situ arm roughly halves the flux, and the feature-ablation controls (no gradient / no H2 / no catalyst) zero it.

**Claim.** Over Fe/Ni/S catalysts the model reproduces the Preiner-2020 yield hierarchy - formate about 200 mM, acetate about 100 uM, pyruvate about 10 uM - and sustains net organic-carbon flux inside a bounded region set by the pH gradient and dissolved H2, while the no-gradient, no-H2 and no-catalyst controls collapse it. A discriminating in-situ arm (Tutolo-2026) shows the usable proton-motive gradient falling from 427 to 43 mV, roughly halving the sustained flux. Because the CO2-reduction kinetics are lab-calibrated to Preiner rather than derived from first-principles surface chemistry, this establishes a computational bridge from serpentinization to an organic-carbon feedstock, not yet metabolism.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- C: pH-gradient electrochemistry
- D: mineral-catalyzed carbon fixation

**Proposed models / methods**

- Nernst–Planck–Poisson (electroneutral) on FiPy/FEniCSx
- Cantera/custom heterogeneous kinetics
- eQuilibrator feasibility
- PyMC Bayesian calibration to Preiner/Hudson/Altair

**Key assumptions**

- Encode pH(T,P,ionic_strength), never laboratory pH as a constant environmental state.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Geology to reduced carbon — favorable` | `viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation` | 0 | — |
| `Tutolo-2026 in-situ (unfavorable arm)` | `viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation` | 0 | gradient_efficiency=0.1 |

Declared parameter sets (`study.yaml` variants):

- **tutolo-2026** — `gradient_efficiency=0.1`
- **control-no-gradient** — `gradient_efficiency=1e-06`
- **control-no-h2** — `h2=1e-12`
- **control-no-catalyst** — `catalyst_active=1e-06`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation`**

A reusable H2/pH-driven carbon-fixation module. Module C = genuine FiPy finite-volume Nernst-Planck pH-gradient electrochemistry (pmf, current density; needs fipy). Module D = genuine eQuilibrator ΔG' feasibility gate (needs equilibrator_api) + lab-calibrated (Preiner/Hudson) mineral-catalyzed CO2 reduction to formate/acetate/pyruvate. gradient_efficiency=1.0 is the favorable arm; ~0.1 is the Tutolo-2026 in-situ collapse. catalyst_active/h2=0 give the controls.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation = {
    'gradient_efficiency': 1.0,   # Usable fraction of the geochemical ΔpH at in-situ T/P (1.0 favorable; ~0.1 Tutolo-2026 collapse; 0 = no-gradient control)
    'catalyst_active'    : 1.0,   # Active Fe/Ni/S mineral catalyst (0 = no-catalyst control)
    'h2'                 : 0.005,   # Dissolved H2 feed (M); ~0 = no-H2 control
    'co2'                : 0.002,   # Dissolved CO2 feed (M)
    'ph_vent'            : 9.5,   # Alkaline vent-side pH (from og-00)
    'ph_ocean'           : 6.0,   # Acidic early-ocean-side pH
    'dilution_rate'      : 0.0,   # Open-flow washout (1/s); 0 = closed batch (yield studies), >0 = open vent reactor → sustained-throughput regime (phase map)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `pH` | float | dimensionless | [0.0, 14.0] |
| `pressure` | float | pascal | [100000.0, 50000000.0] |
| `H2` | float | molar | >= 0.0 |
| `CO2` | float | molar | >= 0.0 |
| `ionic_strength` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `formate` | float | molar | >= 0.0 |
| `acetate` | float | molar | >= 0.0 |
| `pyruvate` | float | molar | >= 0.0 |
| `current_density` | float | ampere/meter**2 |  |
| `electric_potential` | float | volt |  |
| `net_carbon_flux` | float | mole/second |  |

**Conditions**

- `charge_neutrality` (invariant): `abs(net_charge) < tol`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`
- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-01-geology-to-reduced-carbon ===
STUDY = 'og-01-geology-to-reduced-carbon'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Geology_to_reduced_carbon_favorable = 100
INTERVAL_Geology_to_reduced_carbon_favorable = 1.0
STEPS_Tutolo_2026_in_situ_unfavorable_arm = 100
INTERVAL_Tutolo_2026_in_situ_unfavorable_arm = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation if 'ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation = materialize_composite('viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation['state']}, core=core)
            _comp.run(STEPS_Geology_to_reduced_carbon_favorable)
        print('ran', 'Geology to reduced carbon — favorable', f'(viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation) for', STEPS_Geology_to_reduced_carbon_favorable, 'steps')
    except Exception as _e:
        print(f"Geology to reduced carbon — favorable not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation if 'ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation = materialize_composite('viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation['state']}, core=core)
            _comp.run(STEPS_Tutolo_2026_in_situ_unfavorable_arm)
        print('ran', 'Tutolo-2026 in-situ (unfavorable arm)', f'(viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation) for', STEPS_Tutolo_2026_in_situ_unfavorable_arm, 'steps')
    except Exception as _e:
        print(f"Tutolo-2026 in-situ (unfavorable arm) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**proton-gradient-profile**


In [ ]:
# proton-gradient-profile
show_viz(_render_one('local:ProtonGradientProfile', {'composite': 'viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation', 'n_steps': 80, 'title': 'Proton gradient → reduced carbon'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation if 'ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation' in dir() else None)))

**carbon-fixation-timeseries**


In [ ]:
# carbon-fixation-timeseries
show_viz(_render_one('local:CarbonFixationViz', {'composite': 'viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation', 'n_steps': 120, 'title': 'Geology to reduced carbon (eQuilibrator + FiPy)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation if 'ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation' in dir() else None)))

**carbon-fixation-phase-map**


In [ ]:
# carbon-fixation-phase-map
show_viz(_render_one('local:CarbonFixationPhaseMap', {'title': 'The carbon-fixation landscape (Decision view)', 'grid_json': 'workspace/studies/og-01-geology-to-reduced-carbon/artifacts/carbon_fixation_phase_grid.json'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| formate-yield-reproduces-preiner _(primary)_ | kind=derived path=settled_formate_mM detail=batch formate yield mM=None | op range low 50.0 high 220.0 detail formate in Preiner range (~200 mM ceiling) |
| acetate-yield-reproduces-preiner _(primary)_ | kind=derived path=settled_acetate_uM detail=batch acetate yield uM=None | op range low 20.0 high 150.0 detail acetate in Preiner range |
| pyruvate-yield-reproduces-preiner _(primary)_ | kind=derived path=settled_pyruvate_uM detail=batch pyruvate yield uM=None | op range low 2.0 high 15.0 detail pyruvate in Preiner range |
| net-carbon-flux-nonzero-region _(primary)_ | kind=derived path=peak_net_carbon_flux detail=peak organic-C fixation rate | op > value 1e-06 detail peak net carbon flux > 0 (viable region) |
| free-energy-directionality-logged _(primary)_ | kind=derived path=free_energy_logged_fraction detail=fraction of steps with logged ΔG' | op >= value 1.0 detail ΔG' logged every step |
| chemiosmotic-backpressure-loads-gradient _(primary)_ | kind=derived path=pmf_backpressure_sag_frac detail=peak fractional pmf sag while fixation draws the proton reservoir down | op range low 0.05 high 0.95 detail a meaningful back-pressure sag (gradient is spent) without total collapse |
| nernst-planck-flux-conserved _(diagnostic)_ | kind=derived path=max_abs_electro_flux_residual detail=max |FVM residual| | op <= value 0.01 detail max |flux divergence residual| small |
| ph-gradient-drives-formate _(primary)_ | kind=cross_run detail=favorable formate vs no-gradient / no-H2 / no-catalyst | op controls_absent detail formate present with gradient <1 mM in every control None |
| in-situ-ph-correction _(diagnostic)_ | kind=cross_run detail=favorable pmf vs Tutolo-2026 pmf | op arms_reported detail report both arms; flag if the Tutolo pmf collapses |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Formate up to ~200 mM, acetate ~100 uM, pyruvate ~10 uM at 100 C | product concentrations by mineral/T/P | Preiner 2020 | Met by the favorable arm — batch formate ~200 mM, acetate ~100 uM, pyruvate ~10 uM (Preiner hierarchy), via real eQuilibrator-gated, lab-calibrated mineral kinetics. |
| CO2+H2->formate across pH gradient at room T, ~1.5 bar | formate yield vs delta-pH + controls | Hudson 2020 | Met — formate is present with the pH-gradient proton-motive force and collapses to <1 mM in every control (no gradient / no H2 / no catalyst). |


## Study: The First Metabolism (`og-02-reduced-carbon-to-protometabolism`)

**Question.** Under continuous feed and washout, does a curated prebiotic reaction network persist and self-amplify as a connected system, rather than merely producing isolated organic molecules?

**Objective.** Insert an experimentally-motivated reaction graph and test whether metabolism-like subnetworks emerge under flow.

**Hypothesis.** A curated graph (formate/acetate/pyruvate/glyoxylate + rTCA fragments) seeded with Muchowska/Moran edges sustains nonzero throughput and at least one autocatalytic motif within a feasible resource budget.

**Claim.** Under continuous feed and washout, a balanced, experimentally-constrained minimal reaction network either sustains net carbon throughput and genuine autocatalytic amplification through stoichiometrically valid reactions that survive washout, or it does not — in which case the earlier lumped-network amplification (gain peaking near 29.5) is shown to be an artifact of an element- unbalanced reaction rather than endogenous emergence. Curated persistence is no longer claimed as emergence; the corrected-curated, measured-nonenzymatic, and rule-generated arms are compared on equal footing, each held to mass/charge balance.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- E: protometabolic network formation
- D: surface chemistry

**Proposed models / methods**

- SBML + libRoadRunner/Tellurium or COPASI
- SciPy solve_ivp
- element-balance + thermodynamic-consistency checks before kinetic fitting
- GillesPy2 for low-copy stochastic regimes

**Key assumptions**

- Rate laws/mineral dependencies carry provenance; autocatalysis is an emergent observable, not an assumption.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Reduced carbon to protometabolism (balanced rebuild)` | `viva_genesis.composites.open_chemical_network.open_chemical_network` | 0 | model_proto_metabolism=curated_rtca |

Declared parameter sets (`study.yaml` variants):

- **control-no-co2** — `feed_CO2=1e-12`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.open_chemical_network.open_chemical_network`**

og-02's continuous-feed reactor rebuilt as the Phase-3 open chemical network: the reaction network is a model CLASS on the proto_metabolism face (model_proto_metabolism: curated_rtca | reaction_universe | null_network), all three integrated by the genuine libRoadRunner engine under the same Phase-1-posterior feed (CO2, H2, pyruvate), graded by one class-agnostic post-hoc detector (flux-carrying simple cycles with stoichiometric gain > 1) and audited by a NetworkLedgerProcess (exact element residual, integrated carbon balance, dissipation >= 0). The universe class is generated from Wolos-2020-style rules over linear C/H/O skeletons with eQuilibrator dG' (vg-data package reaction_universe); the null class is its degree-preserving, carbon-matched rewiring.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_open_chemical_network_open_chemical_network = {
    'model_proto_metabolism': 'reaction_universe',   # Model class behind the 'proto_metabolism' face: curated_rtca | null_network | reaction_universe
    'feed_CO2'              : 0.0014,   # Vent-buffered CO2 set-point (M) [vg-data: calibrated, Phase-1 posterior]
    'feed_H2'               : 0.0108,   # Vent-buffered H2 set-point (M) [vg-data: calibrated, Phase-1 posterior]
    'feed_pyruvate'         : 0.0001,   # Pyruvate CSTR inflow (M) from og-01 reduced carbon [vg-data: inferred]
    'feed_formaldehyde'     : 0.0,   # Formaldehyde CSTR inflow (M); 0 canonical, 1e-3 in the formose arm [vg-data: hypothetical]
    'k_wash'                : 0.05,   # CSTR washout / dilution rate (1/s) [vg-data: inferred]
    'null_seed'             : 0,   # Null-network draw (null_network class only)
    'max_carbons'           : 4,   # Carbon cap of the generated universe (reaction_universe / null_network) [vg-data: hypothetical]
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `formate` | float | molar | >= 0.0 |
| `acetate` | float | molar | >= 0.0 |
| `pyruvate` | float | molar | >= 0.0 |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `pH` | float | dimensionless | [0.0, 14.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `net_carbon_flux` | float | mole/second |  |

**Conditions**

- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-02-reduced-carbon-to-protometabolism ===
STUDY = 'og-02-reduced-carbon-to-protometabolism'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Reduced_carbon_to_protometabolism_balanced_rebuild = 100
INTERVAL_Reduced_carbon_to_protometabolism_balanced_rebuild = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_open_chemical_network_open_chemical_network if 'ov_viva_genesis_composites_open_chemical_network_open_chemical_network' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_open_chemical_network_open_chemical_network = materialize_composite('viva_genesis.composites.open_chemical_network.open_chemical_network', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_open_chemical_network_open_chemical_network['state']}, core=core)
            _comp.run(STEPS_Reduced_carbon_to_protometabolism_balanced_rebuild)
        print('ran', 'Reduced carbon to protometabolism (balanced rebuild)', f'(viva_genesis.composites.open_chemical_network.open_chemical_network) for', STEPS_Reduced_carbon_to_protometabolism_balanced_rebuild, 'steps')
    except Exception as _e:
        print(f"Reduced carbon to protometabolism (balanced rebuild) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**protometabolic-network**


In [ ]:
# protometabolic-network
show_viz(_render_one('local:ProtometabolicNetwork', {'composite': 'viva_genesis.composites.protometabolism.protometabolic_network', 'n_steps': 60, 'title': 'The protometabolic network igniting'}, RUNS_DB, STUDY_YAML))

**protometabolism-network**


In [ ]:
# protometabolism-network
show_viz(_render_one('local:ProtometabolismViz', {'composite': 'viva_genesis.composites.protometabolism.protometabolic_network', 'n_steps': 120, 'title': 'Protometabolic network (libRoadRunner)'}, RUNS_DB, STUDY_YAML))

**protometabolism-gain-phase-map**


In [ ]:
# protometabolism-gain-phase-map
show_viz(_render_one('local:ProtometabolismGainPhaseMap', {'title': 'The protometabolic ignition landscape (Decision view)', 'grid_json': 'workspace/studies/og-02-reduced-carbon-to-protometabolism/artifacts/protometabolism_gain_grid.json'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| sustained-carbon-throughput _(primary)_ | kind=derived path=settled_net_carbon_flux detail=steady net carbon fixation rate | op > value 1e-06 detail net throughput > 0 (no imposed accumulation) |
| autocatalytic-gain-present _(primary)_ | kind=derived path=peak_autocatalytic_gain detail=peak OAA self-amplification ratio | op >= value 1.0 detail >= 1 reinforcing/autocatalytic motif |
| network-closure-vs-oneoff _(supporting)_ | kind=derived path=network_connected_final detail=networkx connectivity of the species-reaction graph | op >= value 1.0 detail connected network (not one-offs) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Fe2+ promotes 9/11 TCA intermediates + 5 universal precursors from pyruvate/glyoxylate | edge yields + network reachability | Muchowska 2019 | Represented as the lumped reductive-TCA turn (the curated rTCA class on libRoadRunner), graded; edge-by-edge 9/11 reachability is the reaction_universe comparison, not yet run |


## Study: The First Individual (`og-03-dilution-to-localized-individuals`)

**Question.** Can vent pore physics concentrate a dilute vent feed strongly enough to form a persistent, localized individual rather than washing everything into the ocean?

**Objective.** Add pore geometry, adsorption, convection and thermophoresis; find where retention + concentration cross persistence thresholds.

**Hypothesis.** Thermophoresis + convection + adsorption in evolving pores raise local concentration enough that synthesis outruns hydrolysis/leakage in a bounded geometry region; elsewhere washout dominates.

**Claim.** Resolving diffusion, thermophoresis, adsorption, surface catalysis and advection for every actual reactant, a bounded-geometry pore either increases useful product flux or network persistence at identical feeds relative to a well-mixed control, or it does not. The defensible result is the existence and location of a localization regime and its effect on the network — not a precise single-solute extreme-enrichment magnitude, which is solver-sensitive and precursor-dependent.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- F: transport/concentration/retention
- B: evolving pores

**Proposed models / methods**

- FiPy finite-volume with Soret term
- OpenPNM/PoreSpy pore geometry
- reproduce Baaske enrichment
- Damkohler/Peclet/Soret scaling

**Key assumptions**

- Concentration arises from the balance of advection, diffusion, thermodiffusion, adsorption, precipitation and degradation — not assumed.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Dilution to localized individuals` | `viva_genesis.composites.thermophoretic_concentration.thermophoretic_concentration` | 0 | flow_velocity=0.0001, porosity=0.3, pore_length=0.005, delta_T=30.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.thermophoretic_concentration.thermophoretic_concentration`**

A reusable thermophoretic-concentration module for the Baaske thermal trap. Module F = genuine FiPy finite-volume accumulation column (needs fipy): thermophoresis (Soret) + buoyant convection concentrate a dilute feed, enrichment exponential in pore length (>10^8 favorable, ~1 in controls). flow_velocity=0 / porosity=1 give the no-trap controls.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration = {
    'flow_velocity': 0.0001,   # Buoyant convective flow in the pore (m/s); 0 = no-convection control
    'porosity'     : 0.3,   # Pore porosity/openness (0-1); ~1 = open-pore (wash-through) control
    'temperature'  : 333.15,   # Mean pore temperature (K)
    'pore_length'  : 0.005,   # Pore length (m); longer = more enrichment
    'delta_T'      : 30.0,   # Cross-pore temperature difference (K)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `flow_velocity` | float | meter/second | >= 0.0 |
| `porosity` | float | dimensionless | [0.0, 1.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `enrichment_factor` | float | dimensionless | >= 1.0 |
| `residence_time` | float | second | >= 0.0 |

**Conditions**

- `spatial_flux_conservation` (invariant): `divergence(flux) == source_minus_sink`
- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-03-dilution-to-localized-individuals ===
STUDY = 'og-03-dilution-to-localized-individuals'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Dilution_to_localized_individuals = 100
INTERVAL_Dilution_to_localized_individuals = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration if 'ov_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration = materialize_composite('viva_genesis.composites.thermophoretic_concentration.thermophoretic_concentration', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration['state']}, core=core)
            _comp.run(STEPS_Dilution_to_localized_individuals)
        print('ran', 'Dilution to localized individuals', f'(viva_genesis.composites.thermophoretic_concentration.thermophoretic_concentration) for', STEPS_Dilution_to_localized_individuals, 'steps')
    except Exception as _e:
        print(f"Dilution to localized individuals not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**thermophoretic-trap**


In [ ]:
# thermophoretic-trap
show_viz(_render_one('local:ThermophoreticTrapViz', {'composite': 'viva_genesis.composites.thermophoretic_concentration.thermophoretic_concentration', 'title': 'The thermal trap concentrates an individual'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration if 'ov_viva_genesis_composites_thermophoretic_concentration_thermophoretic_concentration' in dir() else None)))

**pore-ignition-coupling**


In [ ]:
# pore-ignition-coupling
show_viz(_render_one('local:PoreIgnitionCoupling', {'title': 'Localization lifts the feed across the ignition threshold (Mechanism view)', 'grid_json': 'workspace/studies/og-02-reduced-carbon-to-protometabolism/artifacts/protometabolism_gain_grid.json'}, RUNS_DB, STUDY_YAML))

**pore-localization-map**


In [ ]:
# pore-localization-map
show_viz(_render_one('local:PoreLocalizationMap', {'grid_json': 'workspace/studies/og-03-dilution-to-localized-individuals/artifacts/pore_localization_grid.json', 'title': 'The pore-localization frontier (Decision view)'}, RUNS_DB, STUDY_YAML))

**pore-enrichment-solver-sensitivity**


In [ ]:
# pore-enrichment-solver-sensitivity
show_viz(_render_one('local:PoreEnrichmentSolverSensitivity', {'comparison_json': 'workspace/references/spatial-transport-solver-comparison.json', 'title': 'Enrichment is bounded: reduced column vs resolved FEM (Decision view)'}, RUNS_DB, STUDY_YAML))

**multisolute-colocalization-map**


In [ ]:
# multisolute-colocalization-map
show_viz(_render_one('local:PoreColocalizationMap', {'grid_json': 'workspace/studies/og-03-dilution-to-localized-individuals/artifacts/colocalization_grid.json', 'title': 'Does the prebiotic mixture co-localize? (Decision view)'}, RUNS_DB, STUDY_YAML))

**pore-accumulation-3d**


In [ ]:
# pore-accumulation-3d
show_viz(_render_one('local:ThermophoreticTrap3D', {'title': 'Inside the pore — the accumulation column in 3-D (World view)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| thermophoretic-enrichment-reproduces-baaske _(primary)_ | kind=derived path=settled_accumulation_peclet detail=ln(enrichment) = beta × pore length (reduced column) | op arms_reported detail reduced-column Pe_acc ≈ 18.4 (~10^8-fold) reported as the closed-cell/reduced-model figure; the resolved FEM does not confirm it past the convergence-verified Pe_acc ≈ 13 |
| enrichment-resolved-within-convergence _(diagnostic)_ | kind=derived path=log10_enrichment detail=localization occurs within the resolvable range (positive enrichment agreement region)=None | op > value 0.0 detail log10_enrichment > 0 (the resolved trap concentrates above inflow within Pe_acc ≲ 13) |
| retention-beats-destruction _(primary)_ | kind=derived path=log10_enrichment detail=log10 of the settled enrichment fold (reduced column; resolved value within Pe_acc ≲ 13) | op arms_reported detail reduced-column ≥10^6-fold reported as the closed-cell figure; net accumulation within demonstrated convergence is the gated claim (enrichment-resolved-within-convergence) |
| open-pore-control-does-not-concentrate _(primary)_ | kind=cross_run detail=favorable enrichment vs open-pore control | op controls_absent detail enrichment collapses to ~1 in the open-pore control |
| enrichment-exponential-in-pore-length _(diagnostic)_ | kind=cross_run detail=favorable vs 10× shorter pore | op arms_reported detail short pore enriches ≥100× less than the favorable pore |
| spatial-flux-conserved _(diagnostic)_ | kind=derived path=max_abs_trap_flux_residual detail=max |FVM flux-divergence residual| | op <= value 0.01 detail FVM residual small (direct LU solve) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| >10^8-fold nucleotide accumulation in favorable modeled pore geometries | enrichment vs geometry | Baaske 2007 | Partially met by the real FiPy thermophoretic trap — the favourable pore reaches the extreme-accumulation regime (accumulation Péclet 18.4) and the native (porosity × pore length) phase map localizes at ~17% of the grid with open/short-pore controls not concentrating. The exact Baaske fold-magnitude (reduced-model ~1e6–1e7) is solver-sensitive above Pe ≈ 13 (FiPy↔FEniCS diverge by up to ~5.6 decades there), so localization is confirmed but the precise fold is not. |


## Study: The First Boundary (`og-04-mineral-compartments-to-protocells`)

**Question.** Do mixed-amphiphile membranes improve retention while staying permeable enough to keep access to the vent-supplied gradient and feedstocks?

**Objective.** Introduce mixed amphiphiles + membrane assembly; measure whether organic boundaries enhance persistence without isolating the system.

**Hypothesis.** Over a salt/pH/divalent-cation/T window, vesicles form and selectively retain while remaining permeable enough to exploit the external gradient — a non-empty individuality-vs-access region exists.

**Claim.** In a bounded alkaline / seawater-salt window mixed single-chain amphiphiles form viable vesicles (about 8000) where single fatty acids form none, and those compartments retain internal chemistry (retention 0.91) while staying permeable to the external gradient (access 1.0), a retention-without-isolation product near 0.90, with the feasibility map marking a viable region at 63/256 points (about 25%). This provides a plausible transition from mineral localization to an organic boundary. The membrane model is coarse-grained and calibrated to the Jordan-2019 vesicle experiments rather than membrane MD, and whether enclosing a reaction network actually improves its persistence over the open pore is left to downstream studies.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- G: amphiphiles, mineral pores and protocells

**Proposed models / methods**

- custom coarse-grained membrane kinetics
- Smoldyn/ReaDDy spatial stochastic
- OpenMM/LAMMPS to calibrate permeability laws

**Key assumptions**

- Mineral pores provide initial individuality; organic boundaries are a later, condition-dependent transition.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Mineral compartments to protocells` | `viva_genesis.composites.mixed_amphiphile_protocell.mixed_amphiphile_protocell` | 0 | mix_fraction=0.4, pH=9.0, ionic_strength=0.35, temperature=318.15 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.mixed_amphiphile_protocell.mixed_amphiphile_protocell`**

A reusable mixed-amphiphile protocell module (Module G). Coarse-grained vesicle self-assembly + permeability calibrated to Jordan 2019: mixed single-chain amphiphiles broaden the pH / salt / divalent stability window and form vesicles that retain chemistry while staying permeable to the gradient (individuality without isolation). mix_fraction≈0 / high salt / acidic pH give the controls. Coarse-grained reproduction (not wrapped MD).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell = {
    'mix_fraction'  : 0.4,   # Co-surfactant fraction (0 = single fatty acid; higher = broader Jordan window)
    'pH'            : 9.0,   # Environment pH (alkaline vent)
    'ionic_strength': 0.35,   # Ionic strength (M); ~seawater 0.35
    'temperature'   : 318.15,   # Temperature (K)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `pH` | float | dimensionless | [0.0, 14.0] |
| `ionic_strength` | float | molar | >= 0.0 |
| `temperature` | float | kelvin | [273.0, 673.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `membrane_area` | float | meter**2 | >= 0.0 |
| `permeability` | float | meter/second | >= 0.0 |
| `compartment_volume` | float | liter | >= 0.0 |
| `viable_compartments` | integer | dimensionless | >= 0 |

**Conditions**

- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`
- `spatial_flux_conservation` (invariant): `divergence(flux) == source_minus_sink`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-04-mineral-compartments-to-protocells ===
STUDY = 'og-04-mineral-compartments-to-protocells'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Mineral_compartments_to_protocells = 100
INTERVAL_Mineral_compartments_to_protocells = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell if 'ov_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell = materialize_composite('viva_genesis.composites.mixed_amphiphile_protocell.mixed_amphiphile_protocell', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell['state']}, core=core)
            _comp.run(STEPS_Mineral_compartments_to_protocells)
        print('ran', 'Mineral compartments to protocells', f'(viva_genesis.composites.mixed_amphiphile_protocell.mixed_amphiphile_protocell) for', STEPS_Mineral_compartments_to_protocells, 'steps')
    except Exception as _e:
        print(f"Mineral compartments to protocells not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**protocell-boundary-world**


In [ ]:
# protocell-boundary-world
show_viz(_render_one('local:ProtocellBoundaryWorld', {'mix_fraction': 0.4, 'ionic_strength': 0.35, 'pH': 9.0, 'title': 'The compartment and the gradient (World view)'}, RUNS_DB, STUDY_YAML))

**protocell-window**


In [ ]:
# protocell-window
show_viz(_render_one('local:ProtocellWindowViz', {'composite': 'viva_genesis.composites.mixed_amphiphile_protocell.mixed_amphiphile_protocell', 'title': 'The protocell window broadens — mixed amphiphiles (Mechanism view)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell if 'ov_viva_genesis_composites_mixed_amphiphile_protocell_mixed_amphiphile_protocell' in dir() else None)))

**protocell-retention-access**


In [ ]:
# protocell-retention-access
show_viz(_render_one('local:ProtocellPhaseMap', {'grid_json': 'workspace/studies/og-04-mineral-compartments-to-protocells/artifacts/protocell_grid.json', 'title': 'A boundary that retains without isolating (Decision view)'}, RUNS_DB, STUDY_YAML))

**protocell-vesicle-3d**


In [ ]:
# protocell-vesicle-3d
show_viz(_render_one('local:ProtocellVesicle3D', {'title': 'The protocell in 3-D — a boundary that retains without isolating (World view)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| vesicles-form-in-window _(primary)_ | kind=derived path=settled_viable_compartments detail=viable vesicle count in the favorable arm | op > value 0.0 detail vesicles form (>0) in the mixed-amphiphile window |
| retention-without-isolation _(primary)_ | kind=derived path=settled_individuality_access detail=retention × access | op >= value 0.3 detail both retention and access nonzero (a non-empty individuality-vs-access region) |
| mixed-amphiphile-advantage-over-single _(primary)_ | kind=cross_run detail=favorable (mixed) viable vs single-fatty-acid control | op controls_absent detail mixed forms vesicles; single fatty acid does not at the same salt |
| membrane-transition-robust _(diagnostic)_ | kind=cross_run detail=favorable vs extreme-salt and acidic controls | op arms_reported detail extreme-salt and acidic controls abolish vesicles (survival region is bounded) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Mixed single-chain amphiphiles form vesicles across broader alkaline pH/salinity/divalent/T than single-fatty-acid models | vesicle stability map | Jordan 2019 | Met by the coarse-grained Jordan-calibrated protocell — mixed amphiphiles form ~8000 viable vesicles at alkaline vent pH and seawater salt where a single fatty acid forms none, and the (co-surfactant × ionic strength) feasibility map marks a viable region (individuality > 0.1) at 63/256 points. |


## Study: Owning Its Energy (`og-05-environmental-to-internal-bioenergetics`)

**Question.** Can an evolving compartment reduce its dependence on an externally imposed proton gradient by internalizing electrochemical work, and does that pay for itself?

**Objective.** Allow catalytic ion transport and active pumping; test whether selection favors internalized gradient generation.

**Hypothesis.** When membrane permeability, Na+/H+ coupling and catalyst efficiency vary under selection, systems with increasing internal gradient generation win on throughput + persistence — but only if membranes stay sufficiently leaky (Sojo constraint).

**Claim.** Presented as a conditional selection/cost model, internalizing energy handling pays only where the specified chemistry confers positive net useful work after all synthesis, leak and maintenance costs; the three regimes (direct gradient use / chemical carrier / active pumping) are distinguished and compared rather than collapsed into one inserted pump. Where no specified chemistry clears the cost, internalization does not pay and is not claimed.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- H: bioenergetic internalization

**Proposed models / methods**

- evolutionary control loop over membrane parameters
- reward = throughput + persistence
- Sojo leaky-membrane model as baseline

**Key assumptions**

- Internalization is an evolutionary control problem, not a scripted upgrade.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Environmental to internal bioenergetics` | `viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization` | 0 | catalyst_efficiency=1.0, v_start=0.4, decline_rate=0.01 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization`**

A reusable bioenergetic-internalization module (Module H). An evolutionary control loop (replicator dynamics, Sojo-2014 leaky-membrane fitness) under a declining external gradient: as the geochemical pmf collapses, an efficient-pump population internalizes gradient generation and persists, while a weak-pump control collapses (the Sojo constraint). catalyst_efficiency sets the arm. Coarse-grained, deterministic reproduction (not wrapped ALife).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization = {
    'catalyst_efficiency': 1.0,   # Active-pump (Na+/H+) efficiency (1.0 efficient; ~0.3 weak-pump control)
    'v_start'            : 0.4,   # Initial external gradient (V)
    'decline_rate'       : 0.01,   # External-gradient decline per step (V/s)
    'gradient_floor'     : 0.02,   # Floor the external gradient decays to (V); low floor forces internalization-or-die
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `electric_potential` | float | volt |  |
| `catalyst_efficiency` | float | dimensionless | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `lineage_fitness` | float | dimensionless |  |
| `free_energy_flux` | float | watt |  |
| `internalization` | float | dimensionless | [0.0, 1.0] |
| `survivors` | float | dimensionless | [0.0, 1.0] |
| `mean_permeability` | float | meter/second | >= 0.0 |

**Conditions**

- `energy_directionality` (invariant): `free_energy_flux_logged == true`
- `charge_neutrality` (invariant): `abs(net_charge) < tol`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-05-environmental-to-internal-bioenergetics ===
STUDY = 'og-05-environmental-to-internal-bioenergetics'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Environmental_to_internal_bioenergetics = 100
INTERVAL_Environmental_to_internal_bioenergetics = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization if 'ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization = materialize_composite('viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization['state']}, core=core)
            _comp.run(STEPS_Environmental_to_internal_bioenergetics)
        print('ran', 'Environmental to internal bioenergetics', f'(viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization) for', STEPS_Environmental_to_internal_bioenergetics, 'steps')
    except Exception as _e:
        print(f"Environmental to internal bioenergetics not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**bioenergetic-internalization**


In [ ]:
# bioenergetic-internalization
show_viz(_render_one('local:BioenergeticInternalizationViz', {'composite': 'viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization', 'title': 'Selection internalizes the gradient'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization if 'ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization' in dir() else None)))

**bioenergetic-provenance**


In [ ]:
# bioenergetic-provenance
show_viz(_render_one('local:BioenergeticProvenanceViz', {'title': 'Energy provenance — external to internal (Mechanism view)'}, RUNS_DB, STUDY_YAML))

**bioenergetic-survival-map**


In [ ]:
# bioenergetic-survival-map
show_viz(_render_one('local:BioenergeticSurvivalMap', {'grid_json': 'workspace/studies/og-05-environmental-to-internal-bioenergetics/artifacts/bioenergetic_survival_grid.json', 'title': 'Survival off the collapsing gradient (Decision view)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| selection-internalizes-gradient-generation _(primary)_ | kind=derived path=final_internalization detail=active-pump fraction of the surviving population at the collapsed gradient | op >= value 0.6 detail the population internalizes (pump fraction >= 0.6) as the external gradient falls |
| reduced-gradient-dependence _(primary)_ | kind=derived path=final_survivors detail=surviving strategy mass at the collapsed external gradient | op >= value 0.5 detail internalizing lineage persists (survivors >= 0.5) at the collapsed gradient |
| sojo-constraint-weak-pump-collapses _(primary)_ | kind=cross_run detail=efficient-pump survivors vs weak-pump control | op controls_absent detail the weak-pump control collapses (survivors ~0) where the efficient pump persists |
| energy-cost-accounted _(supporting)_ | kind=derived path=final_internalization detail=internalized population has net-positive flux | op > value 0.0 detail a net-positive-flux internalized region exists |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| LUCA-like cells can use proton gradients only if membranes remain sufficiently leaky | permeability vs viability | Sojo 2014 / Lane 2012 | Met by the real replicator selection loop — the survival Decision map over (pump efficiency × gradient floor) is bounded by the Sojo cost line (η ≈ 0.30): efficient pumps internalize and persist (~94% of the grid survives) while weak pumps collapse at the gradient transition. |


## Study: The First Heredity (`og-06-chemistry-to-heredity`)

**Question.** Does heredity improve lineage persistence enough to pay for itself, and must that require assuming a long RNA-like template?

**Objective.** Introduce competing heredity modules and test whether inherited state improves persistence under feasible budgets.

**Hypothesis.** At least one heredity module raises lineage persistence + information retention above its non-heritable control under a feasible resource budget; the winner is not assumed a priori.

**Claim.** Separating chemical memory, inherited network composition, differential reproduction and template copying, heredity is demonstrated only where a chemical difference actually inherited by a daughter causes reproducible differences in survival or reproduction. Heredity need not presume a long high-fidelity RNA template, but each mechanism is judged on an inherited chemical difference with a measured consequence, not on a provided inheritance rule.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- J: polymers, information and heredity (as competing modules)

**Proposed models / methods**

- GillesPy2/StochPy SSA
- rule-based chemistry (BioNetGen/NFsim) for sequence combinatorics
- separate catalyst discovery from inheritance
- explicit fidelity/cost/competition

**Key assumptions**

- Treat informational chemistry as a branch point; do not presume a viable RNA replication mechanism.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Chemistry to heredity` | `viva_genesis.composites.heredity_module_comparison.heredity_module_comparison` | 0 | module=rna-like, temperature=353.15 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.heredity_module_comparison.heredity_module_comparison`**

A reusable heredity-module comparison (Module J). A genuine libRoadRunner/Tellurium SBML ODE integration (needs tellurium) of a quasispecies replicator (Eigen error threshold): a heredity module (RNA-like / peptide / low-fidelity / non-heritable) replicates a polymer with whole-sequence fidelity Q = fidelity^length; above the threshold the functional information is retained and inheritance pays, below it errors win. Compares modules head-to-head on lineage fitness.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison = {
    'module'     : 'rna-like',   # Heredity-module preset: rna-like | peptide | low-fidelity | non-heritable
    'temperature': 353.15,   # Temperature (K; ~80 C vent)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `polymer_length` | integer | dimensionless | >= 1 |
| `lineage_fitness` | float | dimensionless |  |
| `information_retention` | float | dimensionless | [0.0, 1.0] |
| `functional_templates` | float | dimensionless | >= 0.0 |

**Conditions**

- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-06-chemistry-to-heredity ===
STUDY = 'og-06-chemistry-to-heredity'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Chemistry_to_heredity = 100
INTERVAL_Chemistry_to_heredity = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison if 'ov_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison = materialize_composite('viva_genesis.composites.heredity_module_comparison.heredity_module_comparison', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison['state']}, core=core)
            _comp.run(STEPS_Chemistry_to_heredity)
        print('ran', 'Chemistry to heredity', f'(viva_genesis.composites.heredity_module_comparison.heredity_module_comparison) for', STEPS_Chemistry_to_heredity, 'steps')
    except Exception as _e:
        print(f"Chemistry to heredity not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**heredity-lineage-tree**


In [ ]:
# heredity-lineage-tree
show_viz(_render_one('local:HeredityLineageTree', {'composite': 'viva_genesis.composites.heredity_lineage.heredity_lineage_comparison', 'grid_json': 'workspace/studies/og-06-chemistry-to-heredity/artifacts/heredity_error_threshold_grid.json', 'title': 'Heredity as an inherited lineage'}, RUNS_DB, STUDY_YAML))

**heredity-error-threshold**


In [ ]:
# heredity-error-threshold
show_viz(_render_one('local:HeredityErrorThresholdViz', {'composite': 'viva_genesis.composites.heredity_module_comparison.heredity_module_comparison', 'title': 'Heredity above the error threshold'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison if 'ov_viva_genesis_composites_heredity_module_comparison_heredity_module_comparison' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| inheritance-beats-noninheritance _(primary)_ | kind=derived path=final_lineage_fitness detail=RNA-like lineage fitness at steady state | op > value 0.1 detail inheriting lineage fitness net-positive (and beats the non-heritable control) |
| information-retained-above-error-threshold _(primary)_ | kind=derived path=final_information_retention detail=C/(C+E) retention at steady state | op >= value 0.4 detail functional information retained (above the error threshold) |
| heredity-modules-compared _(primary)_ | kind=cross_run detail=RNA-like vs peptide lineage fitness | op arms_reported detail both modules viable; RNA-like ranks above peptide |
| error-catastrophe-below-threshold _(diagnostic)_ | kind=cross_run detail=RNA-like vs low-fidelity module | op controls_absent detail the low-fidelity module collapses (retention ~0 fitness negative) None |
| replication-feasible-budget _(supporting)_ | kind=derived path=final_lineage_fitness detail=net fitness after replication cost | op > value 0.0 detail net-positive within the replication budget |
| heredity-lineage-compositional-persists _(primary)_ | kind=cross_run detail=compositional vs non-heritable functional persistence in the agent-based lineage | op arms_reported detail compositional persistence above threshold and above the non-heritable control (realized heritability separates them) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Mineral/CO-driven peptide formation at 100 C, pH 7-10 | peptide formation rate/conditions | Huber & Wachtershauser 1998 | Represented as the peptide heredity arm in the agent-based lineage comparison (graded fitness 0.24, below RNA-like 0.38 and the compositional/hybrid arms); a reduced coupling, not a first-principles peptide-formation-rate reproduction |


## Study: Ecology Before LUCA (`og-07-ecology-before-luca`)

**Question.** Can a connected population of heterogeneous protocells develop capabilities - functional redundancy and robustness - that no single already-complete ancestral compartment holds on its own?

**Objective.** Run heterogeneous populations over a connected pore network; compare emergent capability redundancy (robustness) to the isolated counterfactual, and the collective repertoire to the LUCA reconstruction as a constraint.

**Hypothesis.** Heterogeneous populations develop metabolic diversity, cross-feeding and robustness single compartments do not; the emergent capability set overlaps the anaerobic H2/CO2/FeS LUCA reconstruction as a boundary.

**Claim.** A connected metacommunity of eight heterogeneous pore compartments develops functional redundancy that an isolated compartment cannot, reaching robustness about 0.71 against the isolated counterfactual's 0.42 and beating it on every one of eight seeds, so most functions get at least two-fold coverage and survive loss of a compartment. This emerges only in a bounded corner of the phase map: enough inter-compartment diversity to give cross-feeding something to share, enough exchange to spread it, and low enough parasite load not to erase it. The collective repertoire's roughly 0.6-0.7 overlap with an anaerobic H2/CO2/FeS LUCA reconstruction is reported as a far-end boundary the emergent set approaches, not as evidence that the earliest compartments were already LUCA-like.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- ecology: populations + evolution
- LUCA boundary condition

**Proposed models / methods**

- Vivarium multi-agent composites over shared spatial fields
- COBRApy for late-stage genome-scale benchmarks
- evolutionary algorithms

**Key assumptions**

- Use LUCA as a constraint on capabilities eventually acquired, not evidence the first replicators were LUCA-like.
- Each compartment is a coarse-grained capability vector over a protometabolic function set, not an instantiated metabolic network.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Ecology before LUCA` | `viva_genesis.composites.cross_feeding_metacommunity.cross_feeding_metacommunity` | 0 | n_compartments=8, exchange_rate=0.3, diversity=0.75, hgt_rate=0.1, seed=1 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.cross_feeding_metacommunity.cross_feeding_metacommunity`**

A reusable cross-feeding metacommunity module (Module L). N connected compartments, each a coarse-grained capability vector over a protometabolic function set, evolving under cross-feeding (byproduct of one feeds the next capability in a neighbour), horizontal transfer, and parasite/cooperator roles. Connected heterogeneous populations develop capability redundancy (robustness) and a collective repertoire single compartments do not; the emergent set's overlap with the anaerobic H2/CO2/FeS LUCA reconstruction is reported as a far-end boundary, not a target. Coarse-grained, deterministic reproduction (not wrapped ALife).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity = {
    'n_compartments': 8,   # Number of connected compartments
    'exchange_rate' : 0.3,   # Cross-feeding strength (use isolated flag, not 0, to disable)
    'diversity'     : 0.75,   # Inter-compartment heterogeneity (0 identical; 1 all different)
    'parasite_load' : 0.0,   # Fraction of compartments that drain neighbours
    'hgt_rate'      : 0.1,   # Horizontal-transfer rate (0 = no spread)
    'isolated'      : False,   # Isolated counterfactual arm: no cross-feeding/HGT/parasitism (use this, not exchange_rate=0)
    'seed'          : 1,   # RNG seed (reproducible assignment)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `flow_velocity` | float | meter/second | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `lineage_fitness` | float | dimensionless |  |

**Conditions**

- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`
- `spatial_flux_conservation` (invariant): `divergence(flux) == source_minus_sink`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-07-ecology-before-luca ===
STUDY = 'og-07-ecology-before-luca'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Ecology_before_LUCA = 100
INTERVAL_Ecology_before_LUCA = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity if 'ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity = materialize_composite('viva_genesis.composites.cross_feeding_metacommunity.cross_feeding_metacommunity', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity['state']}, core=core)
            _comp.run(STEPS_Ecology_before_LUCA)
        print('ran', 'Ecology before LUCA', f'(viva_genesis.composites.cross_feeding_metacommunity.cross_feeding_metacommunity) for', STEPS_Ecology_before_LUCA, 'steps')
    except Exception as _e:
        print(f"Ecology before LUCA not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**ecology-three-view**


In [ ]:
# ecology-three-view
show_viz(_render_one('local:EcologyMetacommunityThreeView', {'composite': 'viva_genesis.composites.cross_feeding_metacommunity.cross_feeding_metacommunity', 'grid_json': 'workspace/studies/og-07-ecology-before-luca/artifacts/ecology_robustness_grid.json', 'title': 'Campaign VII — ecology before LUCA'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity if 'ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity' in dir() else None)))

**metacommunity-phase-map**


In [ ]:
# metacommunity-phase-map
show_viz(_render_one('local:MetacommunityPhaseMap', {'composite': 'viva_genesis.composites.cross_feeding_metacommunity.cross_feeding_metacommunity', 'grid_json': 'workspace/studies/og-07-ecology-before-luca/artifacts/metacommunity_robustness_grid.json', 'title': 'Where cross-feeding robustness emerges'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity if 'ov_viva_genesis_composites_cross_feeding_metacommunity_cross_feeding_metacommunity' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cross-feeding-robustness _(primary)_ | kind=derived path=metacommunity_robustness detail=settled metacommunity robustness (functional redundancy) of the connected run | op > value 0.5 detail a majority of capabilities are redundantly covered (robust to a single-compartment loss) |
| connected-beats-isolated _(primary)_ | kind=cross_run detail=connected vs isolated metacommunity robustness across an 8-seed sweep | op arms_reported detail connected robustness strictly higher than isolated on every seed (measured min gain reported) |
| luca-capability-overlap _(supporting)_ | kind=cross_run detail=collective repertoire overlap with the anaerobic LUCA core (Weiss 2016) | op arms_reported detail overlap reported as a boundary not a target None |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| LUCA: anaerobic, H2-dependent, CO2-fixing, FeS-rich (355 protein families) | emergent capability overlap | Weiss 2016 / Moody 2024 | Measured — collective repertoire overlaps the anaerobic LUCA core at ~0.6-0.7 (reported boundary) |


## Study: Escape From the Vent (`og-08-escape-from-the-vent`)

**Question.** What is the minimum set of innovations a lineage needs to persist once geological support - the external gradient, mineral access and resource feed - is withdrawn?

**Objective.** Relax environmental support via ablation sweeps and identify the minimum innovation set for autonomy.

**Hypothesis.** A bounded innovation set (internal energy conversion + resource import + heredity) allows persistence as gradient strength/mineral access/feed are reduced; below it, systems stay obligately vent-tied.

**Claim.** Within this specific coarse-grained model, autonomy appears as a coordinated set of innovations (internal pump, active importer, heredity) acquired together, with any single removal collapsing persistence. This is retained as a reduced, model-specific hypothesis only: the general claim that this set is minimally sufficient for escape does not survive the Study 22 structure variation and is withdrawn.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- all modules integrated
- autonomy readout

**Proposed models / methods**

- multi-agent composites with relaxed boundaries
- ablation sweeps over gradient/mineral/feed
- population genetics

**Key assumptions**

- Autonomy is reduced dependence on the exact vent interface, measured by ablation — not a binary.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Escape from the vent` | `viva_genesis.composites.off_gradient_persistence.off_gradient_persistence` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.off_gradient_persistence.off_gradient_persistence`**

A reusable off-gradient-persistence module (Module M). A lineage's co-limited (Liebig-minimum) free-energy + resource budget under relaxing geological support (external gradient, mineral access, feed all declining over the run), with three toggleable innovations: internal energy conversion (has_pump), active resource import (has_importer) and information maintenance (has_heredity). Because the budget is co-limited the innovation set is irreducible — as support -> 0 only the full set keeps net budget above the survival floor, so a knockout sweep identifies {pump, importer, heredity} as the minimum robust set for autonomy. Coarse-grained, deterministic reproduction (not a wrapped ALife engine).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence = {
    'gradient_support'   : 1.0,   # Starting external-gradient support, relaxed to 0 over the run (dimensionless, 0-1)
    'mineral_access'     : 1.0,   # Starting mineral / FeS-catalyst access, relaxed to 0 (dimensionless, 0-1)
    'feed'               : 1.0,   # Starting geochemical feed, relaxed to 0 (dimensionless, 0-1)
    'has_pump'           : 1.0,   # Internal energy conversion innovation (1 present; 0 knockout)
    'has_importer'       : 1.0,   # Active resource-import innovation (1 present; 0 knockout)
    'has_heredity'       : 1.0,   # Information-maintenance innovation; without it the machinery degrades (1 present; 0 knockout)
    'catalyst_efficiency': 1.0,   # Internal-pump catalytic efficiency (dimensionless)
    'decline_rate'       : 0.02,   # Support relaxation per step (1/s); support reaches 0 partway through the run
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `gradient_support` | float | dimensionless | [0.0, 1.0] |
| `mineral_access` | float | dimensionless | [0.0, 1.0] |
| `feed` | float | dimensionless | [0.0, 1.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `survivors` | float | dimensionless | [0.0, 1.0] |
| `lineage_fitness` | float | dimensionless |  |
| `free_energy_flux` | float | watt |  |

**Conditions**

- `energy_directionality` (invariant): `free_energy_flux >= 0 and co-limited budget (min of energy, resource)`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-08-escape-from-the-vent ===
STUDY = 'og-08-escape-from-the-vent'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Escape_from_the_vent = 100
INTERVAL_Escape_from_the_vent = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence if 'ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence = materialize_composite('viva_genesis.composites.off_gradient_persistence.off_gradient_persistence', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence['state']}, core=core)
            _comp.run(STEPS_Escape_from_the_vent)
        print('ran', 'Escape from the vent', f'(viva_genesis.composites.off_gradient_persistence.off_gradient_persistence) for', STEPS_Escape_from_the_vent, 'steps')
    except Exception as _e:
        print(f"Escape from the vent not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**escape-three-view**


In [ ]:
# escape-three-view
show_viz(_render_one('local:EscapeOffGradientThreeView', {'composite': 'viva_genesis.composites.off_gradient_persistence.off_gradient_persistence', 'grid_json': 'workspace/studies/og-08-escape-from-the-vent/artifacts/off_gradient_grid.json', 'title': 'Campaign VIII — escape from the vent'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence if 'ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence' in dir() else None)))

**off-gradient-survival-frontier**


In [ ]:
# off-gradient-survival-frontier
show_viz(_render_one('local:OffGradientPhaseMap', {'composite': 'viva_genesis.composites.off_gradient_persistence.off_gradient_persistence', 'grid_json': 'workspace/studies/og-08-escape-from-the-vent/artifacts/off_gradient_grid.json', 'title': 'The survival frontier off the gradient'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence if 'ov_viva_genesis_composites_off_gradient_persistence_off_gradient_persistence' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| survival-off-gradient _(primary)_ | kind=derived path=offgradient_survival detail=surviving fraction at the end of the run as support -> 0=None | op >= value 0.5 detail a lineage persists off-gradient (surviving fraction >= 0.5) |
| minimum-innovation-set _(primary)_ | kind=cross_run detail=full set vs each single knockout surviving fraction off-gradient=None | op arms_reported detail full set persists off-gradient and every single knockout collapses (each innovation is necessary) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Alkaline-vent origin — the first cells are vent-tied, dependent on the geochemical proton gradient and FeS mineral catalysis; escaping to free-living requires internalising energy generation. | off-gradient persistence vs innovation set | Martin & Russell 2007; Lane & Martin 2012; Sojo et al. 2014 | Reproduced as a coarse-grained co-limited budget — the full innovation set persists off-gradient; each single knockout collapses. |


## Study: The Adversarial Control (`og-adv-generic-nonequilibrium`)

**Question.** Are the campaign's positive gates genuinely dependent on vent-specific mechanisms, or would any generic driven nonequilibrium chemistry pass them? It should not.

**Objective.** Run the og-01..03 behavior tests on a vent-feature-ablated control; the campaign's gates must reject it, and the vent must pass the same tests.

**Hypothesis.** The og-01/og-02/og-03 gates REJECT this control; if they pass it, the gates measure generic nonequilibrium chemistry, not the vent mechanism.

**Claim.** Run with the vent's characteristic features ablated - no sustained H2, no Fe/Ni/S catalytic interface and no pore concentration - the same real composites fail all five discriminating gates (carbon flux, formate yield, protometabolic throughput, thermophoretic Peclet and enrichment), while the vent configuration passes all five, with the tightest rejection margin about 1.8 decades. This is a falsification control: a pass elsewhere in the campaign certifies the vent mechanism rather than simply any nonequilibrium chemistry producing activity. The extreme enrichment magnitude itself is a reduced-model, solver-sensitive figure and is used only as a pass/fail discriminator here.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- the same real og-01/02/03 composites with vent features (H2, Fe/Ni/S catalyst, pore concentration) disabled

**Proposed models / methods**

- flat gradient, inert surface, open pore, well-mixed — a generic abiotic nonequilibrium mixture
- re-run the og-01/02/03 behavior-test gates unchanged and require rejection

**Key assumptions**

- This study passes by REJECTING; a control that passes the gates would mean the gates are not vent-specific.
- The control is a parameterization of the real og-01/02/03 composites, so it tests the same models, not a mock.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Generic non-vent control (carbon-fixation composite, vent features off)` | `viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation` | 0 | gradient_efficiency=0.0, catalyst_active=0.0, h2=1e-09 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation`**

A reusable H2/pH-driven carbon-fixation module. Module C = genuine FiPy finite-volume Nernst-Planck pH-gradient electrochemistry (pmf, current density; needs fipy). Module D = genuine eQuilibrator ΔG' feasibility gate (needs equilibrator_api) + lab-calibrated (Preiner/Hudson) mineral-catalyzed CO2 reduction to formate/acetate/pyruvate. gradient_efficiency=1.0 is the favorable arm; ~0.1 is the Tutolo-2026 in-situ collapse. catalyst_active/h2=0 give the controls.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation = {
    'gradient_efficiency': 1.0,   # Usable fraction of the geochemical ΔpH at in-situ T/P (1.0 favorable; ~0.1 Tutolo-2026 collapse; 0 = no-gradient control)
    'catalyst_active'    : 1.0,   # Active Fe/Ni/S mineral catalyst (0 = no-catalyst control)
    'h2'                 : 0.005,   # Dissolved H2 feed (M); ~0 = no-H2 control
    'co2'                : 0.002,   # Dissolved CO2 feed (M)
    'ph_vent'            : 9.5,   # Alkaline vent-side pH (from og-00)
    'ph_ocean'           : 6.0,   # Acidic early-ocean-side pH
    'dilution_rate'      : 0.0,   # Open-flow washout (1/s); 0 = closed batch (yield studies), >0 = open vent reactor → sustained-throughput regime (phase map)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `pH` | float | dimensionless | [0.0, 14.0] |
| `H2` | float | molar | >= 0.0 |
| `CO2` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `net_carbon_flux` | float | mole/second |  |
| `formate` | float | molar | >= 0.0 |

**Conditions**

- `charge_neutrality` (invariant): `abs(net_charge) < tol`
- `energy_directionality` (invariant): `free_energy_flux_logged == true`
- `mass_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O) < tol`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-adv-generic-nonequilibrium ===
STUDY = 'og-adv-generic-nonequilibrium'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Generic_non_vent_control_carbon_fixation_composite_vent_features_off = 100
INTERVAL_Generic_non_vent_control_carbon_fixation_composite_vent_features_off = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation if 'ov_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation = materialize_composite('viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_carbon_fixation_hydrothermal_carbon_fixation['state']}, core=core)
            _comp.run(STEPS_Generic_non_vent_control_carbon_fixation_composite_vent_features_off)
        print('ran', 'Generic non-vent control (carbon-fixation composite, vent features off)', f'(viva_genesis.composites.carbon_fixation.hydrothermal_carbon_fixation) for', STEPS_Generic_non_vent_control_carbon_fixation_composite_vent_features_off, 'steps')
    except Exception as _e:
        print(f"Generic non-vent control (carbon-fixation composite, vent features off) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**vent-specificity-three-view**


In [ ]:
# vent-specificity-three-view
show_viz(_render_one('local:VentSpecificityThreeView', {'title': 'Adversarial control — vent-specificity of the gates', 'grid_json': 'workspace/studies/og-adv-generic-nonequilibrium/artifacts/vent_specificity_grid.json'}, RUNS_DB, STUDY_YAML))

**vent-specificity-map**


In [ ]:
# vent-specificity-map
show_viz(_render_one('local:VentSpecificityMap', {'title': 'Adversarial control — vent-specificity of the gates', 'grid_json': 'workspace/studies/og-adv-generic-nonequilibrium/artifacts/vent_specificity_grid.json'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| gates-reject-generic _(primary)_ | kind=observable detail=run og-01/02/03 gates on the generic control; each measured value below its threshold | op controls_absent detail all five gates reject generic (net flux / formate / throughput / Péclet / enrichment below threshold) |
| vent-specificity-demonstrated _(primary)_ | kind=observable detail=vent vs generic on identical gates | op arms_reported detail vent passes 5/5 where the control fails 5/5 |


## Study: The Nitrogen Bottleneck (`og-res-nitrogen-assimilation`)

**Question.** Can an FeS-mediated vent reduce the early ocean's oxidised and inert nitrogen (nitrate, nitrite, N2) to ammonium fast enough to keep pace with carbon fixation, or does nitrogen become the first limiting resource?

**Objective.** Add nitrogen as its own module (FeS-mediated NO3-/NO2- -> NH4+ with minor N2 fixation and carbon-demanded reductive amination), track the C:N supply ratio, and map the region where reduced-N production keeps pace with the carbon-fixation demand.

**Hypothesis.** On an Fe/Ni/S catalyst with an H2 reductant, nitrate is reduced through nitrite to ammonium and drawn into amino-acid precursors by reductive amination; reduced N keeps pace with reduced carbon (C:N supply ratio near Redfield) only inside a bounded region of (nitrate supply x FeS activity), and outside it — low feed or passivated mineral — the C:N ratio blows up and nitrogen, not carbon or energy, becomes the first limiting resource.

**Claim.** In the favourable regime FeS-mediated reduction of nitrate through nitrite sustains a standing ammonium pool of about 1.0 mM and holds the C:N supply ratio near 5, inside the viable near-Redfield band, with nitrogen conserved to machine precision. Strip out the FeS catalyst or the H2 reductant and ammonium falls below a micromolar, while scarce nitrate pushes the C:N ratio to about 47. Nitrogen therefore keeps pace only within a bounded nitrate-supply by FeS-activity region, and outside it nitrogen - not carbon or energy - becomes the first limiting resource.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- I: FeS-mediated nitrate/nitrite reduction to ammonium
- I: minor abiotic N2 fixation on fresh FeS
- I: carbon-demanded reductive amination toward amino-acid precursors

**Proposed models / methods**

- FeS-catalysed dissimilatory nitrate reduction kinetics (H2-gated, temperature-windowed)
- Redfield-like C:N supply-ratio accounting
- Open-flow (feed + washout) sustained-throughput reactor

**Key assumptions**

- Reduced nitrogen (NH4+) is a hard co-requirement with reduced carbon; the early ocean's accessible N is oxidised (NO3-/NO2-) or inert (N2).
- The useful decision observable is the C:N SUPPLY ratio (production vs demand), not any single pool.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Nitrogen as a limiting resource — favourable` | `viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor` | 0 | — |
| `Nitrogen-limited (scarce nitrate feed)` | `viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor` | 0 | no3_supply=2e-06 |

Declared parameter sets (`study.yaml` variants):

- **nitrogen-limited** — `no3_supply=2e-06`
- **control-no-fes** — `fes_activity=1e-06`
- **control-no-h2** — `h2=1e-12`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor`**

A reusable FeS-mediated reduced-nitrogen module. The real ReducedNitrogenProductionProcess reduces NO3-/NO2- to NH4+ on an Fe/Ni/S catalyst (H2-dependent, temperature-windowed), adds a minor abiotic N2-fixation source, and draws ammonium into amino-acid precursors by reductive amination at a rate set by the carbon-fixation demand. Tracks the C:N supply ratio (carbon demand / reduced-N production) to test whether reduced nitrogen keeps pace with reduced carbon. fes_activity=0 / h2=0 are the no-mineral / no-reductant controls.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor = {
    'no3_supply'  : 2e-05,   # Open-flow nitrate feed (M/s); sets the ceiling on sustained reduced-N production
    'fes_activity': 1.0,   # Active Fe/Ni/S catalyst in [0,1] (0 = no-mineral control)
    'h2'          : 0.005,   # Dissolved H2 reductant (M); ~0 = no-reductant control
    'temperature' : 353.15,   # Temperature (K); FeS nitrate-reduction optimum ~80 C
    'c_demand'    : 0.0001,   # Carbon-fixation demand pulling reduced N into precursors (mol/s)
    'no3_init'    : 0.001,   # Initial nitrate pool (M)
    'n2'          : 0.0005,   # Dissolved N2 available for minor fixation (M)
    'loss_rate'   : 0.01,   # Open-flow washout of NH4+/precursor (1/s); 0 = closed batch (conservation control)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `NO3` | float | molar | >= 0.0 |
| `H2` | float | molar | >= 0.0 |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `fes_activity` | float | dimensionless | [0.0, 1.0] |
| `c_demand` | float | mole/second | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `NH4` | float | molar | >= 0.0 |
| `amino_precursor` | float | molar | >= 0.0 |
| `nh4_production_rate` | float | mole/second | >= 0.0 |
| `cn_supply_ratio` | float | dimensionless | >= 0.0 |
| `total_n` | float | molar | >= 0.0 |

**Conditions**

- `nitrogen_mass_balance` (invariant): `abs(delta_total_N - (feed - washout)) < tol`
- `reductant_necessity` (invariant): `nh4_production == 0 when (fes_activity == 0 or H2 == 0)`
- `non_negative_pools` (invariant): `NO3 >= 0 and NO2 >= 0 and NH4 >= 0`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-res-nitrogen-assimilation ===
STUDY = 'og-res-nitrogen-assimilation'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Nitrogen_as_a_limiting_resource_favourable = 100
INTERVAL_Nitrogen_as_a_limiting_resource_favourable = 1.0
STEPS_Nitrogen_limited_scarce_nitrate_feed = 100
INTERVAL_Nitrogen_limited_scarce_nitrate_feed = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor if 'ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor = materialize_composite('viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor['state']}, core=core)
            _comp.run(STEPS_Nitrogen_as_a_limiting_resource_favourable)
        print('ran', 'Nitrogen as a limiting resource — favourable', f'(viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor) for', STEPS_Nitrogen_as_a_limiting_resource_favourable, 'steps')
    except Exception as _e:
        print(f"Nitrogen as a limiting resource — favourable not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor if 'ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor = materialize_composite('viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor['state']}, core=core)
            _comp.run(STEPS_Nitrogen_limited_scarce_nitrate_feed)
        print('ran', 'Nitrogen-limited (scarce nitrate feed)', f'(viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor) for', STEPS_Nitrogen_limited_scarce_nitrate_feed, 'steps')
    except Exception as _e:
        print(f"Nitrogen-limited (scarce nitrate feed) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**nitrogen-assimilation-timeseries**


In [ ]:
# nitrogen-assimilation-timeseries
show_viz(_render_one('local:NitrogenAssimilationViz', {'composite': 'viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor', 'n_steps': 120, 'title': 'Reduced-nitrogen assimilation (World view)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor if 'ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor' in dir() else None)))

**c-to-n-supply-balance**


In [ ]:
# c-to-n-supply-balance
show_viz(_render_one('local:NitrogenCarbonBalanceViz', {'composite': 'viva_genesis.composites.nitrogen_assimilation.reduced_nitrogen_reactor', 'n_steps': 120, 'title': 'Does reduced N keep pace with carbon? (Mechanism view)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor if 'ov_viva_genesis_composites_nitrogen_assimilation_reduced_nitrogen_reactor' in dir() else None)))

**nitrogen-assimilation-phase-map**


In [ ]:
# nitrogen-assimilation-phase-map
show_viz(_render_one('local:NitrogenAssimilationPhaseMap', {'title': 'Nitrogen as a limiting resource (Decision view)', 'grid_json': 'workspace/studies/og-res-nitrogen-assimilation/artifacts/nitrogen_assimilation_phase_grid.json'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| ammonium-production-sustained _(primary)_ | kind=derived path=settled_nh4_uM detail=settled ammonium pool uM=None | op > value 100.0 detail settled NH4+ > 100 uM (a sustained reduced-N pool) |
| reduced-nitrogen-source-exists _(primary)_ | kind=derived path=peak_nh4_production_rate detail=peak ammonium production rate | op > value 1e-06 detail peak NH4+ production > 0 (viable reduced-N source) |
| nitrogen-keeps-pace-with-carbon _(primary)_ | kind=derived path=cn_supply_ratio_settled detail=settled C:N supply ratio | op range low 1.0 high 8.0 detail C:N in the viable keeps-pace band (near Redfield) |
| nitrogen-mass-conserved _(diagnostic)_ | kind=derived path=max_abs_n_mass_residual detail=max |per-step N-mass residual| | op <= value 1e-09 detail max |nitrogen-mass residual| at machine precision |
| feature-ablation-no-ammonium _(primary)_ | kind=cross_run detail=favourable ammonium vs no-FeS / no-H2 | op controls_absent detail ammonium present with FeS+H2 <1 uM in every control None |
| nitrogen-supply-sets-cn _(diagnostic)_ | kind=cross_run detail=favourable C:N vs scarce-nitrate C:N | op arms_reported detail report both arms; flag where nitrogen becomes limiting first |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Ammonia forms from dinitrogen on iron-sulfide surfaces (low yield, fresh-surface dependent) at ~70-80 C | ammonium production on FeS vs controls | Doerr 2003 | Met qualitatively — the FeS-mediated route produces ammonium only with an active mineral and an H2 reductant; N2 fixation is modelled as a deliberately minor source alongside the dominant nitrate-reduction route. |
| Native-metal / mineral catalysis drives reductive prebiotic chemistry toward amino-acid precursors | reduced-N draw into amino-acid precursors | Varma 2018 / Huber 1997 | Met — reductive amination draws ammonium into an amino-acid-precursor pool at a rate set by the carbon-fixation demand (the C:N coupling). |


## Study: The Phosphorus Bottleneck (`og-res-phosphorus-availability`)

**Question.** Can the same thermal-gradient vent pore that concentrates carbon and nitrogen also make phosphorus chemically available, or does phosphate precipitate and adsorb before it is useful?

**Objective.** Build a real P-speciation process (phosphate <-> polyphosphate <-> reactive-P with a precipitation/adsorption mineral sink), couple it to the og-03 thermal pore, and map where reactive-P availability beats loss and co-concentrates with carbon.

**Hypothesis.** In a thermal-gradient pore (coupled from og-03), enrichment drives phosphate condensation and activation steeply enough that reactive-P formation outruns the first-order precipitation/adsorption sink in a bounded (source × enrichment) region — and that same enrichment co-concentrates P alongside carbon; without the gradient (open pore), precipitation wins and no reactive P accumulates.

**Claim.** A P-speciation network coupled to the thermal pore makes reactive phosphorus net-positive only in a bounded phosphate-source by pore-enrichment window where condensation and activation outrun the precipitation and adsorption sink; at the favourable point reactive P accumulates to about 94% of the supplied phosphorus with only about 6% precipitated. The same enrichment that concentrates carbon co-concentrates phosphorus, whereas the open-pore ablation precipitates about 96% of it before it is useful. In the model phosphorus is thus a localization-gated bottleneck rather than a freely available input.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- I: limiting-resource phosphorus speciation
- F: transport/concentration/retention (og-03 pore coupling)

**Proposed models / methods**

- SBML + libRoadRunner/Tellurium P-speciation network
- og-03 FiPy thermophoretic trap as the enrichment coupling
- P-atom mass balance + precipitation/adsorption sink

**Key assumptions**

- Reactive-P availability arises from the balance of condensation, activation, hydrolysis and precipitation/adsorption — not assumed; the precipitation sink is explicit.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Phosphorus availability: the phosphorus bottleneck` | `viva_genesis.composites.phosphorus_availability.phosphorus_availability` | 0 | phosphate_supply=0.0001, pore_enrichment=30.0, temperature=333.15, loss_rate=0.08 |

Declared parameter sets (`study.yaml` variants):

- **control-open-pore** — `pore_enrichment=1.0`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.phosphorus_availability.phosphorus_availability`**

A reusable phosphorus-availability module. A genuine libRoadRunner/Tellurium SBML ODE integration (needs tellurium) of a P-speciation network (phosphate <-> polyphosphate <-> reactive-P with a precipitation/adsorption mineral sink) driven by a dissolved-phosphate supply, the og-03 thermal-gradient pore enrichment, and pore temperature. Emergent: the standing reactive-P pool, reactive-P production vs precipitation loss (net availability), and the precipitated fraction — does reactive P become available before phosphate precipitates?


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_phosphorus_availability_phosphorus_availability = {
    'phosphate_supply': 0.0001,   # Dissolved orthophosphate supply / initial pool (M)
    'pore_enrichment' : 30.0,   # Thermal-gradient pore enrichment from the og-03 trap (>=1; 1 = open pore / no gradient control)
    'temperature'     : 333.15,   # Mean pore temperature (K)
    'loss_rate'       : 0.08,   # Precipitation / adsorption loss rate k_prec (1/s) — the mineral-sink bottleneck
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `phosphate_supply` | float | molar | >= 0.0 |
| `pore_enrichment` | float | dimensionless | >= 1.0 |
| `temperature` | float | kelvin | [273.0, 673.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `reactive_P` | float | molar | >= 0.0 |
| `net_reactive_P` | float | mole/second |  |
| `precipitated_fraction` | float | dimensionless | [0.0, 1.0] |

**Conditions**

- `phosphorus_mass_balance` (invariant): `abs(total_P - supplied_P) / supplied_P < tol`
- `nonnegative_pools` (invariant): `phosphate >= 0 and polyphosphate >= 0 and reactive_P >= 0 and mineral_P >= 0`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-res-phosphorus-availability ===
STUDY = 'og-res-phosphorus-availability'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Phosphorus_availability_the_phosphorus_bottleneck = 100
INTERVAL_Phosphorus_availability_the_phosphorus_bottleneck = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_phosphorus_availability_phosphorus_availability if 'ov_viva_genesis_composites_phosphorus_availability_phosphorus_availability' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_phosphorus_availability_phosphorus_availability = materialize_composite('viva_genesis.composites.phosphorus_availability.phosphorus_availability', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_phosphorus_availability_phosphorus_availability['state']}, core=core)
            _comp.run(STEPS_Phosphorus_availability_the_phosphorus_bottleneck)
        print('ran', 'Phosphorus availability: the phosphorus bottleneck', f'(viva_genesis.composites.phosphorus_availability.phosphorus_availability) for', STEPS_Phosphorus_availability_the_phosphorus_bottleneck, 'steps')
    except Exception as _e:
        print(f"Phosphorus availability: the phosphorus bottleneck not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**phosphorus-speciation**


In [ ]:
# phosphorus-speciation
show_viz(_render_one('local:PhosphorusSpeciationViz', {'composite': 'viva_genesis.composites.phosphorus_availability.phosphorus_availability', 'n_steps': 120, 'title': 'Phosphorus bottleneck — P-species trajectories (World view)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_phosphorus_availability_phosphorus_availability if 'ov_viva_genesis_composites_phosphorus_availability_phosphorus_availability' in dir() else None)))

**phosphorus-mechanism**


In [ ]:
# phosphorus-mechanism
show_viz(_render_one('local:PhosphorusMechanismViz', {'composite': 'viva_genesis.composites.phosphorus_availability.phosphorus_availability', 'n_steps': 120, 'title': 'Phosphorus bottleneck — reactive-P production vs loss (Mechanism view)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_phosphorus_availability_phosphorus_availability if 'ov_viva_genesis_composites_phosphorus_availability_phosphorus_availability' in dir() else None)))

**phosphorus-availability-map**


In [ ]:
# phosphorus-availability-map
show_viz(_render_one('local:PhosphorusAvailabilityMap', {'title': 'The phosphorus bottleneck — reactive-P availability landscape (Decision view)', 'grid_json': 'workspace/studies/og-res-phosphorus-availability/artifacts/phosphorus_availability_grid.json'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| reactive-p-available _(primary)_ | kind=derived path=phosphorus_net_reactive_availability detail=settled net reactive-P availability = production - precipitation loss | op > value 0.0 detail net availability > 0 (reactive P produced faster than lost) |
| p-coconcentrates-with-carbon _(primary)_ | kind=cross_run detail=og-03 pore enrichment vs the carbon co-concentration floor; P reactive-availability at that enrichment | op arms_reported detail the same pore regime co-concentrates P with C (enrichment >= carbon floor AND P reactive-availability positive) |
| open-pore-no-enrichment-control _(primary)_ | kind=cross_run detail=favorable enriched pore vs open-pore / no-gradient control | op controls_absent detail reactive-P availability collapses (net <= 0 ~fully precipitated) without enrichment None |
| p-mass-conserved _(diagnostic)_ | kind=derived path=phosphorus_p_mass_residual detail=max |total P - supplied| / supplied | op <= value 1e-06 detail P-mass residual small (P-atom-conserving network) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Reactive reduced-P (schreibersite-derived) and condensed/activated phosphates form under basic conditions; dissolved phosphate is prone to precipitation/adsorption. | reactive-P availability vs precipitated fraction | Pasek 2017 | measured — reactive-P availability + precipitated fraction graded from the real run |


## Study: The Energy Currency (`og-energy-proto-energetics`)

**Question.** Before modern pumps and ATP, can a primitive chemical system capture environmental geochemical free energy into a transferable store that can perform work?

**Objective.** Represent the exergonic-to-endergonic coupling as explicit energetic bookkeeping (a free-energy ledger), not a favourable reaction direction; map where a transferable currency is maintained and measure the stored-work fraction.

**Hypothesis.** A pre-ATP carrier (thioester / acyl-phosphate class) coupled to an exergonic geochemical drive maintains a transferable currency above threshold under endergonic load — but only where the drive ΔG clears the carrier's high-energy bond and the coupling is efficient enough to beat discharge + hydrolysis. A bounded fraction of the geochemical free energy is captured as stored usable work; the rest is dissipated. Remove the coupling or starve the drive below the bond and the currency collapses.

**Claim.** A specified thioester- and/or acyl-phosphate-class carrier, written as an explicit chemical cycle (synthesis → activation → use → hydrolysis) of named species with a full free-energy ledger, either supports useful coupled work with no hidden energy inflow, or it does not. The abstract currency is replaced by balanced chemistry: capture is credited only when every synthesis, activation, discharge, leak and maintenance cost is accounted and the geochemical drive supplies the high-energy bond.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- energetics: pre-ATP energy currency (thioesters / acyl phosphates / polyphosphates)

**Proposed models / methods**

- free-energy ledger (input = useful work + stored + dissipated) as a conservation invariant
- charge / discharge / hydrolysis kinetics of a transferable carrier pool
- phase map over (geochemical free-energy drive x coupling efficiency)

**Key assumptions**

- Capturing environmental free energy into a transferable store is the transition — not merely a favourable reaction direction.
- A carrier can only be charged when the drive ΔG clears its high-energy bond (a thermodynamic gate, not just a rate).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Energy carrier cycle (explicit thioester/acyl-phosphate chemistry)` | `viva_genesis.composites.og_12_energy_carrier_cycle.og_12_energy_carrier_cycle` | 0 | drive_free_energy=50.0, bond_energy_kjmol=31.5 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.og_12_energy_carrier_cycle.og_12_energy_carrier_cycle`**

Study 12: the energy currency as explicit chemistry. GeochemicalDriveProcess supplies an exergonic drive ΔG that activates a named thioester carrier only when it clears the high-energy bond; the carrier does useful work against an endergonic load and leaks by hydrolysis. A closed free-energy ledger (budget_residual ~ 0) credits useful coupled work only with no hidden energy inflow — replacing the abstract charged-pool currency.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_og_12_energy_carrier_cycle_og_12_energy_carrier_cycle = {
    'drive_free_energy': 50.0,   # Geochemical drive ΔG supplied (kJ/mol)
    'bond_energy_kjmol': 31.5,   # Carrier high-energy bond (thioester ~31.5; acyl-phosphate ~43)
    'load'             : 20.0,   # Endergonic load the carrier does work against (kJ/mol)
    'k_hydrolyze'      : 0.02,   # Hydrolysis leak rate (1/s)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `drive_free_energy` | float | kilojoule/mole | >= 0.0 |
| `coupling_efficiency` | float | dimensionless | [0.0, 1.0] |
| `load` | float | dimensionless | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `energy_charge` | float | molar | >= 0.0 |
| `energy_charge_fraction` | float | dimensionless | [0.0, 1.0] |
| `energy_input` | float | kilojoule/liter | >= 0.0 |
| `energy_work` | float | kilojoule/liter | >= 0.0 |
| `energy_stored` | float | kilojoule/liter | >= 0.0 |
| `energy_dissipated` | float | kilojoule/liter | >= 0.0 |
| `energy_stored_work_fraction` | float | dimensionless | [0.0, 1.0] |
| `energy_ledger_residual` | float | kilojoule/liter |  |

**Conditions**

- `free_energy_accounting` (invariant): `abs(energy_input - (energy_work + energy_stored + energy_dissipated)) < tol`
- `carrier_bounds` (invariant): `0 <= energy_charge <= carrier_pool`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-energy-proto-energetics ===
STUDY = 'og-energy-proto-energetics'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Energy_carrier_cycle_explicit_thioester_acyl_phosphate_chemistry = 100
INTERVAL_Energy_carrier_cycle_explicit_thioester_acyl_phosphate_chemistry = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_12_energy_carrier_cycle_og_12_energy_carrier_cycle if 'ov_viva_genesis_composites_og_12_energy_carrier_cycle_og_12_energy_carrier_cycle' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_12_energy_carrier_cycle_og_12_energy_carrier_cycle = materialize_composite('viva_genesis.composites.og_12_energy_carrier_cycle.og_12_energy_carrier_cycle', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_12_energy_carrier_cycle_og_12_energy_carrier_cycle['state']}, core=core)
            _comp.run(STEPS_Energy_carrier_cycle_explicit_thioester_acyl_phosphate_chemistry)
        print('ran', 'Energy carrier cycle (explicit thioester/acyl-phosphate chemistry)', f'(viva_genesis.composites.og_12_energy_carrier_cycle.og_12_energy_carrier_cycle) for', STEPS_Energy_carrier_cycle_explicit_thioester_acyl_phosphate_chemistry, 'steps')
    except Exception as _e:
        print(f"Energy carrier cycle (explicit thioester/acyl-phosphate chemistry) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**proto-energetics-world**


In [ ]:
# proto-energetics-world
show_viz(_render_one('local:ProtoEnergeticsWorldViz', {'composite': 'viva_genesis.composites.proto_energetics.proto_energetics', 'title': 'Proto-energetics — charging and spending the currency (World view)'}, RUNS_DB, STUDY_YAML))

**proto-energetics-provenance**


In [ ]:
# proto-energetics-provenance
show_viz(_render_one('local:ProtoEnergeticsProvenanceViz', {'title': 'Proto-energetics — the energy-provenance ledger (Mechanism view)'}, RUNS_DB, STUDY_YAML))

**proto-energetics-map**


In [ ]:
# proto-energetics-map
show_viz(_render_one('local:ProtoEnergeticsMap', {'composite': 'viva_genesis.composites.proto_energetics.proto_energetics', 'grid_json': 'workspace/studies/og-energy-proto-energetics/artifacts/proto_energetics_phase_grid.json', 'title': 'Proto-energetics — capturing free energy into a transferable currency (Decision view)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| currency-maintained _(primary)_ | kind=derived path=energy_final_charge_fraction detail=charged fraction of the carrier pool at the end of the run under load=None | op >= value 0.25 detail the currency is maintained (charged fraction >= 0.25) against discharge + hydrolysis |
| captures-free-energy _(primary)_ | kind=derived path=energy_stored_work_fraction detail=(delivered work + currently stored)/input free energy at the end of the run | op >= value 0.3 detail capture into transferable currency exceeds 30% of the tapped geochemical free energy |
| ablation-no-coupling-or-drive-collapses _(primary)_ | kind=cross_run detail=no-coupling and no-drive controls vs the favourable arm | op controls_absent detail both ablations collapse (charged fraction ~0) where the favourable arm maintains the currency |
| free-energy-conserved _(supporting)_ | kind=derived path=energy_ledger_residual detail=worst-case |input - (work + stored + dissipated)| over the run | op <= value 1e-09 detail the ledger residual stays at machine precision (free-energy accounting invariant) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Pre-ATP carriers (thioesters, acyl phosphates) couple exergonic to endergonic chemistry as a transferable energy currency before membrane bioenergetics / ATP | charged fraction + stored-work fraction vs drive and coupling | Buckel & Thauer 2013 (flavin-based electron bifurcation / energy coupling); Lane & Martin 2010 (chemiosmotic coupling at the vent) | Reproduced as explicit free-energy bookkeeping (coarse-grained reduced kinetics) |


## Study: Growth and Division (`og-growth-division`)

**Question.** Can an internal autocatalytic metabolism change a compartment's physical fate - driving growth and division through osmotic pressure alone - without assigning any abstract fitness score?

**Objective.** Couple a reduced autocatalytic osmolyte source to compartment osmotic physics (osmotic pressure → solvent influx → volume → division) with a semi-permeable membrane; map the (network throughput × membrane permeability) plane and locate the viable growth→division band between starvation and lysis, conserving mass across growth and division.

**Hypothesis.** Product accumulation from an internal autocatalytic source generates osmotic pressure that drives solvent influx and volume growth to a division threshold; there is a bounded membrane-permeability band in which the compartment sustains net growth leading to division, flanked by starvation (too permeable, the product leaks before osmotic pressure can build) and lysis (too impermeable, the pressure bursts the membrane).

**Claim.** An internal autocatalytic source accumulates an osmolyte whose osmotic pressure draws solvent across the membrane and grows the compartment to a division threshold; in the matched regime it divides eight times over the run, with an osmolyte mass-balance residual of about 4e-32 mol. Only about 25% of the throughput by permeability grid forms a viable growth-to-division band, flanked by starvation when the membrane is too permeable and the product leaks before pressure can build, and lysis when it is too impermeable and the pressure bursts the membrane, while zero throughput produces essentially no growth. The model therefore delivers a chemically selected, dividing protocell through mass-conserving osmotic coupling, with no abstract fitness function assigned.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- H+: metabolism changes the compartment's physical fate (osmotic growth → division)
- osmotic pressure as a physical selection mechanism (no abstract fitness score)

**Proposed models / methods**

- custom coarse-grained osmotic membrane kinetics (van 't Hoff + solubility-diffusion permeability)
- reduced autocatalytic source (og-02 rTCA-style turn)
- parameter sweep over (throughput × permeability) → decision surface

**Key assumptions**

- The compartment (og-04 protocell) and the autocatalytic network (og-02 rTCA source) are given; this stage couples them to osmotic physics to test whether chemistry drives a physical division.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Metabolism to osmotic growth and division (favorable)` | `viva_genesis.composites.osmotic_growth_division.osmotic_growth_division` | 0 | network_throughput=1e-19, membrane_permeability=1e-09, autocatalytic_rate=0.5, division_threshold=6.4e-17, leak_rate=1.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.osmotic_growth_division.osmotic_growth_division`**

A reusable osmotic growth-and-division module (metabolism → growth & division). An internal autocatalytic osmolyte source coupled to compartment osmotic physics: product accumulation → osmotic pressure (van 't Hoff) → solvent influx → volume growth → division at a size threshold. Membrane permeability sets the fate: too permeable starves (product leaks, no osmotic build-up), too impermeable lyses (pressure bursts the membrane), matched gives sustained growth → division. Units-typed, mass-conserving coarse-grained reproduction (not wrapped membrane MD).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division = {
    'network_throughput'   : 1e-19,   # Internal autocatalytic/metabolic osmolyte source rate (mole/second); ~0 = ablation control
    'autocatalytic_rate'   : 0.5,   # Self-amplification gain of the source (dimensionless)
    'membrane_permeability': 1e-09,   # Membrane solute permeability (meter/second); the fate knob (leaky→starve, tight→lyse)
    'division_threshold'   : 6.4e-17,   # Volume at which the compartment divides (liter); default 1.6× initial volume
    'leak_rate'            : 1.0,   # Multiplier on the solute permeability for the outward leak (dimensionless)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `network_throughput` | float | mole/second | >= 0.0 |
| `membrane_permeability` | float | meter/second | >= 0.0 |
| `external_osmolyte` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `volume` | float | liter | >= 0.0 |
| `internal_concentration` | float | molar | >= 0.0 |
| `osmotic_pressure` | float | pascal | >= 0.0 |
| `division_count` | integer | dimensionless | >= 0 |
| `osmolyte_balance_residual` | float | mole |  |

**Conditions**

- `mass_balance` (invariant): `abs(osmolyte_balance_residual) < tol`
- `osmotic_coupling_directionality` (invariant): `osmotic_pressure >= 0 and solvent influx follows the osmotic gradient (dV/dt = Lp*A*Pi)`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-growth-division ===
STUDY = 'og-growth-division'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Metabolism_to_osmotic_growth_and_division_favorable = 100
INTERVAL_Metabolism_to_osmotic_growth_and_division_favorable = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division if 'ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division = materialize_composite('viva_genesis.composites.osmotic_growth_division.osmotic_growth_division', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division['state']}, core=core)
            _comp.run(STEPS_Metabolism_to_osmotic_growth_and_division_favorable)
        print('ran', 'Metabolism to osmotic growth and division (favorable)', f'(viva_genesis.composites.osmotic_growth_division.osmotic_growth_division) for', STEPS_Metabolism_to_osmotic_growth_and_division_favorable, 'steps')
    except Exception as _e:
        print(f"Metabolism to osmotic growth and division (favorable) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**osmotic-growth-world**


In [ ]:
# osmotic-growth-world
show_viz(_render_one('local:OsmoticGrowthWorld', {'composite': 'viva_genesis.composites.osmotic_growth_division.osmotic_growth_division', 'title': 'Growth & division — volume, concentration, division events'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division if 'ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division' in dir() else None)))

**osmotic-coupling-mechanism**


In [ ]:
# osmotic-coupling-mechanism
show_viz(_render_one('local:OsmoticCouplingMechanism', {'composite': 'viva_genesis.composites.osmotic_growth_division.osmotic_growth_division', 'title': 'The osmotic coupling — why too permeable starves, too impermeable lyses'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division if 'ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division' in dir() else None)))

**growth-division-map**


In [ ]:
# growth-division-map
show_viz(_render_one('local:GrowthDivisionMap', {'composite': 'viva_genesis.composites.osmotic_growth_division.osmotic_growth_division', 'grid_json': 'workspace/studies/og-growth-division/artifacts/growth_division_grid.json', 'title': 'Metabolism → growth & division — the viable band'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division if 'ov_viva_genesis_composites_osmotic_growth_division_osmotic_growth_division' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| net-growth-to-division _(primary)_ | kind=derived path=growth_division_count detail=cumulative division count in the favorable (matched) arm | op > value 0.0 detail the compartment divides (>0) in the matched throughput/permeability regime |
| mass-conserved-across-division _(primary)_ | kind=derived path=osmotic_balance_residual_max detail=worst-case |osmolyte mass-balance residual| over the run (mol) | op < value 1e-18 detail mass conserved across the growth/division cycle (residual below a negligible mole tolerance) |
| division-not-lysis-or-starvation _(primary)_ | kind=cross_run detail=matched (divide) vs too-permeable (starve) and too-impermeable (lyse) arms | op arms_reported detail matched divides; too permeable = starve (no division no growth); too impermeable = lyse (viable band is bounded) None |
| ablation-no-throughput-no-growth _(primary)_ | kind=cross_run detail=favorable (source on) vs zero-throughput ablation (source off) | op controls_absent detail favorable divides; zero-throughput ablation shows no division and no net growth |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Internal product accumulation generates osmotic pressure that drives compartment growth and division; fatty-acid / mixed-amphiphile vesicles grow and divide under osmotic and physico-chemical forcing without dedicated division machinery. | viable growth→division band over (throughput × permeability) | Jordan 2019 (vesicle feasibility); Muchowska 2019 (autocatalytic rTCA source); osmotic-growth vesicle division (Nat. Chem. 2023, pending reference) | Reproduced as a coarse-grained, units-typed, mass-conserving osmotic-coupling model — the matched regime grows and divides; the band is bounded by starvation and lysis. |


## Study: Calibrating the Vent: What Fraction of Observed-Consistent Vent States Feed Downstream Chemistry (`og-cal-vent-environment`)

**Question.** Of the real-vent states consistent with the Lost City field observations (outlet temperature, pH, dissolved H2), what fraction also satisfies what downstream carbon fixation needs (H2 >= 1 mM, a >= 3-unit pH gradient, T in range), and which observation does the actual constraining?

**Objective.** Run the environmental-calibration pipeline on the real geochemical world: LHS prior from the vg-data package vent_environment -> ensemble per spatial-transport model class -> ABC conditioning on the field windows -> posterior-predictive fraction meeting the downstream requirements -> Morris screening -> the Feasibility / Robustness / Discrimination report with the ledger verified.

**Hypothesis.** A Lost-City-consistent posterior exists inside the empirically motivated prior of the og-00 world, most of it also satisfies the downstream requirements, the result is the same under the convecting and the prescribed-upflow transport classes, and the conservation ledger closes exactly on every draw.

**Claim.** Conditioning the real PHREEQC + FiPy vent on the Lost City field windows, 12.5% of the empirically motivated prior (6 of 48 draws) reproduces all three observations, and 100% of that posterior supplies the H2 and pH gradient downstream carbon fixation requires. The dissolved-H2 window does all the constraining: pH is buffered at 9.4-9.9 by the peridotite assemblage and carries no information, temperature is loose, and the answer is the same whether or not the vent wall convects. The conservation ledger closes exactly on all 96 runs. This is a geochemical calibration, not an origin-of-life claim, and the H2 threshold rests on a full-redox-equilibrium assumption and an extrapolated DIC prior that a kinetic carbon-redox class could still shift.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- A: serpentinization as boundary flux (PHREEQC speciation)
- B: flow/mixing (convecting vs prescribed-upflow transport classes)

**Proposed models / methods**

- vg-data prior package (value-or-distribution + provenance class per parameter)
- Latin-hypercube ensemble through the real PHREEQC + FiPy geochemical_world per model class
- ABC rejection on field observation windows; posterior-predictive requirement mask
- Morris elementary-effects screening; paired class differences; conservation ledger

**Key assumptions**

- The prior is the vg-data package vent_environment (six free parameters; provenance audit: 0 measured, 2 calibrated, 1 inferred, 3 extrapolated, 2 hypothetical).
- Uniform likelihood on observation windows (ABC rejection): a vent state is consistent when every readout lies inside its window.
- Full redox equilibrium each step (PHREEQC, llnl.dat): DIC is reduced before free H2 appears; a kinetic carbon-redox class is the planned alternative.
- Downstream requirements are og-01's favorable regime (H2 >= 1 mM, delta pH >= 3 against pH 6.0, T <= 100 C), classed inferred.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Geochemical world (vent-calibration operating point)` | `viva_genesis.composites.geochemical_world.geochemical_world` | 0 | — |

Declared parameter sets (`study.yaml` variants):

- **uniform-upflow-transport** — `model_spatial_transport=fipy_uniform_upflow`
- **reduced-serpentinization** — `model_serpentinization=reduced`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.geochemical_world.geochemical_world`**

og-00's real serpentinizing vent (genuine PHREEQC peridotite speciation feeding a genuine FiPy finite-volume advection-diffusion solve) rebuilt as the Phase-1 geochemical world: both mechanisms are selected through typed faces (model_serpentinization: phreeqc | reduced; model_spatial_transport: fipy convecting | fipy_uniform_upflow | fenics), so the model CLASS is a categorical sweep axis beside the physical parameters, and a ConservationLedgerProcess audits integrated sources against accumulated pools, charge balance, flux divergence and heat directionality (ledger/ledger_ok). The free parameters are the vg-data package 'vent_environment' (value-or-distribution, units, context, source, provenance class), which the environmental-calibration ensemble samples.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_geochemical_world_geochemical_world = {
    'model_serpentinization' : 'phreeqc',   # Model class behind the 'serpentinization' face: phreeqc | reduced
    'model_spatial_transport': 'fipy',   # Model class behind the 'spatial_transport' face: fenics | fipy | fipy_uniform_upflow
    'flow_velocity'          : 0.001,   # Boundary water flux through the reaction zone (m/s) [vg-data: inferred]
    'olivine_rate'           : 0.15,   # Olivine reaction extent per flow-saturated step [vg-data: hypothetical]
    'pyroxene_fraction'      : 0.5,   # Diopside per mole olivine (peridotite vs dunite) [vg-data: calibrated]
    'fe_fraction'            : 0.25,   # Fayalite fraction of the dissolving olivine [vg-data: extrapolated]
    'heat_to_fluid_fraction' : 0.0008,   # Serpentinization enthalpy retained in the fluid [vg-data: calibrated]
    'fluid_dic'              : 0.002,   # Inflowing-fluid DIC, Hadean-ocean proxy (molar) [vg-data: extrapolated]
    'nx'                     : 40,   # FiPy finite-volume cells across the interface column
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `flow_velocity` | float | meter/second | >= 0.0 |
| `olivine_rate` | float | dimensionless | >= 0.0 |
| `fluid_dic` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `H2_flux` | float | mole/second | >= 0.0 |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `pH` | float | dimensionless | [0.0, 14.0] |

**Conditions**

- `mass_balance` (invariant): `ledger.H2_residual == 0 and ledger.alkalinity_residual == 0`
- `charge_neutrality` (invariant): `abs(charge_imbalance) < 1e-6`
- `spatial_flux_conservation` (invariant): `flux_divergence_residual < 1e-3`
- `energy_directionality` (invariant): `ledger.energy_directionality == 1`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-cal-vent-environment ===
STUDY = 'og-cal-vent-environment'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Geochemical_world_vent_calibration_operating_point = 100
INTERVAL_Geochemical_world_vent_calibration_operating_point = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_geochemical_world_geochemical_world if 'ov_viva_genesis_composites_geochemical_world_geochemical_world' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_geochemical_world_geochemical_world = materialize_composite('viva_genesis.composites.geochemical_world.geochemical_world', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_geochemical_world_geochemical_world['state']}, core=core)
            _comp.run(STEPS_Geochemical_world_vent_calibration_operating_point)
        print('ran', 'Geochemical world (vent-calibration operating point)', f'(viva_genesis.composites.geochemical_world.geochemical_world) for', STEPS_Geochemical_world_vent_calibration_operating_point, 'steps')
    except Exception as _e:
        print(f"Geochemical world (vent-calibration operating point) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**vent-environment-posterior**


In [ ]:
# vent-environment-posterior
show_viz(_render_one('local:VentEnvironmentPosterior', {'title': 'Lost-City-conditioned vent posterior (Decision view)', 'ensemble_json': 'workspace/studies/og-cal-vent-environment/artifacts/vent_environment_ensemble.json', 'report_json': 'workspace/studies/og-cal-vent-environment/artifacts/vent_environment_report.json'}, RUNS_DB, STUDY_YAML))

**vent-reactor-timeseries**


In [ ]:
# vent-reactor-timeseries
show_viz(_render_one('local:VentReactorTimeseries', {'composite': 'viva_genesis.composites.geochemical_world.geochemical_world', 'n_steps': 120, 'title': 'Geochemical world at the operating point (World view)'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_geochemical_world_geochemical_world if 'ov_viva_genesis_composites_geochemical_world_geochemical_world' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| ledger-closes _(primary)_ | kind=derived path=max_ledger_mass_residual detail=worst-case |pool - integral| / integral over H2 and alkalinity | op <= value 1e-12 detail matter ledger residual at machine precision |
| energy-ledger-closes _(primary)_ | kind=derived path=max_ledger_energy_residual detail=worst-case |heat pool - integrated heat| / integral | op <= value 1e-12 detail energy ledger residual at machine precision |
| heat-released-not-absorbed _(diagnostic)_ | kind=derived path=energy_directionality_final detail=1 while the heat source has never been negative | op >= value 1.0 detail directionality witness holds through the run |
| charge-conserved _(diagnostic)_ | kind=derived path=max_abs_charge_imbalance detail=worst-case |charge imbalance| over the run | op <= value 1e-06 detail |charge imbalance| < 1e-6 throughout |
| dissolved-h2-in-lost-city-range _(primary)_ | kind=derived path=settled_dissolved_h2_mM detail=settled H2_flux / (flow_velocity x flux_area) mmol/kg=None | op range low 1.0 high 15.0 detail dissolved H2 in [1, 15] mmol/kg |
| posterior-viable-vent-fraction _(primary)_ | kind=derived path=posterior_viable_fraction detail=viable | consistent, headline transport class, from artifacts/vent_environment_report.json | op > value 0.0 detail posterior-viable fraction > 0 |
| structure-agreement-across-transport-classes _(primary)_ | kind=derived path=structure_agreement_viable detail=fraction of identical draws on which every class agrees about viability | op >= value 0.8 detail cross-class agreement >= 0.8 |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Lost City end-member fluids 40-91 C, pH 9-10.8, dissolved H2 1-15 mmol/kg | settled outlet T, pH; dissolved H2 backed out of the H2 flux | Kelley 2001 / Kelley 2005 / Ludwig 2006 / Proskurowski 2008 | Used as the ABC conditioning windows (vg-data vent_environment.observations); 6/48 prior draws reproduce all three. |


## Study: The Vent Wall as a Galvanic Cell: Which Limitation Stops Carbon Fixation, and Do the Model Classes Agree? (`og-01-cal-electrochemistry`)

**Question.** Treating the vent wall as an explicit galvanic cell (Nernst potentials, Butler-Volmer kinetics), do the empirical and the electrochemical model classes agree on which limitation (thermodynamic, electrical, catalytic, transport) stops carbon fixation across the region both call favorable - the pre-registered test being at least 80% agreement on the limitation label?

**Objective.** Build the carbon_fixation face with three classes (empirical = og-01's kinetics; butler_volmer = Nernst + Butler-Volmer galvanic cell, the primary new class; microkinetic = CatMAP/Cantera, pixi-only), the MineralConductivityProcess that derives gradient_efficiency from circuit physics, the ChargeLedgerProcess (Kirchhoff residual per tick, faradaic efficiency <= 1, current-voltage residual) and the vg-data package electrochemical_interface; sweep (pH x H2 x conductivity x Ni) x classes in open flow; attribute the limitation at every point; measure cross-class agreement; report Feasibility / Robustness / Discrimination.

**Hypothesis.** Explicit electrochemistry reproduces the Altair-2025 current window at the nominal operating point with the Preiner product hierarchy, derives og-01's gradient efficiency from the wall's conductivity (an insulating wall collapses it, the Tutolo-2026 arm, as a consequence rather than a knob), closes a charge + electron ledger exactly, and labels every swept point with its dominant limitation. Pre-registered gate: the classes agree on the label on >= 80% of the region both call favorable.

**Claim.** No. The two classes agree on the dominant-limitation label on only 47% of the favorable region, below the pre-registered 80% gate, so the gate fails and the attribution is structure-sensitive. Under explicit electrochemistry the vent cell is activation (catalyst) limited everywhere the wall conducts and electrical only below ~1e-6 S/m, but the empirical rate law's single Michaelis drive term lumps electrical and catalytic limitation together and invents a transport limit the cell physics never reaches. The electrochemistry itself is real, calibrated to the Altair-2025 current window, and ledger-exact on all 750 runs; what the study cannot yet deliver is a structure-independent answer to why fixation stops, recorded as an open decision for the expert rather than dressed up as a result.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- A: empirical gated kinetics (og-01) with the gradient efficiency derived from the wall circuit
- B: the wall as a galvanic cell — Nernst potentials, Butler-Volmer kinetics, ohmic mineral + ionic path, faradaic split
- C: adsorbate microkinetics with site coverage and passivation (deferred; pixi)

**Proposed models / methods**

- carbon_fixation face with the model class as a categorical sweep axis
- Nernst + Butler-Volmer galvanic-cell solve (Brent) per tick; Koutecky-Levich transport limits; Bruggeman ionic path
- Loss-fraction partition of the fresh-fluid emf -> limitation label per point; cross-class agreement + confusion
- Charge + electron ledger (Kirchhoff residual, faradaic efficiency <= 1, current-voltage residual)

**Key assumptions**

- Steady galvanic cell each tick: the CO2/HCOOH couple sets the cathode emf (acetate and pyruvate are side branches of the same current); H2 and CO2 are boundary feeds; standard potentials at 25 C.
- Exchange currents calibrated to the Altair current window, not to per-condition curves; faradaic efficiency set by the competing H2-evolution branch and proton transport at pH 6.
- The empirical class is og-01's rate law unchanged, with the catalyst scaled by the same Ni gain and the gradient efficiency derived from the circuit.
- Favorable region = throughput >= 10% of each class's own maximum; label = largest loss fraction (FE < 0.5 -> catalytic).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Electrochemical interface — Butler-Volmer, open flow` | `viva_genesis.composites.electrochemical_interface.electrochemical_carbon_fixation` | 0 | model_carbon_fixation=butler_volmer, dilution_rate=2.0 |

Declared parameter sets (`study.yaml` variants):

- **empirical-class** — `model_carbon_fixation=empirical`, `dilution_rate=2.0`
- **microkinetic-class-pixi-only** — `model_carbon_fixation=microkinetic`, `dilution_rate=2.0`
- **insulating-wall** — `model_carbon_fixation=butler_volmer`, `dilution_rate=2.0`, `conductivity=1e-09`
- **iron-only-catalyst** — `model_carbon_fixation=butler_volmer`, `dilution_rate=2.0`, `ni_fraction=0.0`
- **control-no-catalyst** — `model_carbon_fixation=butler_volmer`, `dilution_rate=2.0`, `catalyst_active=0.0`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.electrochemical_interface.electrochemical_carbon_fixation`**

og-01's H2/pH-driven carbon fixation rebuilt as the Phase-2 electrochemical interface: the carbon-fixation mechanism is a model CLASS on the carbon_fixation face (model_carbon_fixation: empirical | butler_volmer | microkinetic). The empirical class is og-01's lab-calibrated gated kinetics driven by the real FiPy Nernst-Planck proton gradient + proton budget, with gradient_efficiency DERIVED from the wall's circuit (MineralConductivityProcess: conductivity, thickness, porosity vs the electrodes' charge-transfer resistance) instead of a knob. The butler_volmer class solves the vent wall as a galvanic cell each tick (Nernst half-cell potentials from the fluid redox state, Butler-Volmer kinetics scaled by catalyst activity / Ni fraction / T, Koutecky-Levich transport limits, the ohmic mineral + ionic path, the faradaic split over formate / acetate / pyruvate calibrated to the Altair-2025 current window and the Preiner-2020 yield hierarchy). The microkinetic class (CatMAP/Cantera) builds here and runs under pixi. Every class publishes the thermodynamic / electrical / catalytic / transport loss fractions and a limitation code, and a ChargeLedgerProcess audits its electron balance (Kirchhoff residual, faradaic efficiency <= 1, current-voltage residual -> ledger/charge_ledger_ok). Free parameters are the vg-data package 'electrochemical_interface'.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_electrochemical_interface_electrochemical_carbon_fixation = {
    'model_carbon_fixation': 'butler_volmer',   # Model class behind the 'carbon_fixation' face: butler_volmer | empirical | microkinetic
    'ph_vent'              : 9.5,   # Alkaline vent-side pH (from og-00)
    'ph_ocean'             : 6.0,   # Acidic early-ocean-side pH
    'h2'                   : 0.005,   # Dissolved H2 feed, vent side (M)
    'co2'                  : 0.002,   # Dissolved CO2 feed, ocean side (M)
    'temperature'          : 373.15,   # Interface temperature (K)
    'conductivity'         : 0.001,   # Electronic conductivity of the mineral layer (S/m) [vg-data: extrapolated]
    'layer_thickness'      : 0.001,   # Mineral layer thickness (m) [vg-data: inferred]
    'ni_fraction'          : 0.75,   # Ni mole fraction of the Fe/Ni catalytic phase (0 greigite .. 0.75 awaruite) [vg-data: measured]
    'porosity'             : 0.35,   # Wall porosity (ionic return path)
    'catalyst_active'      : 1.0,   # Active Fe/Ni/S mineral catalyst fraction (0 = no-catalyst control)
    'j0_co2'               : 1e-05,   # CO2-reduction exchange current density at Ni3Fe, 100 C (A/m^2) [vg-data: calibrated]
    'boundary_layer'       : 0.001,   # Diffusion boundary layer at the electrodes (m) [vg-data: inferred]
    'dilution_rate'        : 0.0,   # Open-flow washout (1/s); 0 = closed batch, >0 = sustained-throughput regime
    'nx'                   : 20,   # FiPy cells across the Nernst-Planck barrier
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `ph_vent` | float | dimensionless | [0.0, 14.0] |
| `h2` | float | molar | >= 0.0 |
| `conductivity` | float | siemens/meter | >= 0.0 |
| `ni_fraction` | float | dimensionless | [0.0, 1.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `net_carbon_flux` | float | mole/second | >= 0.0 |
| `current_density` | float | ampere/meter**2 | >= 0.0 |
| `faradaic_efficiency` | float | dimensionless | [0.0, 1.0] |

**Conditions**

- `charge_conservation` (invariant): `ledger.charge_residual <= 1e-9 every tick`
- `faradaic_bound` (invariant): `ledger.faradaic_efficiency <= 1`
- `current_voltage_consistency` (invariant): `ledger.iv_residual_max <= 1e-6`
- `loss_partition` (invariant): `loss_thermodynamic + loss_electrical + loss_catalytic + loss_transport == 1 while current flows (butler_volmer)`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-01-cal-electrochemistry ===
STUDY = 'og-01-cal-electrochemistry'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Electrochemical_interface_Butler_Volmer_open_flow = 100
INTERVAL_Electrochemical_interface_Butler_Volmer_open_flow = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_electrochemical_interface_electrochemical_carbon_fixation if 'ov_viva_genesis_composites_electrochemical_interface_electrochemical_carbon_fixation' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_electrochemical_interface_electrochemical_carbon_fixation = materialize_composite('viva_genesis.composites.electrochemical_interface.electrochemical_carbon_fixation', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_electrochemical_interface_electrochemical_carbon_fixation['state']}, core=core)
            _comp.run(STEPS_Electrochemical_interface_Butler_Volmer_open_flow)
        print('ran', 'Electrochemical interface — Butler-Volmer, open flow', f'(viva_genesis.composites.electrochemical_interface.electrochemical_carbon_fixation) for', STEPS_Electrochemical_interface_Butler_Volmer_open_flow, 'steps')
    except Exception as _e:
        print(f"Electrochemical interface — Butler-Volmer, open flow not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**electrochemistry-mechanism-map**


In [ ]:
# electrochemistry-mechanism-map
show_viz(_render_one('local:ElectrochemistryMechanismMap', {'title': 'The vent wall as a galvanic cell — an honest negative', 'mechanism_map_json': 'workspace/studies/og-01-cal-electrochemistry/artifacts/electrochemistry_mechanism_map.json', 'report_json': 'workspace/studies/og-01-cal-electrochemistry/artifacts/electrochemistry_report.json'}, RUNS_DB, STUDY_YAML))

**electrochemistry-polarization**


In [ ]:
# electrochemistry-polarization
show_viz(_render_one('local:ElectrochemistryPolarization', {'title': 'Vent wall as a galvanic cell at the nominal point (Mechanism view)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| failure-mode-labelled _(primary)_ | kind=derived path=failure_mode_label_coverage detail=min over classes of the fraction of points with a finite label | op >= value 1.0 detail label coverage 1.0 in every class |
| failure-mode-attributed _(primary)_ | kind=derived path=failure_mode_agreement_favorable detail=fraction of favorable-both points with identical labels | op >= value 0.8 detail cross-class label agreement >= 0.8 in the favorable region |
| mechanism-map-ledger-closes _(primary)_ | kind=derived path=mechanism_map_ledger_all_ok detail=1 when charge_ledger_ok on every point | op >= value 1.0 detail ledger ok on all points |
| charge-ledger-closes _(primary)_ | kind=derived path=charge_ledger_residual_max detail=worst-case charge residual | op <= value 1e-09 detail charge residual within tolerance |
| faradaic-efficiency-bounded _(primary)_ | kind=derived path=faradaic_efficiency_max detail=worst-case faradaic efficiency over the run | op <= value 1.000001 detail FE <= 1 (1e-6 rounding allowance) |
| current-voltage-residual _(diagnostic)_ | kind=derived path=iv_residual_max detail=worst-case |residual| / emf_fresh | op <= value 1e-06 detail circuit residual below 1e-6 |
| current-in-altair-window _(primary)_ | kind=derived path=settled_current_density detail=settled cell current density | op range low 1e-05 high 0.01 detail j in [1e-5, 1e-2] A/m^2 |
| selectivity-to-carbon _(diagnostic)_ | kind=derived path=settled_faradaic_efficiency detail=settled fraction of the current reducing CO2 | op >= value 0.5 detail faradaic efficiency to carbon >= 0.5 |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Altair-2025 macroscale reactor currents ~10 nA .. 10 uA sufficient for carbon reduction | settled Butler-Volmer current density on a 1e-3 m^2 interface | Altair et al. 2025 (campaign dossier ref 12; parameter-registry electrochemical_current) | Used as the calibration window of the exchange currents (vg-data electrochemical_interface.observations.cell_current_density); 79.7% of the grid inside it. |
| Preiner 2020 product hierarchy formate >> acetate >> pyruvate, awaruite (Ni3Fe) the most productive catalyst | electron branch fractions and the Ni gain of the CO2-reduction exchange current | Preiner et al. 2020 | Branch fractions inferred from the yield hierarchy; the Ni gain is a hypothetical shape fitted to the ordering. |


## Study: The Open Network: Do Autocatalytic Motifs Emerge From Rule-Generated Chemistry, Beyond Chance? (`og-02-alt-reaction-universe`)

**Question.** If the prebiotic reaction network is generated from reaction rules rather than hand-curated, with real eQuilibrator thermodynamics and fed the calibrated vent state, do persistent autocatalytic motifs appear at all, and are they more frequent than in matched null networks?

**Objective.** Build the proto_metabolism face (curated_rtca | reaction_universe | null_network), the vg-data package reaction_universe (rules, eQuilibrator dG', catalyst requirements), the NetworkLedgerProcess (exact element residual, integrated carbon balance, dissipation >= 0) and a class-agnostic post-hoc motif detector; run the three classes under two feeds (vent posterior; + 10 mM formaldehyde) with 24 matched nulls per arm, an ignition map, flux-control analysis and a stiffness check.

**Hypothesis.** Under the vent posterior feed (CO2 1.4 mM, H2 10.8 mM, pyruvate 0.1 mM) the generated universe sustains a connected multi-species network but no autocatalytic motif (the C1 -> C2 bottleneck is thermodynamic at vent concentrations); supplying the formose feedstock (formaldehyde) lets a motif emerge from the rules and that motif count is enriched relative to degree-preserving, carbon/oxygen-matched nulls; the curated class's motif is detected but imposed and its stoichiometry fails the element ledger.

**Claim.** Fed the calibrated vent posterior, the rule-generated universe (121 species, 413 reactions) sustains a persistent 14-species network but contains no autocatalytic motif, because every C-C-bond-forming route is uphill at vent concentrations. Supplying 10 mM formaldehyde makes exactly one motif emerge - the formose cycle, recovered post hoc and absent in all 24 matched null networks (enrichment p = 0.04). Graded by the same detector, og-02's curated rTCA motif turns out to be imposed, and its lumped turn is not element-balanced, creating 4 carbon atoms per cycle, while the generated universe's ledger closes exactly. The study demonstrates that autocatalysis is not free under real thermodynamics, but the one motif that ignites needs a formaldehyde feed the vent chemistry does not itself produce.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- E: protometabolic network formation (rule-generated, not curated)
- D: surface chemistry as catalyst classes with finite site pools

**Proposed models / methods**

- Rule-based network generation (Wolos 2020 style) over linear C/H/O skeletons
- eQuilibrator component-contribution dG' + group-additivity fallback; detailed-balance kinetics
- libRoadRunner CVODE integration; post-hoc autocatalytic-core enumeration (Blokhuis 2020)
- Degree-preserving (C,O)-matched null networks (Maslov-Sneppen); empirical enrichment
- Element + free-energy ledger; flux-control summation theorem; tolerance / stiffness check

**Key assumptions**

- Linear C/H/O skeletons up to C4; eleven reaction rules (H2 reductions, enol / reductive carboxylation, formose initiation, aldol, isomerisation, alpha/beta decarboxylation, Cannizzaro); every reaction reversible with k_down = k0 (rule class) and k_up = k0 exp(-|dG'0|/RT).
- dG' potentials at pH 9.38 / I 0.25 M / 322 K (the vent posterior example point); water activity 1; CO2 and H2 vent-buffered, organics as CSTR inflow.
- Catalyst classes (mineral: reductions / carboxylations; base: aldol / isomerisation / Cannizzaro) share a finite site pool f = sites / (1 + load / K_sat), updated per tick.
- Null = degree-preserving swaps of organic species within (C, O)-count classes; dG' recomputed from the same potentials; same exchange / degradation reactions.
- A persistent autocatalytic motif = a simple cycle of flux-carrying reactions (|v| > 1e-12 M/s, species > 1e-9 M) with stoichiometric gain > 1 (Blokhuis 2020 type-I core); currency species CO2 / H2 / H2O excluded as cycle members.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Open chemical network — reaction_universe at the vent feed` | `viva_genesis.composites.open_chemical_network.open_chemical_network` | 0 | model_proto_metabolism=reaction_universe |

Declared parameter sets (`study.yaml` variants):

- **formose-feed** — `feed_formaldehyde=0.01`
- **null-network** — `model_proto_metabolism=null_network`, `null_seed=0`
- **curated-rtca** — `model_proto_metabolism=curated_rtca`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.open_chemical_network.open_chemical_network`**

og-02's continuous-feed reactor rebuilt as the Phase-3 open chemical network: the reaction network is a model CLASS on the proto_metabolism face (model_proto_metabolism: curated_rtca | reaction_universe | null_network), all three integrated by the genuine libRoadRunner engine under the same Phase-1-posterior feed (CO2, H2, pyruvate), graded by one class-agnostic post-hoc detector (flux-carrying simple cycles with stoichiometric gain > 1) and audited by a NetworkLedgerProcess (exact element residual, integrated carbon balance, dissipation >= 0). The universe class is generated from Wolos-2020-style rules over linear C/H/O skeletons with eQuilibrator dG' (vg-data package reaction_universe); the null class is its degree-preserving, carbon-matched rewiring.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_open_chemical_network_open_chemical_network = {
    'model_proto_metabolism': 'reaction_universe',   # Model class behind the 'proto_metabolism' face: curated_rtca | null_network | reaction_universe
    'feed_CO2'              : 0.0014,   # Vent-buffered CO2 set-point (M) [vg-data: calibrated, Phase-1 posterior]
    'feed_H2'               : 0.0108,   # Vent-buffered H2 set-point (M) [vg-data: calibrated, Phase-1 posterior]
    'feed_pyruvate'         : 0.0001,   # Pyruvate CSTR inflow (M) from og-01 reduced carbon [vg-data: inferred]
    'feed_formaldehyde'     : 0.0,   # Formaldehyde CSTR inflow (M); 0 canonical, 1e-3 in the formose arm [vg-data: hypothetical]
    'k_wash'                : 0.05,   # CSTR washout / dilution rate (1/s) [vg-data: inferred]
    'null_seed'             : 0,   # Null-network draw (null_network class only)
    'max_carbons'           : 4,   # Carbon cap of the generated universe (reaction_universe / null_network) [vg-data: hypothetical]
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `feed_CO2` | float | molar | >= 0.0 |
| `feed_H2` | float | molar | >= 0.0 |
| `feed_pyruvate` | float | molar | >= 0.0 |
| `feed_formaldehyde` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `net_carbon_flux` | float | mole/second |  |
| `autocatalytic_gain` | float | dimensionless | >= 0.0 |
| `network_connected` | float | dimensionless | [0.0, 1.0] |

**Conditions**

- `element_balance` (invariant): `ledger.element_residual_max == 0`
- `carbon_balance` (invariant): `ledger.carbon_residual_max < 1e-4`
- `energy_directionality` (invariant): `ledger.free_energy_directionality == 1`
- `motifs_post_hoc` (invariant): `no rule names a cycle; motifs are detected by viva_genesis.protometabolism.motifs on every class`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-02-alt-reaction-universe ===
STUDY = 'og-02-alt-reaction-universe'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Open_chemical_network_reaction_universe_at_the_vent_feed = 100
INTERVAL_Open_chemical_network_reaction_universe_at_the_vent_feed = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_open_chemical_network_open_chemical_network if 'ov_viva_genesis_composites_open_chemical_network_open_chemical_network' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_open_chemical_network_open_chemical_network = materialize_composite('viva_genesis.composites.open_chemical_network.open_chemical_network', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_open_chemical_network_open_chemical_network['state']}, core=core)
            _comp.run(STEPS_Open_chemical_network_reaction_universe_at_the_vent_feed)
        print('ran', 'Open chemical network — reaction_universe at the vent feed', f'(viva_genesis.composites.open_chemical_network.open_chemical_network) for', STEPS_Open_chemical_network_reaction_universe_at_the_vent_feed, 'steps')
    except Exception as _e:
        print(f"Open chemical network — reaction_universe at the vent feed not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**reaction-universe-enrichment**


In [ ]:
# reaction-universe-enrichment
show_viz(_render_one('local:ReactionUniverseEnrichment', {'title': 'Motif enrichment vs matched nulls, and where motifs ignite (Decision view)', 'report_json': 'workspace/studies/og-02-alt-reaction-universe/artifacts/motif_enrichment.json'}, RUNS_DB, STUDY_YAML))

**reaction-universe-subnetwork**


In [ ]:
# reaction-universe-subnetwork
show_viz(_render_one('local:ReactionUniverseSubnetwork', {'title': 'The persistent subnetwork and its post-hoc formose cycle (Mechanism view)', 'report_json': 'workspace/studies/og-02-alt-reaction-universe/artifacts/motif_enrichment.json', 'arm': 'formose_feed'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| network-ledger-closes _(primary)_ | kind=derived path=network_ledger_ok_final detail=ledger_ok on the final tick | op >= value 1.0 detail every ledger witness within tolerance |
| element-residual-exact _(diagnostic)_ | kind=derived path=max_element_residual detail=worst-case stoichiometric element residual | op <= value 1e-12 detail element balance exact by construction |
| free-energy-directionality _(diagnostic)_ | kind=derived path=free_energy_directionality_final detail=1 while dissipation has never been below -1e-12 W | op >= value 1.0 detail directionality witness holds through the run |
| persistent-network-at-phase1-feed _(primary)_ | kind=derived path=final_n_persistent_species detail=organic species above conc_floor on the final tick | op >= value 10.0 detail >= 10 persistent species |
| rtca-branch-emerges-from-rules _(supporting)_ | kind=derived path=settled_malate_M detail=settled malate (last quartile) | op > value 1e-09 detail malate persists above the species floor |
| no-motif-at-phase1-feed _(diagnostic)_ | kind=derived path=universe_motif_count detail=universe motif count, arm phase1_feed, from artifacts/motif_enrichment.json | op >= value 0.0 detail informative readout (any value passes) |
| motifs-emerge-not-imposed _(primary)_ | kind=derived path=motif_enrichment_p detail=empirical p of the universe motif count vs 24 nulls, arm formose_feed | op <= value 0.05 detail enrichment p <= 0.05 (with >= 1 motif see motif-present-formose-arm) None |
| motif-present-formose-arm _(primary)_ | kind=derived path=universe_motif_count detail=universe motif count, arm formose_feed | op >= value 1.0 detail >= 1 persistent motif |
| curated-rtca-fails-element-ledger _(diagnostic)_ | kind=derived path=curated_element_residual detail=max|E S| over the curated SBML (post hoc) | op > value 0.0 detail residual > 0 documents the violation (expected 8) |
| flux-control-summation-theorem _(diagnostic)_ | kind=derived path=fcc_summation_residual_max detail=from artifacts/reaction_universe_verification.json | op <= value 0.01 detail summation residual <= 1e-2 on dominant fluxes |
| integrator-tolerance-insensitive _(diagnostic)_ | kind=derived path=tolerance_sensitivity_max detail=worst relative species deviation across tolerance settings | op <= value 0.01 detail <= 1% relative deviation |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| The formose reaction is autocatalytic in glycolaldehyde via aldol / isomerisation / retro-aldol of the tetrose, with a slow umpolung initiation from formaldehyde alone | the detected motif's species and reactions | Breslow 1959 | Recovered post hoc from the rule set (glycolaldehyde -> glyceraldehyde -> DHA -> erythrulose -> erythrose -> 2 glycolaldehyde, gain 2) without being named by any rule. |
| Rule-generated prebiotic networks contain self-regenerating cycles that appear only after several synthetic generations | motif count vs matched nulls | Wolos 2020 | One cycle at C <= 4 in C/H/O; enrichment p = 0.04 over 24 nulls. |


## Study: The Endogenous Boundary (`og-04-alt-endogenous-boundary`)

**Question.** Can a protocell boundary form from the campaign's own reduced-carbon chemistry - an amphiphile synthesized from acetate + formate that clears the critical aggregation concentration - rather than from an injected amphiphile pool, and is its formation caused by that synthesis?

**Objective.** Drive AmphiphileSynthesisProcess from a dissolved acetate+formate pool into the boundary face (single_chain | mixed | thioester_aminothiol); gate that a BoundaryFormationStep _add produces agents 0 -> >=1 in the favorable arm and 0 in the synthesis-ablated control, with the RewriteLedger confirming the rewrite conserves matter.

**Hypothesis.** When acetate + formate condense to an amphiphile that clears the critical aggregation concentration under vesicle-favourable (alkaline, saline) conditions, a viable boundary forms and a new individual appears in the agents map by an _add structural rewrite; with amphiphile synthesis ablated no individual forms, so boundary formation is caused by the endogenous chemistry.

**Claim.** Vent-fed chemistry, through explicit chain-extension, headgroup and degradation reactions across multiple amphiphile species, either reaches an experimentally-grounded aggregation-competent composition on its own — assembling via the real og-31 / viva-openmm membrane closure — or it does not, in which case boundary formation is not supported by defensible endogenous kinetics. The single assumed acetate+formate→amphiphile condensation is retired in favour of named reactions with tracked rates and losses.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- H+: the boundary is built from the campaign's own carbon chemistry, not an added amphiphile
- structural-rewrite individuation (_add on the agents map) rather than a continuous field

**Proposed models / methods**

- AmphiphileSynthesisProcess (acetate + formate -> fatty-acid amphiphile)
- boundary face (Jordan-2019 mixed-amphiphile vesicle; thioester/aminothiol alternative)
- BoundaryFormationStep (_add structural rewrite) + RewriteLedgerProcess (conservation)


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Endogenous amphiphiles by explicit chain extension` | `viva_genesis.composites.og_17_amphiphile_chain_extension.og_17_amphiphile_chain_extension` | 0 | acetate_feed=0.05, k_extend=0.4, cac_molar=0.005 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.og_17_amphiphile_chain_extension.og_17_amphiphile_chain_extension`**

Study 17: endogenous amphiphiles by explicit chemistry. AmphiphileChainExtensionProcess elongates fatty acids 2 carbons at a time from the vent-carbon feed (acetate), with degradation and a length-dependent critical aggregation concentration — replacing the lumped acetate+formate condensation. Reports the chain-length distribution + whether the composition clears the CAC (aggregation_competent >= 1), the input og-31's Martini membrane run would assemble. No exogenous amphiphile feed; losses tracked.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_og_17_amphiphile_chain_extension_og_17_amphiphile_chain_extension = {
    'acetate_feed': 0.02,   # Vent-carbon (acetate, C2) feed concentration (molar)
    'k_extend'    : 0.4,   # Chain-elongation rate (1/s)
    'k_degrade'   : 0.02,   # Degradation/hydrolysis rate (1/s)
    'cac_molar'   : 0.005,   # Long-chain critical aggregation concentration (molar)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `k_synth` | float | 1/(molar*second) | >= 0.0 |
| `acetate0` | float | molar | >= 0.0 |
| `formate0` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `n_agents` | float | dimensionless | >= 0.0 |
| `boundary_formed` | float | dimensionless |  |
| `rewrite_ok` | float | dimensionless |  |

**Conditions**

- `rewrite_mass_balance`: `rewrite_residual_max == 0 (the _add moves matter localized -> individual, conserved)`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-04-alt-endogenous-boundary ===
STUDY = 'og-04-alt-endogenous-boundary'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Endogenous_amphiphiles_by_explicit_chain_extension = 100
INTERVAL_Endogenous_amphiphiles_by_explicit_chain_extension = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_17_amphiphile_chain_extension_og_17_amphiphile_chain_extension if 'ov_viva_genesis_composites_og_17_amphiphile_chain_extension_og_17_amphiphile_chain_extension' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_17_amphiphile_chain_extension_og_17_amphiphile_chain_extension = materialize_composite('viva_genesis.composites.og_17_amphiphile_chain_extension.og_17_amphiphile_chain_extension', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_17_amphiphile_chain_extension_og_17_amphiphile_chain_extension['state']}, core=core)
            _comp.run(STEPS_Endogenous_amphiphiles_by_explicit_chain_extension)
        print('ran', 'Endogenous amphiphiles by explicit chain extension', f'(viva_genesis.composites.og_17_amphiphile_chain_extension.og_17_amphiphile_chain_extension) for', STEPS_Endogenous_amphiphiles_by_explicit_chain_extension, 'steps')
    except Exception as _e:
        print(f"Endogenous amphiphiles by explicit chain extension not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**endogenous-boundary-formation**


In [ ]:
# endogenous-boundary-formation
show_viz(_render_one('local:EndogenousBoundaryViz', {'title': 'Boundary forms from chemistry (favorable vs synthesis-ablated)'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| boundary-forms-from-chemistry _(primary)_ | kind=derived path=boundary_forms_from_chemistry detail=favorable forms individuals (n_agents>=1) AND synthesis-ablated forms none; from artifacts/endogenous_boundary_report.json | op >= value 1.0 detail boundary caused by the endogenous chemistry — forms in the favorable arm, not in the synthesis-ablated control |
| rewrite-conserves-matter _(primary)_ | kind=derived path=endogenous_rewrite_conservation_ok detail=min over steps of the RewriteLedger conservation flag; from artifacts/endogenous_boundary_report.json | op >= value 1.0 detail rewrite_residual_max == 0 at every structural rewrite |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
|  |  |  |  |


## Study: Physical Reproduction (`og-growth-alt-physical-reproduction`)

**Question.** Can an individual physically reproduce - grow to a division threshold and divide into daughters that grow and divide in turn - building a multi-generation lineage with matter and volume conserved at every division?

**Objective.** Drive AgentGrowthProcess -> DivisionStep over the agents map from a single founder; gate that the lineage reaches >=3 generations and that every _divide conserves matter and volume (daughter volumes sum to the mother's — partition residual 0, total volume constant).

**Hypothesis.** A founder individual whose division_state grows to threshold divides by a _divide structural rewrite that halves its matter and volume; the daughters inherit half each (their amounts and volumes sum to the mother's) and grow and divide in turn, so a >=3-generation lineage forms in the agents map with the partition residual 0 at every division.

**Claim.** A founder individual grows to threshold and divides by a structural rewrite that halves its matter and volume, its daughters do the same, and the lineage reaches 5 generations in the agents map, past the 3-generation bar, with the partition residual exactly 0 at every division. Matter and volume are conserved throughout. What the model demonstrates is the conserving division-and-lineage mechanism; growth itself is abstracted to a division trigger rather than the full osmotic volume growth, which coupling the two would supply.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- H+: reproduction is a structural rewrite (_divide) on the agents map, not a counter
- conserving partition (FMP §9.1) — daughters'' amounts + volumes sum to the mother''s

**Proposed models / methods**

- AgentGrowthProcess (division_state -> threshold) + DivisionStep (_divide, matter + volume halved)
- LineageStatsProcess (deepest generation, total volume) + RewriteLedgerProcess (partition residual)


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Founder lineage — growth to a >=3-generation lineage` | `viva_genesis.composites.physical_reproduction.physical_reproduction` | 0 | growth_per_step=0.4, founder_volume=8e-12, founder_formate=8.0, founder_acetate=4.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.physical_reproduction.physical_reproduction`**

og-growth-alt: a founder individual grows to a division threshold and divides by a _divide structural rewrite (matter + volume halved); the daughters grow and divide in turn, building a >=3-generation lineage in the agents map. A RewriteLedgerProcess audits that every division conserves matter (partition residual 0) and LineageStatsProcess reports the deepest generation + the total volume, which stays constant across divisions (the daughters' volumes sum to the mother's).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_physical_reproduction_physical_reproduction = {
    'growth_per_step': 0.34,   # division_state added per step (growth to the division threshold) [vg-data: division_threshold]
    'founder_volume' : 8e-12,   # Founder individual volume (liter)
    'founder_formate': 8.0,   # Founder internal formate (mol)
    'founder_acetate': 4.0,   # Founder internal acetate (mol)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `growth_per_step` | float | dimensionless | >= 0.0 |
| `founder_volume` | float | liter | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `max_generation` | float | dimensionless | >= 0.0 |
| `total_volume` | float | liter | >= 0.0 |
| `partition_residual_max` | float | dimensionless |  |

**Conditions**

- `partition_conservation`: `partition_residual_max == 0 (daughter amounts + volumes sum to the mother)`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-growth-alt-physical-reproduction ===
STUDY = 'og-growth-alt-physical-reproduction'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Founder_lineage_growth_to_a_3_generation_lineage = 100
INTERVAL_Founder_lineage_growth_to_a_3_generation_lineage = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_physical_reproduction_physical_reproduction if 'ov_viva_genesis_composites_physical_reproduction_physical_reproduction' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_physical_reproduction_physical_reproduction = materialize_composite('viva_genesis.composites.physical_reproduction.physical_reproduction', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_physical_reproduction_physical_reproduction['state']}, core=core)
            _comp.run(STEPS_Founder_lineage_growth_to_a_3_generation_lineage)
        print('ran', 'Founder lineage — growth to a >=3-generation lineage', f'(viva_genesis.composites.physical_reproduction.physical_reproduction) for', STEPS_Founder_lineage_growth_to_a_3_generation_lineage, 'steps')
    except Exception as _e:
        print(f"Founder lineage — growth to a >=3-generation lineage not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**physical-reproduction-lineage**


In [ ]:
# physical-reproduction-lineage
show_viz(_render_one('local:PhysicalReproductionViz', {'title': 'A multi-generation lineage in the agents map'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| lineage-reaches-three-generations _(primary)_ | kind=derived path=peak_generation detail=deepest generation the lineage REACHES (peak over the run, not the time-mean); from artifacts/physical_reproduction_report.json | op >= value 3.0 detail a >=3-generation lineage forms (peak reaches generation 5) |
| division-conserves-volume _(primary)_ | kind=derived path=partition_residual_peak detail=worst (max) |mother - sum(daughters)| over every _divide; from artifacts/physical_reproduction_report.json | op <= value 1e-09 detail partition residual ~ 0 at every division (conserving split) |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
|  |  |  |  |


## Study: The Internalization Threshold: When Does Owning an Energy Store Pay, Per Coupling Class? (`og-energy-alt-internalization-threshold`)

**Question.** With the energy-coupling mechanism placed inside a protocell as five competing model classes, at what coupling activity does owning an internal energy store pay (its benefit exceeding its leak plus maintenance), and how does that threshold depend on the carrier chemistry and the membrane leak?

**Objective.** Build the energy_coupling face (passive_gradient | h_na_gradient_leaky_membrane | thioester_acyl_phosphate | polyphosphate | fe_redox) with the per-individual autonomy readout internal / (internal + external) power, the vg-data package energy_currencies (drives, H+/Na+ permeabilities, carrier bonds and decay, per-cell turnover and maintenance), the FreeEnergyLedgerProcess (free_energy_in >= work + dissipated with the store, exact every tick) and the protocell_energetics composite; sweep passive -> active coupling across the classes, the membrane leak and the gradient state, and find where internalization pays.

**Hypothesis.** Internalization pays only where the store can be charged above the load's free energy faster than it leaks: the thioester-class carrier (bond 35 kJ/mol, leaking by hydrolysis) pays from a low activity at the nominal membrane; the ion-gradient store drains through the Na+ leak at fatty-acid permeabilities and pays only in tight membranes; the polyphosphate store (bond 20 kJ/mol) cannot clear a 30 kJ/mol load unless nearly fully charged; the Fe-redox store sits near the load potential and is marginal. The threshold is thermodynamic (the store potential must clear the load), not a fitted rate, so the classes separate on their leak law and bond energy — a structure-sensitive answer the sweep reports as such.

**Claim.** Owning an energy store pays only where it can be charged above the load's free energy faster than it leaks, and which chemistry clears that bar at a given membrane is the real answer: a thioester/acyl-phosphate carrier internalizes from a tenth of the drive at fatty-acid-like leak, an Fe-redox store only near full activity (~0.80), a pumped H+/Na+ gradient only once the membrane is ~100x tighter to Na+ (the Sojo constraint, per individual), and a polyphosphate store not at all against a 30 kJ/mol load. The free-energy ledger closes to rounding on all 990 runs, so these are real thermodynamic thresholds rather than bookkeeping artefacts. But the answer is structure-sensitive - the classes agree on only 46% of the grid - and every per-cell number (drive turnover, maintenance, bond energies) is hypothetical and unswept beyond the nominal grid.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- H: bioenergetic internalization — inside the individual, as competing coupling classes
- energetics: pre-ATP currencies (thioester / acyl phosphate / polyphosphate / reduced metal) and the Sojo leaky-membrane ion gradient

**Proposed models / methods**

- energy_coupling face with five structurally different store laws (none / ionic capacitor / chemical ideal-mixing) behind one per-individual power budget
- thermodynamic store gates (drive must clear the store potential to charge; the store potential must clear the load to deliver work)
- free-energy ledger (free_energy_in >= work + dissipated with the store) as a contract invariant
- sweep over class x coupling activity x membrane permeability x gradient state; pays = net power above the passive baseline

**Key assumptions**

- One individual (radius 0.5 um) between the external ion gradient (tapped through its H+ leak, F P_H A dc pmf, captured at 0.3) and a chemical drive delivered by substrate turnover (1e-21 mol/s at full activity) that needs no leak.
- Store potentials: chemical carriers mu(q) = bond + RT ln(q/(1-q)) (ideal mixing, relative to the resting charge); ion gradient mu(q) = n F pmf_max q (capacitor, n = 3 ions per transfer, pmf_max 0.2 V).
- Charging needs the drive to clear mu(q) (capped exactly); work needs mu(q) to clear the 30 kJ/mol load; leaks relax toward the resting charge at hydrolysis / oxidation rates plus carrier permeation (ratio x P_H x A/V).
- Maintenance = 1e-19 W base + 5e-18 W x activity; the passive baseline pays only the base.
- pays = settled net power above the passive baseline at the same membrane and gradient by > 1e-21 W.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Protocell energetics — thioester class at half activity, nominal membrane` | `viva_genesis.composites.protocell_energetics.protocell_energetics` | 0 | model_energy_coupling=thioester_acyl_phosphate, coupling_activity=0.5 |

Declared parameter sets (`study.yaml` variants):

- **passive** — `model_energy_coupling=passive_gradient`
- **ion-gradient** — `model_energy_coupling=h_na_gradient_leaky_membrane`, `coupling_activity=1.0`
- **polyphosphate** — `model_energy_coupling=polyphosphate`, `coupling_activity=1.0`
- **fe-redox** — `model_energy_coupling=fe_redox`, `coupling_activity=1.0`
- **tight-membrane** — `permeability=1e-08`
- **collapsed-gradient** — `electric_potential=0.02`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.protocell_energetics.protocell_energetics`**

A protocell's energy budget with the energy-coupling mechanism as a model CLASS on the energy_coupling face (model_energy_coupling: passive_gradient | h_na_gradient_leaky_membrane | thioester_acyl_phosphate | polyphosphate | fe_redox). The individual (radius, membrane permeability) sits between the external ion gradient (tapped only through its leak) and the chemical drive (H2/CO2 redox, or Fe(II) disproportionation for fe_redox; vg-data energy_currencies); coupling_activity moves it from passive to active. Publishes the per-individual power budget (external / internal / maintenance / leak / net power, the store, autonomy = internal / (internal + external)) audited by a FreeEnergyLedgerProcess. Coarse-grained deterministic kinetics (reproducible), not a wrapped engine.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_protocell_energetics_protocell_energetics = {
    'model_energy_coupling': 'thioester_acyl_phosphate',   # Model class behind the 'energy_coupling' face: fe_redox | h_na_gradient_leaky_membrane | passive_gradient | polyphosphate | thioester_acyl_phosphate
    'coupling_activity'    : 0.5,   # Passive -> active axis: fraction of the drive turnover coupled into the internal store (0 = external tap only) [vg-data: hypothetical]
    'permeability'         : 1e-06,   # Membrane H+ permeability (m/s): the leak that feeds the external tap and drains the store [vg-data: extrapolated]
    'electric_potential'   : 0.216,   # External ion gradient (V) at t = 0 [vg-data: calibrated to the Phase-1 posterior dpH]
    'decline_rate'         : 0.0,   # External-gradient decline (V/s); 0 = steady (og-05's collapse scenario when > 0)
    'gradient_floor'       : 0.02,   # Floor the declining gradient decays to (V)
    'drive_free_energy'    : 50.0,   # Chemical drive free energy per turnover (kJ/mol) [vg-data: inferred]
    'fe_drive_free_energy' : 40.0,   # Fe(II)-disproportionation drive (kJ/mol), used by the fe_redox class [vg-data: hypothetical]
    'protocell_radius'     : 5e-07,   # Protocell radius (m) [vg-data: inferred]
    'drive_turnover'       : 1e-21,   # Drive turnover per cell at full activity (mol/s) [vg-data: hypothetical]
    'maintenance_active'   : 5e-18,   # Machinery upkeep at full activity (W) [vg-data: hypothetical]
    'tick_seconds'         : 10.0,   # Physical seconds per composite tick (store timescales are ~1e3 s); processes run at interval 1
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `electric_potential` | float | volt | >= 0.0 |
| `drive_free_energy` | float | kilojoule/mole | >= 0.0 |
| `permeability` | float | meter/second | >= 0.0 |
| `membrane_area` | float | meter**2 | >= 0.0 |
| `compartment_volume` | float | liter | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `free_energy_in` | float | watt | >= 0.0 |
| `external_power` | float | watt | >= 0.0 |
| `internal_power` | float | watt | >= 0.0 |
| `maintenance_power` | float | watt | >= 0.0 |
| `work` | float | watt | >= 0.0 |
| `dissipated` | float | watt | >= 0.0 |
| `energy_stored` | float | joule | >= 0.0 |
| `autonomy` | float | dimensionless | [0.0, 1.0] |

**Conditions**

- `free_energy_ledger` (invariant): `in_cum + E0 - work_cum - diss_cum == E(t) to rounding; E(t) >= 0 (free_energy_in >= work + dissipated)`
- `dissipation_nonnegative` (invariant): `dissipated >= 0 every tick`
- `capture_bound` (invariant): `work_cum / in_cum <= 1`
- `load_gate` (invariant): `internal work flows only once mu(q) >= load_free_energy`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-energy-alt-internalization-threshold ===
STUDY = 'og-energy-alt-internalization-threshold'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Protocell_energetics_thioester_class_at_half_activity_nominal_membrane = 100
INTERVAL_Protocell_energetics_thioester_class_at_half_activity_nominal_membrane = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_protocell_energetics_protocell_energetics if 'ov_viva_genesis_composites_protocell_energetics_protocell_energetics' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_protocell_energetics_protocell_energetics = materialize_composite('viva_genesis.composites.protocell_energetics.protocell_energetics', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_protocell_energetics_protocell_energetics['state']}, core=core)
            _comp.run(STEPS_Protocell_energetics_thioester_class_at_half_activity_nominal_membrane)
        print('ran', 'Protocell energetics — thioester class at half activity, nominal membrane', f'(viva_genesis.composites.protocell_energetics.protocell_energetics) for', STEPS_Protocell_energetics_thioester_class_at_half_activity_nominal_membrane, 'steps')
    except Exception as _e:
        print(f"Protocell energetics — thioester class at half activity, nominal membrane not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**internalization-threshold-map**


In [ ]:
# internalization-threshold-map
show_viz(_render_one('local:InternalizationThresholdMap', {'title': 'Where internalization pays, per energy-coupling class (Decision view)', 'threshold_map_json': 'workspace/studies/og-energy-alt-internalization-threshold/artifacts/internalization_threshold_map.json', 'report_json': 'workspace/studies/og-energy-alt-internalization-threshold/artifacts/internalization_threshold_report.json'}, RUNS_DB, STUDY_YAML))

**protocell-energy-budget**


In [ ]:
# protocell-energy-budget
show_viz(_render_one('local:ProtocellEnergyBudget', {'title': "One individual's power budget per coupling class (Mechanism view)", 'report_json': 'workspace/studies/og-energy-alt-internalization-threshold/artifacts/internalization_threshold_report.json', 'activity': '1'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| free-energy-ledger-closes _(primary)_ | kind=derived path=free_energy_ledger_ok_final detail=free_energy_ledger_ok on the final tick | op >= value 1.0 detail every ledger witness within tolerance |
| free-energy-residual-exact _(diagnostic)_ | kind=derived path=free_energy_residual_max detail=worst-case relative ledger residual | op <= value 1e-09 detail free-energy accounting exact by construction |
| second-law-margin-nonnegative _(diagnostic)_ | kind=derived path=second_law_margin_min detail=minimum margin over the run (joule) | op >= value -1e-24 detail non-negative to rounding (per-cell energies are ~1e-16 J) |
| individual-is-partly-autonomous _(primary)_ | kind=derived path=final_autonomy detail=autonomy on the final tick | op > value 0.05 detail the internal store contributes > 5 % of the useful power |
| internalization-pays _(primary)_ | kind=derived path=n_classes_paying_at_nominal detail=classes with a finite threshold at the nominal membrane + gradient, from artifacts/internalization_threshold_report.json | op >= value 1.0 detail >= 1 class pays at the nominal point |
| thioester-threshold-activity _(primary)_ | kind=derived path=internalization_threshold_activity detail=smallest activity that pays, class thioester_acyl_phosphate | op <= value 1.0 detail a finite threshold inside 0..1 |
| autonomy-at-threshold _(supporting)_ | kind=derived path=autonomy_at_threshold detail=internal / (internal + external) at a*, class thioester_acyl_phosphate | op > value 0.0 detail the store contributes at the threshold |
| ion-gradient-pays-fraction _(diagnostic)_ | kind=derived path=internalization_pays_fraction detail=class h_na_gradient_leaky_membrane | op >= value 0.0 detail informative readout (any value passes) |
| polyphosphate-pays-fraction _(diagnostic)_ | kind=derived path=internalization_pays_fraction detail=class polyphosphate | op >= value 0.0 detail informative readout (any value passes) |
| fe-redox-pays-fraction _(diagnostic)_ | kind=derived path=internalization_pays_fraction detail=class fe_redox | op >= value 0.0 detail informative readout (any value passes) |
| classes-agree-on-pays _(diagnostic)_ | kind=derived path=internalization_structure_agreement detail=structure agreement on the pays masks | op >= value 0.0 detail informative readout (any value passes) |
| threshold-map-ledger-all-ok _(supporting)_ | kind=derived path=threshold_map_ledger_all_ok detail=all classes' ledger all_ok | op >= value 1.0 detail no point with an open ledger |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Early membranes were far more permeable to H+ than Na+, so a Na+ gradient is the better internal store and tapping the natural gradient needs leak while owning one needs a tight membrane | the ion-gradient class's store potential and pays region against permeability | Sojo, Pomiankowski & Lane 2014 | Reproduced structurally — the ion store pays only below a permeability ceiling; the chemical carriers do not share the ceiling. |
| A transferable currency does work only when its phosphorylation-like potential (bond + RT ln q/(1-q)) clears the endergonic step; low-bond carriers must be held near full charge | store potential vs load per class; polyphosphate's charge fraction at the gate | Buckel & Thauer 2013 | Implemented as the draw gate; polyphosphate's 20 kJ/mol bond needs q > 0.98 for a 30 kJ/mol load. |


## Study: Competing Heredity: Which Inheritance Mechanism Transmits the Most Bits Per Joule, and When Is It Attached? (`og-06-alt-information-per-cost`)

**Question.** When six inheritance mechanisms are each simulated exactly at low copy number under matched precursor and energy budgets, which transmits the most heritable information per joule, and where does templated (RNA-like) heredity take over?

**Objective.** Build the heredity face (six classes carved out of og-06's LineageReplicatorProcess, each an exact Gillespie lineage drawing integer precursors and quanta from matched budgets), the HeredityAttachmentStep (precursor budget, energy budget, Eigen floor Q >= 1/sigma), the InformationLedgerProcess (heritable bits per joule and per precursor from a parent-offspring mutual-information estimator with a convergence check; exact energy-store audit) and the vg-data package inheritance_budgets; run the frontier (six classes x attached / control x 6 seeds), the attachment map (six classes x six matched budget scales) and the fidelity transition (four template-type classes x six fidelities) and grade the pre-registered gates.

**Hypothesis.** Every attached class transmits parent-offspring information its unattached control (daughters re-formed de novo from the same budget) does not; the template classes carry the most bits per offspring but pay the most per bit and are attached only at the highest budgets and fidelities; the attractor classes (compositional, network) are cheap, attach at low budgets and carry few bits; a cheap, high-fidelity mineral pattern would dominate both axes if its copy error were as low as hypothesised; the transition to templated heredity is the point at which the Eigen floor admits the template.

**Claim.** Priced under matched budgets, heredity splits by currency: the RNA-like template transmits the most certified information per offspring (1.46 bit) but needs about 30x the precursor and energy budget of the attractor classes and ribozyme-level fidelity (q >= 0.967) to attach at all, while the cheap classes (network, compositional) carry roughly a third of its bits at 10-1000x the bits per joule - a frontier spanning 3.8 decades. og-06's "heredity need not presume RNA" survives as a statement about efficiency, not capacity: the template is the richest organic carrier, the attractors the cheapest. The one class that would dominate both axes, mineral-surface patterns, does so only on a defect-copy error that has never been measured, so the frontier's ordering is reported under declared costs rather than as a ranking.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- J: heredity as six competing model classes under matched budgets, attached conditionally
- Eigen error threshold as the attachment fidelity floor

**Proposed models / methods**

- Exact Gillespie (direct method) lineages at low copy number, one per heredity class
- Conditional heredity attachment (FMP 2.5) on precursor / energy budgets and the Eigen floor
- Binned parent-offspring mutual information with Miller-Madow correction, permutation null and a prefix-ladder + bootstrap convergence check
- Information ledger (bits per joule / per precursor) with an exact energy-store audit

**Key assumptions**

- 32 individuals of 1 fL at carrying capacity, 16 heritable molecules per founder (cap 48), bulk mass 20 doubling to divide; bulk and heritable growth both scale with (1 + sigma phi), sigma = 3; dilution uniform at capacity.
- One common genotype-to-function landscape: a sigmoid in the match fraction (half 0.8, width 0.04), so the master and its near neighbours are functional and the tail is not.
- Matched budgets: every class draws integer molecules from the same chemostat-fed pools (1e-4 M set-points, relaxation 0.5 per tick) and integer quanta (30 kJ/mol) from the same store (8e-16 J per tick, capped).
- Information = parent-offspring MI of the phenotype at a 0.05 resolution, Miller-Madow corrected, net of a 20-permutation null; converged when the prefix ladder agrees within 0.05 bit / 15 % and the bootstrap SE <= 0.05 bit.
- Attachment: precursor ratio, energy ratio and Q / (1/sigma) all >= 1, latched; before attachment daughters re-form heritable material de novo at the class's price.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Competing heredity — rna_like under the attachment step at the nominal matched budget` | `viva_genesis.composites.competing_heredity.competing_heredity` | 0 | model_heredity=rna_like, attachment=step |

Declared parameter sets (`study.yaml` variants):

- **compositional** — `model_heredity=compositional`
- **network** — `model_heredity=network`
- **peptide** — `model_heredity=peptide`
- **hybrid** — `model_heredity=hybrid`
- **mineral-state** — `model_heredity=mineral_state`
- **non-heritable-control** — `attachment=off`
- **forced-attached** — `attachment=on`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.competing_heredity.competing_heredity`**

og-06's heredity lineage rebuilt as the Phase-7 competing-heredity composite: the inheritance mechanism is a model CLASS on the heredity face (model_heredity: compositional | network | peptide | rna_like | hybrid | mineral_state), every class an exact Gillespie lineage of protocell-scale individuals (tens of heritable molecules each) drawing integer precursors and energy quanta from the SAME chemostat-fed pools and energy store. HeredityAttachmentStep attaches heredity only when the precursor budget, the energy budget and the class's expected copy fidelity (Eigen Q >= 1/sigma) allow it — before that, daughters re-form their heritable material de novo (the non-heritable control). InformationLedgerProcess prices the realised parent-offspring mutual information per joule and per precursor and audits the energy store exactly.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_competing_heredity_competing_heredity = {
    'model_heredity'      : 'rna_like',   # Model class behind the 'heredity' face: compositional | hybrid | mineral_state | network | peptide | rna_like
    'attachment'          : 'step',   # step = HeredityAttachmentStep decides; on = heredity forced attached; off = non-heritable control
    'nucleotide_feed'     : 0.0001,   # Activated-nucleotide feed set-point (M) [vg-data: hypothetical]
    'amino_feed'          : 0.0001,   # Activated amino-acid feed set-point (M) [vg-data: extrapolated]
    'amphiphile_feed'     : 0.0001,   # Amphiphile feed set-point (M) [vg-data: inferred]
    'mineral_feed'        : 0.0001,   # Mineral growth-unit feed set-point (M) [vg-data: hypothetical]
    'power'               : 8e-16,   # Energy supply to the shared store (J per tick) [vg-data: hypothetical]
    'per_site_fidelity'   : 0.967,   # Template per-site copy fidelity q (rna_like / hybrid) [vg-data: measured, Johnston 2001]
    'peptide_fidelity'    : 0.7,   # Peptide per-site templating fidelity q_p [vg-data: hypothetical]
    'defect_copy_error'   : 0.005,   # Mineral per-bit defect copy error per layer p_m [vg-data: hypothetical]
    'attach_min_copies'   : 10.0,   # Attachment threshold: copies per individual the budget must cover [vg-data: hypothetical, swept]
    'attach_energy_margin': 1.0,   # Attachment threshold: energy-budget margin [vg-data: hypothetical, swept]
    'n_individuals'       : 32,   # Lineage capacity (individuals)
    'seed'                : 1,   # RNG seed (reproducible SSA)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `attached` | float | dimensionless | [0.0, 1.0] |
| `stored_work` | float | joule | >= 0.0 |
| `nucleotide_precursor` | float | molar | >= 0.0 |
| `amino_precursor` | float | molar | >= 0.0 |
| `amphiphile` | float | molar | >= 0.0 |
| `mineral_precursor` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `information_content` | float | bit | >= 0.0 |
| `heritability` | float | dimensionless |  |
| `fidelity` | float | dimensionless | [0.0, 1.0] |
| `replication_energy` | float | joule | >= 0.0 |
| `mi_converged` | float | dimensionless | [0.0, 1.0] |

**Conditions**

- `energy_store_audit` (invariant): `ledger.energy_residual <= 1e-9`
- `integer_budgets` (invariant): `pool molecules + consumed == initial + fed, exactly (integer counts)`
- `certified_bits_need_convergence` (invariant): `ledger.heritable_bits == 0 unless info.mi_converged == 1`
- `control_is_unattached` (invariant): `attachment = off reforms heritable material de novo from the same budget at the same price`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-06-alt-information-per-cost ===
STUDY = 'og-06-alt-information-per-cost'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Competing_heredity_rna_like_under_the_attachment_step_at_the_nominal_matched_budget = 100
INTERVAL_Competing_heredity_rna_like_under_the_attachment_step_at_the_nominal_matched_budget = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_competing_heredity_competing_heredity if 'ov_viva_genesis_composites_competing_heredity_competing_heredity' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_competing_heredity_competing_heredity = materialize_composite('viva_genesis.composites.competing_heredity.competing_heredity', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_competing_heredity_competing_heredity['state']}, core=core)
            _comp.run(STEPS_Competing_heredity_rna_like_under_the_attachment_step_at_the_nominal_matched_budget)
        print('ran', 'Competing heredity — rna_like under the attachment step at the nominal matched budget', f'(viva_genesis.composites.competing_heredity.competing_heredity) for', STEPS_Competing_heredity_rna_like_under_the_attachment_step_at_the_nominal_matched_budget, 'steps')
    except Exception as _e:
        print(f"Competing heredity — rna_like under the attachment step at the nominal matched budget not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**heredity-information-frontier**


In [ ]:
# heredity-information-frontier
show_viz(_render_one('local:HeredityInformationFrontier', {'title': 'Heritable bits vs energy per offspring, per heredity class, with controls and the Pareto front (Decision view)', 'report_json': 'workspace/studies/og-06-alt-information-per-cost/artifacts/information_frontier.json'}, RUNS_DB, STUDY_YAML))

**heredity-attachment-map**


In [ ]:
# heredity-attachment-map
show_viz(_render_one('local:HeredityAttachmentMap', {'title': 'Where heredity is attached, and the transition to templated heredity (Mechanism view)', 'report_json': 'workspace/studies/og-06-alt-information-per-cost/artifacts/information_frontier.json'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| information-ledger-closes _(primary)_ | kind=derived path=energy_store_residual_max detail=worst-case energy-store residual over the run | op <= value 1e-09 detail energy audit at float round-off |
| mi-estimator-converged _(primary)_ | kind=derived path=mi_converged_final detail=info.mi_converged on the final tick | op >= value 1.0 detail converged |
| information-ledger-certifies _(diagnostic)_ | kind=derived path=information_ledger_ok_final detail=ledger.information_ledger_ok on the final tick | op >= value 1.0 detail certified |
| heredity-attached-under-nominal-budget _(supporting)_ | kind=derived path=heredity_attached_final detail=heredity.attached on the final tick | op >= value 1.0 detail attached |
| canonical-heritable-bits _(supporting)_ | kind=derived path=heritable_bits_final detail=info.information_content on the final tick | op >= value 0.5 detail >= 0.5 bit |
| function-sustained-by-heredity _(diagnostic)_ | kind=derived path=heredity_settled_function detail=last-quartile mean of lineage.mean_function | op > value 0.05 detail function kept above the control's 0.00 |
| ancestry-predicts-phenotype _(primary)_ | kind=derived path=n_classes_ancestry_predicts detail=number of classes passing the per-class rule | op >= value 4.0 detail >= 4 of 6 classes |
| information-per-cost-frontier-resolved _(primary)_ | kind=derived path=frontier_span_decades detail=log10 range of bits per joule across classes | op >= value 1.0 detail >= 1 decade |
| heredity-need-not-presume-rna _(supporting)_ | kind=derived path=nonrna_bits_per_joule_fraction detail=best non-RNA bits per joule / rna_like bits per joule | op >= value 1.0 detail a non-RNA class matches or beats RNA per joule |
| nonrna-bits-per-offspring-fraction _(diagnostic)_ | kind=derived path=nonrna_bits_fraction detail=best non-RNA bits / rna_like bits | op >= value 0.0 detail informative readout (any value passes) |
| attachment-is-conditional _(supporting)_ | kind=derived path=attachment_budget_span detail=mean attached-fraction span low-to-high budget | op >= value 0.5 detail >= 0.5 span |
| templated-takeover-fidelity _(diagnostic)_ | kind=derived path=templated_takeover_fidelity detail=from fidelity_transition.per_class.rna_like | op <= value 0.99 detail a takeover exists inside the swept range |
| frontier-pareto-size _(diagnostic)_ | kind=derived path=frontier_pareto_size detail=certified Pareto front size | op >= value 1.0 detail informative readout |
| frontier-energy-ledger-closes _(diagnostic)_ | kind=derived path=frontier_energy_within_tolerance detail=1 when all runs within tolerance | op >= value 1.0 detail all runs within tolerance |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| A template of length L with per-site fidelity q keeps its information only above the error threshold q^L > 1/sigma | fidelity_transition.per_class.rna_like (attaches from, settled function) | Eigen 1971 | Reproduced at the agent level with exact SSA — rna_like attaches and keeps function from q = 0.967 at L = 30, sigma = 3 (floor 0.964). |
| Compositional (GARD) assemblies transmit information by growth and fission without a template, but with limited capacity and heritability | classes.compositional (heritable bits, heritability, capacity) | Segre 2000; Vasas 2010 | 0.47 bit per offspring at heritability 0.70, capacity 18 bits, 0 quanta — cheap, attaches at 3 % of the budget. |
| An RNA polymerase ribozyme copies at ~97 % per-site fidelity | the rna_like class's nominal per_site_fidelity | Johnston 2001 | The package's one measured entry (0.967); it sits just above the Eigen floor at L = 30. |


## Study: Biochemical Ecology: A Community Function That No Lineage Can Perform, Carried by Real Metabolite Flux (`og-07-alt-real-metabolite-exchange`)

**Question.** If every protocell carries the same real rule-generated chemistry masked to its own catalyst repertoire, can a community of incomplete lineages realise a metabolic function no lineage performs alone, sustained by a real intermediate crossing membranes rather than a symbolic capability transfer?

**Objective.** Build the proto_ecology face (ProtocellNetworkProcess and the matched-null class — the reaction universe inside each agent, masked to a lineage repertoire), the SpatialCouplingStep onto the enrichment field with the exchange rules of a BigraphicalReactiveSystem (uptake, secretion, pore diffusion, feed / washout; every firing counted), the EcologyReducerStep that recomputes og-07's gate readouts from the agents, the EcologyElementLedgerProcess (C / N / P over agents + field) and the vg-data package cnp_resources; run the community, its two monocultures, the private (no-sharing) counterfactual, the complete-cell (generalist) arm, the segregated placement and the rewired-null arms at the canonical point, and the four arms over (diacid permeability x pore residence); grade the pre-registered gates.

**Hypothesis.** The reductive C4 branch (oxaloacetate -> malate -> succinate) is split across the two incomplete lineages: the fixer carboxylates pyruvate to oxaloacetate but cannot reduce it, the reducer can reduce it but never makes it; in a shared pore the dicarboxylate crosses from one to the other and malate appears in the reducer lineage although neither monoculture ever holds it; cutting the sharing (each agent exchanging only with its own copy of the background) removes it; the function depends on the membrane permeability of the exchanged diacid and on the pore residence time, and it emerges while every cell is still incomplete.

**Claim.** A community of one carboxylating and one hydrogenating lineage holds malate - the reductive C4 branch from oxaloacetate - at 6.4 nM while neither monoculture ever makes it and a complete cell makes it at about 20x the level. The function vanishes when nothing is shared, and its oxaloacetate precursor physically crosses between the lineages, so no capability changes hands; it appears while every cell is still incomplete. It exists only above a dicarboxylate membrane permeability of ~3e-9 m/s, one to two decades above the measured passage of charged solutes through fatty-acid vesicles, so the leakiness of the boundary rather than the chemistry decides whether ecology could run on exchanged intermediates - and at ~4e-3 molecules per cell the result is a mean-field expectation that still awaits a stochastic treatment.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- L: ecology as real metabolite exchange between incomplete protocells
- division of labour across catalyst repertoires (carboxylation vs hydrogenation)

**Proposed models / methods**

- The reaction universe (libRoadRunner) inside every agent, masked to a lineage rule repertoire
- A bigraphical reactive system of matter-exchange rules with firing counts
- og-07's gate readouts recomputed from the agents map
- A C / N / P element ledger across agents and field

**Key assumptions**

- Eight 1-fL protocells (4 fixers, 4 reducers) at fixed positions on a 4-cell pore of 10x their volume; feed CO2 1.4 mM, H2 10.8 mM, pyruvate 0.1 mM (the vent posterior point), ammonium 50 uM, phosphate 10 uM; the localization-stage surrogate enrichment profile along the pore.
- A lineage is a subset of the twelve reaction rules; the thermal decarboxylations belong to all; fixer = CO2 reduction + carboxylations, reducer = hydrogenations; the universe (C <= 4, 413 reactions, eQuilibrator dG', detailed balance) is otherwise identical in every agent.
- Exchange is linear permeation by size / charge class (gas 1e-5, small neutral 1e-7, large neutral 3e-8, monoacid 1e-7, diacid 1e-8, ion 1e-10 m/s at the canonical point), applied as the exact two-compartment relaxation per tick; pore diffusion 0.1/s between cells; residence 100 s.
- Functions are product classes (chain length x carboxylation x oxidation state); held above 1 nM, produced above 1e-12 M/s gross internal production.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Metabolite-exchange community — fixer + reducer lineages at the canonical point` | `viva_genesis.composites.metabolite_exchange_community.metabolite_exchange_community` | 0 | lineages=fixer:4,reducer:4, p_diacid=1e-08, p_monoacid=1e-07, k_wash=0.01 |

Declared parameter sets (`study.yaml` variants):

- **mono-fixer** — `lineages=fixer:8`
- **mono-reducer** — `lineages=reducer:8`
- **private-no-sharing** — `coupling=private`
- **generalist-complete-cells** — `lineages=generalist:8`
- **segregated-placement** — `placement=segregated`
- **rewired-null** — `model_proto_ecology=null_network`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.metabolite_exchange_community.metabolite_exchange_community`**

og-07's metacommunity rebuilt on real chemistry (Phase 8): N protocells in the agents map, each running the Phase-3 rule-generated reaction universe (libRoadRunner, eQuilibrator dG') masked to its lineage's catalyst repertoire (fixer: CO2 reduction + carboxylations; reducer: hydrogenations; condenser: aldol / formose; generalist: all), pointed at fixed cells of a pore whose background the Phase-4 localization class shapes species by species. A SpatialCouplingStep fires the exchange rules of a bigraphical reactive system every tick (uptake / secretion through size- and charge-dependent permeabilities, pore diffusion, feed / washout) and counts the firings; an EcologyReducerStep recomputes og-07's gate readouts from the agents (present vs produced functions); an EcologyElementLedgerProcess audits C / N / P over agents + field. coupling=private is the isolated counterfactual; model_proto_ecology=null_network the rewired control.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_metabolite_exchange_community_metabolite_exchange_community = {
    'model_proto_ecology'   : 'reaction_universe',   # Model class behind the 'proto_ecology' face: null_network | reaction_universe
    'model_localization'    : 'multisolute_surrogate',   # Model class behind the 'localization' face: fenics | fenics_multisolute_3d | fipy | multisolute_surrogate | surrogate
    'lineages'              : 'fixer:4,reducer:4',   # Agents per lineage repertoire, e.g. fixer:4,reducer:4 | fixer:8 | generalist:8
    'placement'             : 'mixed',   # mixed | segregated | random positions along the pore (fixed; no motion)
    'coupling'              : 'shared',   # shared = agents exchange with a common pore field; private = each agent sees only the background (isolated counterfactual)
    'n_cells'               : 4,   # Pore field cells along the flow axis [vg-data: hypothetical]
    'pool_volume_multiplier': 10.0,   # Pore volume per agent volume [vg-data: hypothetical]
    'k_wash'                : 0.05,   # Pore feed / washout rate (1/s); 1/residence time [vg-data: inferred, swept]
    'k_diffusion'           : 0.1,   # Exchange rate between neighbouring cells (1/s) [vg-data: hypothetical]
    'p_gas'                 : 1e-05,   # Membrane permeability class p_gas (m/s) [vg-data: extrapolated]
    'p_small_neutral'       : 1e-07,   # Membrane permeability class p_small_neutral (m/s) [vg-data: extrapolated]
    'p_large_neutral'       : 3e-08,   # Membrane permeability class p_large_neutral (m/s) [vg-data: extrapolated]
    'p_monoacid'            : 1e-09,   # Membrane permeability class p_monoacid (m/s) [vg-data: extrapolated]
    'p_diacid'              : 1e-10,   # Membrane permeability class p_diacid (m/s) [vg-data: hypothetical, swept]
    'p_ion'                 : 1e-10,   # Membrane permeability class p_ion (m/s) [vg-data: extrapolated]
    'feed_CO2'              : 0.0014,   # Inlet CO2 (M) [vg-data: calibrated, Phase-1 posterior]
    'feed_H2'               : 0.0108,   # Inlet H2 (M) [vg-data: calibrated, Phase-1 posterior]
    'feed_pyruvate'         : 0.0001,   # Inlet pyruvate (M) [vg-data: inferred]
    'feed_formate'          : 0.0,   # Inlet formate (M) [vg-data: hypothetical]
    'feed_ammonium'         : 5e-05,   # Inlet ammonium (M) [vg-data: extrapolated]
    'feed_phosphate'        : 1e-05,   # Inlet phosphate (M) [vg-data: extrapolated]
    'peclet'                : 8.0,   # Pore advective Peclet for the localization class
    'soret_scale'           : 1.0,   # Scale on the per-species Soret coefficients
    'null_seed'             : 0,   # Rewiring seed for the null_network class
    'function_floor'        : 1e-09,   # Concentration floor for holding a function (M) [vg-data: inferred]
    'seed'                  : 1,   # Placement seed (random placement only)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `internal` | map[vg_concentration] | molar |  |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `internal` | map[vg_concentration] | molar |  |
| `functions_produced` | tree |  |  |
| `rule_flux` | tree |  |  |
| `net_carbon_flux` | float | mole/second |  |
| `dissipation` | float | watt | >= 0.0 |
| `element_residual` | float | dimensionless | >= 0.0 |

**Conditions**

- `cross_system_element_balance` (invariant): `total_X(t) = total_X(0) + in_X - out_X - degraded_X for X in C, N, P (C at integrator tolerance 1e-4; N, P at 1e-9)`
- `exchange_conserves` (invariant): `every exchange-rule firing moves the same moles out of one place and into the other`
- `element_balance_exact` (invariant): `max |E S| == 0 over every agent's reactions`
- `energy_directionality` (invariant): `dissipation >= 0 in every agent every tick`
- `no_capability_transfer` (invariant): `no rule writes a repertoire; functions_produced follows from each agent's own mask`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-07-alt-real-metabolite-exchange ===
STUDY = 'og-07-alt-real-metabolite-exchange'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Metabolite_exchange_community_fixer_reducer_lineages_at_the_canonical_point = 100
INTERVAL_Metabolite_exchange_community_fixer_reducer_lineages_at_the_canonical_point = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_metabolite_exchange_community_metabolite_exchange_community if 'ov_viva_genesis_composites_metabolite_exchange_community_metabolite_exchange_community' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_metabolite_exchange_community_metabolite_exchange_community = materialize_composite('viva_genesis.composites.metabolite_exchange_community.metabolite_exchange_community', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_metabolite_exchange_community_metabolite_exchange_community['state']}, core=core)
            _comp.run(STEPS_Metabolite_exchange_community_fixer_reducer_lineages_at_the_canonical_point)
        print('ran', 'Metabolite-exchange community — fixer + reducer lineages at the canonical point', f'(viva_genesis.composites.metabolite_exchange_community.metabolite_exchange_community) for', STEPS_Metabolite_exchange_community_fixer_reducer_lineages_at_the_canonical_point, 'steps')
    except Exception as _e:
        print(f"Metabolite-exchange community — fixer + reducer lineages at the canonical point not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**community-function-map**


In [ ]:
# community-function-map
show_viz(_render_one('local:CommunityFunctionMap', {'title': 'Where a function no lineage can perform emerges (Decision view)', 'report_json': 'workspace/studies/og-07-alt-real-metabolite-exchange/artifacts/community_function_map.json'}, RUNS_DB, STUDY_YAML))

**metabolite-exchange-map**


In [ ]:
# metabolite-exchange-map
show_viz(_render_one('local:MetaboliteExchangeMap', {'title': "Agents on the pore, the intermediate that crosses, and og-07's readouts from real agents (Mechanism / World view)", 'report_json': 'workspace/studies/og-07-alt-real-metabolite-exchange/artifacts/community_function_map.json'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cross-system-element-ledger-closes _(primary)_ | kind=derived path=element_ledger_ok_final detail=ledger.element_ledger_ok on the final tick | op >= value 1.0 detail every residual within tolerance |
| carbon-residual-at-integrator-tolerance _(diagnostic)_ | kind=derived path=carbon_residual_max_final detail=ledger.C_residual_max on the final tick | op <= value 0.0001 detail integrator tolerance |
| exchange-rules-fire _(supporting)_ | kind=derived path=exchange_firings_final detail=cumulative firings of every exchange rule | op > value 1000.0 detail rules fired |
| metabolites-cross-between-lineages _(supporting)_ | kind=derived path=cross_lineage_flux_total_final detail=cumulative cross-lineage moles | op > value 0.0 detail non-zero cross-lineage flux |
| no-cell-complete _(supporting)_ | kind=derived path=max_cell_completeness_final detail=max over agents of held / union | op < value 1.0 detail no complete cell |
| cross-feeding-robustness _(diagnostic)_ | kind=derived path=metacommunity_robustness detail=settled community.metacommunity_robustness | op > value 0.3 detail a third of the vocabulary redundantly held |
| luca-capability-overlap _(diagnostic)_ | kind=derived path=luca_capability_overlap detail=settled community.luca_capability_overlap | op >= value 0.0 detail informative readout |
| community-function-precedes-cell-completeness _(primary)_ | kind=derived path=n_community_only_functions detail=community-only product classes at the canonical point | op >= value 1.0 detail >= 1 community-only class |
| community-function-vanishes-without-exchange _(primary)_ | kind=derived path=n_vanish_without_exchange detail=community-only classes absent in the private arm | op >= value 1.0 detail vanishes without sharing |
| community-function-flux-sustained _(primary)_ | kind=derived path=community_flux_sustained detail=1 when sustained by a crossing precursor | op >= value 1.0 detail flux-sustained |
| community-max-cell-completeness _(supporting)_ | kind=derived path=community_max_cell_completeness detail=max over agents of held / union | op < value 1.0 detail no complete cell |
| community-vs-complete-cell _(diagnostic)_ | kind=derived path=community_over_generalist detail=community / generalist settled level | op >= value 0.0 detail informative readout |
| emergence-boundary-resolved _(diagnostic)_ | kind=derived path=emergence_p_diacid_nominal detail=from decision_map.emergence_p_diacid_by_k_wash | op <= value 1e-07 detail inside the swept range |
| null-community-only-count _(diagnostic)_ | kind=derived path=null_community_only_max detail=max over null seeds | op >= value 0.0 detail informative readout |
| segregated-placement-keeps-function _(supporting)_ | kind=derived path=segregated_n_community_only detail=community-only classes in the segregated arm | op >= value 1.0 detail function kept |
| connected-beats-isolated _(supporting)_ | kind=derived path=connected_minus_isolated_robustness detail=robustness(shared) - robustness(private) | op > value 0.0 detail shared more robust than private |
| map-ledgers-close _(diagnostic)_ | kind=derived path=map_ledgers_all_ok detail=1 when all runs within tolerance | op >= value 1.0 detail all runs within tolerance |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| LUCA: anaerobic, H2-dependent, CO2-fixing, FeS-rich — embedded in an ecosystem | community.luca_capability_overlap | Weiss 2016 / Moody 2024 | 0.8 for every arm with a carboxylating lineage (ceiling 0.8 — ion pumping is not in a C/H/O network); reported as a boundary. |
| Fatty-acid vesicles pass small uncharged solutes readily and charged solutes slowly | cnp_resources permeability classes (p_small_neutral, p_monoacid, p_diacid) | Jordan 2019 (vesicle system); measured charged-solute passage proposed (Mansy 2008, pending) | The emergence boundary (3e-9 m/s) sits 1-2 decades above the measured charged-solute range; the diacid value is hypothetical and swept. |
| Long tight pores retain solutes (residence set by the thermophoretic trap) | SpatialCouplingStep.k_wash (the residence axis) | Baaske 2007 / Mast 2010 | The function needs 20-100 s residence at 3e-9 m/s; at 500 s the feed depletes and it needs 1e-7 m/s. |


## Study: Integrated Autonomy: The Minimum Sufficient Innovation Set Across Model Structures (`og-08-alt-robust-innovation-set`)

**Question.** In one integrated world where each of {pump, importer, heredity} is a real process carrying its own cost, is that set jointly sufficient for a lineage to persist as the vent's support is withdrawn - the pre-registered test being that the full set persists under at least half the sampled model structures?

**Objective.** Build the integrated world (AblationStep, GeologicalDriverStep, LineageStateProcess, ActiveImportProcess with the ActiveUptakeRule, MaintenanceDemandProcess, LineageSupplyProcess, AutonomyBudgetStep, PopulationBudgetProcess, StructuralEventLogStep; the integrated_autonomy_world composite over the energy_coupling / heredity / boundary / localization faces with a permeability anchor and a maintenance law; the vg-data package integrated_autonomy carrying the vent posterior), run every subset of {pump, importer, heredity} under a sampled structural ensemble at the nominal and a rich ocean floor with the og-adv generic control and the full set over the floor axis, grade the pre-registered gates and report the intersection of the required innovations as a lattice with the continuous autonomy value decomposed by function.

**Hypothesis.** og-08's set is a hypothesis generator, not a result: in the integrated world the innovations are real processes whose benefit and cost depend on the structure around them, so the required set will not be the same under every structure. The pump (an internal store charged by the drive the cell's own substrate pool sustains) should be required wherever anything persists, because the external tap vanishes with the gradient; heredity should be required wherever the inheritance class holds more machinery attached than its de-novo control does; the importer should matter only where the membrane is tight enough for concentration to pay and the ocean background is poor enough to need it; and the ocean floor the withdrawal relaxes toward — a hypothetical — should decide whether any subset persists at all.

**Claim.** Varying pH gradient, H2, CO2, mineral catalysis, temperature and organics one at a time, the study determines which specific environmental support each chemical lineage can internalize or replace and which it cannot — locating where persistence fails and attributing that failure to a named missing support, rather than reporting a single lumped collapse. The earlier joint-sufficiency framing is withdrawn because lumped withdrawal cannot attribute the loss.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- M: autonomy as the integrated budget of real processes under withdrawal
- co-limitation of work and material; machinery held by inheritance

**Proposed models / methods**

- One persistent world assembled from the merged faces with a scheduled ablation
- A structural ensemble over storage model, inheritance class, permeability assumption, transport law and maintenance function
- The innovation lattice with the intersection of required sets across structures
- Seven ledgers and a structural-event log over the integrated world

**Key assumptions**

- One 0.52-pL protocell (the energy-coupling individual) carries the reaction universe (generalist repertoire) and the metacommunity exchange rules; its vent is one vent posterior draw (H2 2.3-10.8 mM, dpH 3.4, 322 K); the withdrawal is linear over 100 minutes after one hour in the vent, every driver relaxing to the ocean floor (the gradient to 0).
- Pump = coupling_activity 1 on the storage class, its drive sustained by the cell's live H2 / organic pool (half-saturation 10 uM) and coupled by the machinery the heredity class holds (mean function over 0.25); importer = the ActiveUptakeRule (1e-19 mol/s per substrate at full machinery, K_m 1 uM, set-point 30 % of the vent level, at most 30 % of the work supply per tick, at the thermodynamic cost over a 0.5 efficiency); heredity = attachment by the HeredityAttachmentStep (off: og-06's non-heritable control).
- Persistence (pre-registered): over the final 15 % of the run the settled work budget and material budget are non-negative and at least half the founders are alive.
- The heredity class runs one lineage-time unit per 60-s world tick; its precursor pools follow a tenth of the cell's organic carbon; 5 % of the lineage's work feeds its energy store.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Integrated autonomy world — full set at the canonical structure, nominal floor` | `viva_genesis.composites.integrated_autonomy.integrated_autonomy_world` | 0 | model_energy_coupling=thioester_acyl_phosphate, model_heredity=rna_like, model_boundary=single_chain, permeability_anchor=extrapolated, model_localization=multisolute_surrogate, maintenance_law=constant, pump=on, importer=on, heredity=on, feed_floor=0.01 |

Declared parameter sets (`study.yaml` variants):

- **no-pump** — `pump=off`
- **no-importer** — `importer=off`
- **no-heredity** — `heredity=off`
- **generic-environment** — `environment=generic`
- **rich-ocean** — `feed_floor=0.1`


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.integrated_autonomy.integrated_autonomy_world`**

og-08's off-gradient test inside ONE persistent world assembled from the merged phases: the lineage lives on what real processes deliver — the external pmf tapped through its membrane leak and an internal store charged by the drive its live substrate pool sustains (an energy_coupling class, the STORAGE MODEL), organic carbon reaching it through a localization class (the TRANSPORT LAW) and the Phase-8 exchange rules at a boundary class's leakiness (the PERMEABILITY ASSUMPTION), the Phase-3 universe inside it catalysed by the mineral surface or by the machinery its heredity class holds (the INHERITANCE CLASS), upkeep priced by a MAINTENANCE LAW, and a population of individuals that divide and go extinct by structural rewrites. An AblationStep withdraws the gradient, the mineral access and the feed; pump (coupling_activity), importer (the ActiveUptakeRule) and heredity (attachment) are the three innovations; environment=generic is the og-adv reject control. Seven ledgers run beside the world and a structural-event log records every event. The vent endmembers are one Phase-1 posterior draw (vg-data integrated_autonomy).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_integrated_autonomy_integrated_autonomy_world = {
    'model_energy_coupling' : 'thioester_acyl_phosphate',   # Model class behind the 'energy_coupling' face: fe_redox | h_na_gradient_leaky_membrane | passive_gradient | polyphosphate | thioester_acyl_phosphate
    'model_heredity'        : 'rna_like',   # Model class behind the 'heredity' face: compositional | hybrid | mineral_state | network | peptide | rna_like
    'model_boundary'        : 'single_chain',   # Model class behind the 'boundary' face: mixed | single_chain | thioester_aminothiol
    'model_localization'    : 'multisolute_surrogate',   # Model class behind the 'localization' face: fenics | fenics_multisolute_3d | fipy | multisolute_surrogate | surrogate
    'maintenance_law'       : 'constant',   # Maintenance function: constant (Phase 6) | leak_repair (scales with leakiness) | turnover (scales with machinery held)
    'pump'                  : 'on',   # Internal energy conversion: on | off (coupling_activity 1 | 0 on the storage class)
    'importer'              : 'on',   # Active resource import: on | off (the ActiveUptakeRule fires | not)
    'heredity'              : 'on',   # Information maintenance: on (HeredityAttachmentStep decides) | off (non-heritable control)
    'environment'           : 'vent',   # vent | generic (og-adv reject control: every driver at its floor from t = 0, no pore concentration)
    'h2_vent'               : 0.0108,   # Dissolved H2 of the vent fluid (M) [vg-data: calibrated, Phase-1 posterior draw]
    'delta_ph'              : 3.5,   # Vent - ocean pH contrast [vg-data: calibrated, Phase-1 posterior draw]
    'temperature'           : 322.15,   # Vent fluid temperature (K) [vg-data: calibrated, Phase-1 posterior draw]
    'dic'                   : 0.002,   # Ocean DIC / CO2 (M), never withdrawn [vg-data: extrapolated]
    'flow_velocity'         : 0.003,   # Vent flow through the reaction zone (m/s) [vg-data: Phase-1 posterior draw]
    'olivine_rate'          : 0.2,   # Serpentinization extent of the geology layer [vg-data: Phase-1 posterior draw]
    'ph_ocean'              : 6.0,   # Ocean-side pH [vg-data: extrapolated]
    'ablation_start'        : 3600.0,   # World time at which withdrawal begins (s)
    'ablation_duration'     : 6000.0,   # Duration of the linear withdrawal (s)
    'feed_floor'            : 0.01,   # Ocean floor of the feed fraction (H2, organics, N, P) [vg-data: hypothetical, swept]
    'gradient_floor'        : 0.0,   # Ocean floor of the gradient fraction
    'mineral_floor'         : -1.0,   # Ocean floor of the mineral access (residual catalytic mineral; < 0 = same as feed_floor) [vg-data: hypothetical]
    'tick_seconds'          : 60.0,   # World seconds per tick (every process runs at this interval)
    'store_seconds'         : 10.0,   # Interval of the storage class + its ledger, the cell's chemistry and the importer (s; the Phase-6 step)
    'heredity_clock_ratio'  : 0.016666666666666666,   # Lineage-time units of the heredity class per world second (one per tick) [vg-data: hypothetical]
    'coupling_activity'     : 1.0,   # Storage-class coupling activity when the pump is on [vg-data: hypothetical]
    'permeability_reference': 1e-06,   # H+ permeability of the fatty-acid reference membrane (m/s) [vg-data: extrapolated]
    'max_relative_leak'     : 100.0,   # Cap on a boundary class's leakiness relative to the fatty-acid vesicle [vg-data: hypothetical]
    'v_max_import'          : 1e-19,   # Importer capacity per substrate at full machinery (mol/s; ~6e4 molecules/s) [vg-data: hypothetical]
    'k_m_import'            : 1e-06,   # Importer half-saturation (M) [vg-data: hypothetical]
    'import_ratio_cap'      : 100.0,   # Inside / outside ratio at which the importer stops [vg-data: hypothetical]
    'import_setpoint'       : 0.3,   # The importer holds each substrate at this fraction of its in-vent level [vg-data: hypothetical]
    'import_budget_fraction': 0.3,   # Share of the lineage's work the importer may spend per tick [vg-data: hypothetical]
    'permeability_anchor'   : 'extrapolated',   # Charged-solute exchange permeabilities: extrapolated (Phase-8 classes) | measured (1000x tighter: the fatty-acid-vesicle charged-solute anchor, og-07-alt decision)
    'k_org'                 : 1e-05,   # Drive half-saturation on the cell's organic substrate (M) [vg-data: hypothetical]
    'k_h2'                  : 1e-05,   # Drive half-saturation on the cell's H2 (M; hydrogenase-like affinity) [vg-data: hypothetical]
    'function_reference'    : 0.25,   # Heredity-class mean function at which the lineage holds its full machinery [vg-data: hypothetical]
    'heredity_allocation'   : 0.05,   # Share of the lineage's work allocated to heredity [vg-data: hypothetical]
    'k_wash'                : 0.01,   # Pore feed / washout rate (1/s) [vg-data: inferred, Phase-8 canonical]
    'peclet'                : 8.0,   # Control Peclet of the multisolute transport class [Phase-8 operating point]
    'pore_flow_velocity'    : 1e-05,   # Flow of the single-tracer transport classes (m/s): accumulation Peclet ~1.8, inside og-03's resolvable range
    'feed_pyruvate'         : 0.0001,   # Organic feed the vent delivers at full support (M) [vg-data: inferred]
    'n_initial'             : 8,   # Founding individuals of the population
    'protocell_radius'      : 5e-07,   # Protocell radius (m): the cell's volume, membrane area and A/V [vg-data: inferred, Phase 6]
    'seed'                  : 1,   # Heredity-class RNG seed
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `gradient_support` | float | dimensionless | [0.0, 1.0] |
| `mineral_access` | float | dimensionless | [0.0, 1.0] |
| `feed_support` | float | dimensionless | [0.0, 1.0] |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `net_power` | float | watt |  |
| `material_net` | float | mole/second |  |
| `autonomy` | float | dimensionless | [0.0, 1.0] |
| `survivors` | float | dimensionless | [0.0, 1.0] |

**Conditions**

- `conservation_ledger` (invariant): `geology: pool == integral of the published source rates`
- `charge_ledger` (invariant): `interface: |anode - (carbon + HER)| / anode < 1e-9, faradaic efficiency <= 1, iv residual < 1e-6`
- `network_ledger` (invariant): `cell: element residual 0, carbon residual at integrator tolerance, dissipation >= 0`
- `ecology_ledger` (invariant): `cell + pore: total_X(t) = total_X(0) + in - out - degraded for C, N, P`
- `free_energy_ledger` (invariant): `store: in + E0 - work - dissipated = E(t), margin >= 0, efficiency <= 1`
- `information_ledger` (invariant): `heredity store: W(t) = W0 + inflow - spent`
- `rewrite_ledger` (invariant): `population: total(t) = total(t-1) - removed + external_flux across every _divide / _remove`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-08-alt-robust-innovation-set ===
STUDY = 'og-08-alt-robust-innovation-set'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Integrated_autonomy_world_full_set_at_the_canonical_structure_nominal_floor = 100
INTERVAL_Integrated_autonomy_world_full_set_at_the_canonical_structure_nominal_floor = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_integrated_autonomy_integrated_autonomy_world if 'ov_viva_genesis_composites_integrated_autonomy_integrated_autonomy_world' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_integrated_autonomy_integrated_autonomy_world = materialize_composite('viva_genesis.composites.integrated_autonomy.integrated_autonomy_world', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_integrated_autonomy_integrated_autonomy_world['state']}, core=core)
            _comp.run(STEPS_Integrated_autonomy_world_full_set_at_the_canonical_structure_nominal_floor)
        print('ran', 'Integrated autonomy world — full set at the canonical structure, nominal floor', f'(viva_genesis.composites.integrated_autonomy.integrated_autonomy_world) for', STEPS_Integrated_autonomy_world_full_set_at_the_canonical_structure_nominal_floor, 'steps')
    except Exception as _e:
        print(f"Integrated autonomy world — full set at the canonical structure, nominal floor not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**robust-innovation-lattice**


In [ ]:
# robust-innovation-lattice
show_viz(_render_one('local:RobustInnovationLattice', {'title': "Is og-08's escape set sufficient? An honest negative (Decision view)", 'report_json': 'workspace/studies/og-08-alt-robust-innovation-set/artifacts/robust_innovation_set.json'}, RUNS_DB, STUDY_YAML))

**autonomy-ablation-trace**


In [ ]:
# autonomy-ablation-trace
show_viz(_render_one('local:AutonomyAblationTrace', {'title': 'The integrated world through the ablation (Mechanism view)', 'report_json': 'workspace/studies/og-08-alt-robust-innovation-set/artifacts/robust_innovation_set.json'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| every-ledger-closes-over-the-integrated-world _(primary)_ | kind=derived path=every_ledger_ok_final detail=every ledger's ok flag (the information ledger by its exact energy-store residual) | op >= value 1.0 detail every ledger within tolerance |
| ablation-withdraws-every-driver _(supporting)_ | kind=derived path=ablation_complete_final detail=ablation phase == 2 on the final tick | op >= value 1.0 detail withdrawal complete |
| structural-events-logged _(supporting)_ | kind=derived path=structural_events_logged detail=cumulative event count | op >= value 3.0 detail events recorded |
| autonomy-decomposed-off-gradient _(diagnostic)_ | kind=derived path=settled_autonomy detail=settled autonomy.autonomy | op >= value 0.0 detail informative readout |
| net-work-budget-off-gradient _(diagnostic)_ | kind=derived path=settled_net_power_offgradient detail=settled autonomy.net_power | op > value -1.0 detail informative readout |
| minimum-sufficient-innovation-set _(primary)_ | kind=derived path=lattice_minimum_sufficient_set_resolved detail=1 when both conditions hold | op >= value 1.0 detail lattice resolved |
| minimum-sufficient-innovation-set-rich-ocean _(supporting)_ | kind=derived path=lattice_minimum_sufficient_set_resolved_rich detail=1 when both conditions hold at the rich floor | op >= value 1.0 detail lattice resolved |
| full-set-persistence-across-structures _(diagnostic)_ | kind=derived path=lattice_full_set_persistence_fraction detail=n persisting / n structures | op >= value 0.0 detail informative readout |
| robust-required-intersection _(diagnostic)_ | kind=derived path=lattice_robust_required_count detail=size of the intersection | op >= value 0.0 detail informative readout |
| og08-hypothesis-reproduced _(diagnostic)_ | kind=derived path=lattice_og08_intersection_matches_rich detail=intersection == og-08 set | op >= value 0.0 detail informative readout |
| generic-environment-rejected _(primary)_ | kind=derived path=lattice_generic_rejected detail=1 when the control is rejected | op >= value 1.0 detail control rejected |
| ensemble-ledgers-close _(primary)_ | kind=derived path=lattice_all_ledgers_ok detail=1 when all runs close | op >= value 1.0 detail all runs within tolerance |
| autonomy-of-the-persisting-lineages _(diagnostic)_ | kind=derived path=lattice_autonomy_persisting_any_rich detail=mean over persisting runs of any subset | op >= value 0.0 detail informative readout |
| ocean-floor-for-majority-persistence _(diagnostic)_ | kind=derived path=lattice_lowest_floor_majority_persists detail=from floor_map | op <= value 1.0 detail inside the swept range |
| per-driver-attribution-resolves-confound _(primary)_ | kind=observable detail=per-driver full-set persistence fraction + required intersection across 24 structures × 3 drivers × 8 subsets | op threshold detail differential per-driver survivability with the confound resolved and ledgers closing at every point |


### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| Escape from the vent needs internalised energy conversion (a pump), resource acquisition and heredity to hold them | the innovation lattice (lattice.required_intersection / required_union) | Martin & Russell 2007; Lane & Martin 2012; Sojo 2014 | Pump and heredity are required in every viable structure (1 / 1); the importer in 0 — resource acquisition is subsumed by permeant gases (CO2, H2) feeding the cell's own fixation and drive. |
| A leaky early membrane cannot hold a gradient or a concentrated pool; tight membranes come with pumps | the permeability-assumption axis (boundary class x anchor) in lattice.by_axis | Sojo 2014; Jordan 2019 (vesicle system) | Under the ecology-stage extrapolated leak the importer's cost (up to 1.9e-18 W, its full budget) exceeds its benefit; under the measured charged-solute anchor it costs ~1e-21 W but the best full-set lineage still sits 1e-19 W short — the leak sets the cost, the upkeep the margin. |


## Study: Bounding the pH endmember — pyroxene fraction and seawater-mixing dilution (`og-00b-ph-endmember-sweep`)

**Question.** How far does the og-00 habitability window move when the host-rock pyroxene fraction and the seawater-mixing dilution are varied, and does the habitable region survive at the acidic, dilute endmember the recent reassessment emphasises?

**Objective.** Sweep pyroxene_fraction (ol/px ratio) x seawater-mixing dilution on the serpentinizing vent-reactor composite, map the steady-state pH and the habitability indicator across the grid, and report the pH endmember bounds and how the habitable region deforms toward the acidic/dilute corner.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Vent reactor at the olivine endmember (og-00 canonical)` | `viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor`**

A reusable alkaline-vent reactor module. Module A = genuine PHREEQC serpentinization (phreeqpython + vendored llnl.dat): olivine dissolution + serpentine/brucite/magnetite equilibrium gives dissolved H2 (reducing power), alkalinity, heat, pe, and a charge-balanced fluid. Module B = genuine FiPy finite-volume advection-diffusion-reaction column transporting those sources across the vent-ocean interface and setting outlet T/pH/porosity. PHREEQC conserves mass+charge and FiPy's FVM conserves flux, satisfying the og-00 contract invariants; the coupled model is driven by a real IPhreeqc speciation (needs phreeqpython) and a real FiPy FVM solve (needs fipy).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor = {
    'flow_velocity': 0.001,   # Boundary water flux through the reaction zone (m/s)
    'olivine_rate' : 0.15,   # Olivine dissolution rate (mol per flow-saturated reaction step)
    'nx'           : 40,   # FiPy finite-volume cells across the interface column
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-00b-ph-endmember-sweep ===
STUDY = 'og-00b-ph-endmember-sweep'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Vent_reactor_at_the_olivine_endmember_og_00_canonical = 100
INTERVAL_Vent_reactor_at_the_olivine_endmember_og_00_canonical = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor if 'ov_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor = materialize_composite('viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_vent_reactor_serpentinizing_vent_reactor['state']}, core=core)
            _comp.run(STEPS_Vent_reactor_at_the_olivine_endmember_og_00_canonical)
        print('ran', 'Vent reactor at the olivine endmember (og-00 canonical)', f'(viva_genesis.composites.vent_reactor.serpentinizing_vent_reactor) for', STEPS_Vent_reactor_at_the_olivine_endmember_og_00_canonical, 'steps')
    except Exception as _e:
        print(f"Vent reactor at the olivine endmember (og-00 canonical) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

## Study: Why do the empirical and Butler-Volmer classes disagree on the limitation label? (`og-01-cal-dx-label-disagreement`)

**Question.** Is the empirical-vs-Butler-Volmer disagreement on the dominant limitation label (they agree on only 47% of the region both call favorable) a real structural difference in how each class represents electrical, catalytic, and transport limits, or an avoidable artifact of binning and threshold placement?

**Objective.** Isolate the source of the cross-class disagreement: decompose the empirical Michaelis drive against the Butler-Volmer overpotential terms point by point; test whether re-binning or a shared label threshold changes agreement; attribute the residual disagreement to a named model-structure cause.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Electrochemical interface — both classes, og-01-cal canonical grid` | `viva_genesis.composites.electrochemical_interface.electrochemical_carbon_fixation` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.electrochemical_interface.electrochemical_carbon_fixation`**

og-01's H2/pH-driven carbon fixation rebuilt as the Phase-2 electrochemical interface: the carbon-fixation mechanism is a model CLASS on the carbon_fixation face (model_carbon_fixation: empirical | butler_volmer | microkinetic). The empirical class is og-01's lab-calibrated gated kinetics driven by the real FiPy Nernst-Planck proton gradient + proton budget, with gradient_efficiency DERIVED from the wall's circuit (MineralConductivityProcess: conductivity, thickness, porosity vs the electrodes' charge-transfer resistance) instead of a knob. The butler_volmer class solves the vent wall as a galvanic cell each tick (Nernst half-cell potentials from the fluid redox state, Butler-Volmer kinetics scaled by catalyst activity / Ni fraction / T, Koutecky-Levich transport limits, the ohmic mineral + ionic path, the faradaic split over formate / acetate / pyruvate calibrated to the Altair-2025 current window and the Preiner-2020 yield hierarchy). The microkinetic class (CatMAP/Cantera) builds here and runs under pixi. Every class publishes the thermodynamic / electrical / catalytic / transport loss fractions and a limitation code, and a ChargeLedgerProcess audits its electron balance (Kirchhoff residual, faradaic efficiency <= 1, current-voltage residual -> ledger/charge_ledger_ok). Free parameters are the vg-data package 'electrochemical_interface'.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_electrochemical_interface_electrochemical_carbon_fixation = {
    'model_carbon_fixation': 'butler_volmer',   # Model class behind the 'carbon_fixation' face: butler_volmer | empirical | microkinetic
    'ph_vent'              : 9.5,   # Alkaline vent-side pH (from og-00)
    'ph_ocean'             : 6.0,   # Acidic early-ocean-side pH
    'h2'                   : 0.005,   # Dissolved H2 feed, vent side (M)
    'co2'                  : 0.002,   # Dissolved CO2 feed, ocean side (M)
    'temperature'          : 373.15,   # Interface temperature (K)
    'conductivity'         : 0.001,   # Electronic conductivity of the mineral layer (S/m) [vg-data: extrapolated]
    'layer_thickness'      : 0.001,   # Mineral layer thickness (m) [vg-data: inferred]
    'ni_fraction'          : 0.75,   # Ni mole fraction of the Fe/Ni catalytic phase (0 greigite .. 0.75 awaruite) [vg-data: measured]
    'porosity'             : 0.35,   # Wall porosity (ionic return path)
    'catalyst_active'      : 1.0,   # Active Fe/Ni/S mineral catalyst fraction (0 = no-catalyst control)
    'j0_co2'               : 1e-05,   # CO2-reduction exchange current density at Ni3Fe, 100 C (A/m^2) [vg-data: calibrated]
    'boundary_layer'       : 0.001,   # Diffusion boundary layer at the electrodes (m) [vg-data: inferred]
    'dilution_rate'        : 0.0,   # Open-flow washout (1/s); 0 = closed batch, >0 = sustained-throughput regime
    'nx'                   : 20,   # FiPy cells across the Nernst-Planck barrier
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-01-cal-dx-label-disagreement ===
STUDY = 'og-01-cal-dx-label-disagreement'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Electrochemical_interface_both_classes_og_01_cal_canonical_grid = 100
INTERVAL_Electrochemical_interface_both_classes_og_01_cal_canonical_grid = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_electrochemical_interface_electrochemical_carbon_fixation if 'ov_viva_genesis_composites_electrochemical_interface_electrochemical_carbon_fixation' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_electrochemical_interface_electrochemical_carbon_fixation = materialize_composite('viva_genesis.composites.electrochemical_interface.electrochemical_carbon_fixation', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_electrochemical_interface_electrochemical_carbon_fixation['state']}, core=core)
            _comp.run(STEPS_Electrochemical_interface_both_classes_og_01_cal_canonical_grid)
        print('ran', 'Electrochemical interface — both classes, og-01-cal canonical grid', f'(viva_genesis.composites.electrochemical_interface.electrochemical_carbon_fixation) for', STEPS_Electrochemical_interface_both_classes_og_01_cal_canonical_grid, 'steps')
    except Exception as _e:
        print(f"Electrochemical interface — both classes, og-01-cal canonical grid not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

## Study: How the survival region deforms with the speed of gradient loss (`og-05b-collapse-rate-sweep`)

**Question.** When the gradient-decline rate is swept and alternative collapse scenarios such as step loss and oscillatory decline are substituted, is survival governed by the speed of gradient loss, the total dose, or the terminal level?

**Objective.** Sweep decline_rate and substitute alternative collapse trajectories on the bioenergetic-internalization composite; measure where the survival region contracts or holds; report which feature of the collapse (rate, dose, or endpoint) controls the boundary.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Bioenergetic internalization (og-05 canonical)` | `viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization`**

A reusable bioenergetic-internalization module (Module H). An evolutionary control loop (replicator dynamics, Sojo-2014 leaky-membrane fitness) under a declining external gradient: as the geochemical pmf collapses, an efficient-pump population internalizes gradient generation and persists, while a weak-pump control collapses (the Sojo constraint). catalyst_efficiency sets the arm. Coarse-grained, deterministic reproduction (not wrapped ALife).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization = {
    'catalyst_efficiency': 1.0,   # Active-pump (Na+/H+) efficiency (1.0 efficient; ~0.3 weak-pump control)
    'v_start'            : 0.4,   # Initial external gradient (V)
    'decline_rate'       : 0.01,   # External-gradient decline per step (V/s)
    'gradient_floor'     : 0.02,   # Floor the external gradient decays to (V); low floor forces internalization-or-die
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-05b-collapse-rate-sweep ===
STUDY = 'og-05b-collapse-rate-sweep'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Bioenergetic_internalization_og_05_canonical = 100
INTERVAL_Bioenergetic_internalization_og_05_canonical = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization if 'ov_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization = materialize_composite('viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_bioenergetic_internalization_bioenergetic_internalization['state']}, core=core)
            _comp.run(STEPS_Bioenergetic_internalization_og_05_canonical)
        print('ran', 'Bioenergetic internalization (og-05 canonical)', f'(viva_genesis.composites.bioenergetic_internalization.bioenergetic_internalization) for', STEPS_Bioenergetic_internalization_og_05_canonical, 'steps')
    except Exception as _e:
        print(f"Bioenergetic internalization (og-05 canonical) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

## Study: Why is the importer a net cost, and which subset conditions permit persistence? (`og-08-alt-dx-importer-cost`)

**Question.** Why is the active importer a net cost under a leaky membrane, which membrane-tightness or structure conditions would flip it to a net benefit, and what distinguishes the few structures that persist under {pump, heredity} from the many that fail?

**Objective.** Decompose the importer's per-structure energy/mass ledger (uptake vs leak) across the innovation lattice; sweep membrane permeability to locate any tightness at which the importer becomes net beneficial; characterise what the 2 persisting {pump, heredity} structures share that the other 22 lack.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Integrated autonomy world (og-08-alt canonical structure)` | `viva_genesis.composites.integrated_autonomy.integrated_autonomy_world` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.integrated_autonomy.integrated_autonomy_world`**

og-08's off-gradient test inside ONE persistent world assembled from the merged phases: the lineage lives on what real processes deliver — the external pmf tapped through its membrane leak and an internal store charged by the drive its live substrate pool sustains (an energy_coupling class, the STORAGE MODEL), organic carbon reaching it through a localization class (the TRANSPORT LAW) and the Phase-8 exchange rules at a boundary class's leakiness (the PERMEABILITY ASSUMPTION), the Phase-3 universe inside it catalysed by the mineral surface or by the machinery its heredity class holds (the INHERITANCE CLASS), upkeep priced by a MAINTENANCE LAW, and a population of individuals that divide and go extinct by structural rewrites. An AblationStep withdraws the gradient, the mineral access and the feed; pump (coupling_activity), importer (the ActiveUptakeRule) and heredity (attachment) are the three innovations; environment=generic is the og-adv reject control. Seven ledgers run beside the world and a structural-event log records every event. The vent endmembers are one Phase-1 posterior draw (vg-data integrated_autonomy).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_integrated_autonomy_integrated_autonomy_world = {
    'model_energy_coupling' : 'thioester_acyl_phosphate',   # Model class behind the 'energy_coupling' face: fe_redox | h_na_gradient_leaky_membrane | passive_gradient | polyphosphate | thioester_acyl_phosphate
    'model_heredity'        : 'rna_like',   # Model class behind the 'heredity' face: compositional | hybrid | mineral_state | network | peptide | rna_like
    'model_boundary'        : 'single_chain',   # Model class behind the 'boundary' face: mixed | single_chain | thioester_aminothiol
    'model_localization'    : 'multisolute_surrogate',   # Model class behind the 'localization' face: fenics | fenics_multisolute_3d | fipy | multisolute_surrogate | surrogate
    'maintenance_law'       : 'constant',   # Maintenance function: constant (Phase 6) | leak_repair (scales with leakiness) | turnover (scales with machinery held)
    'pump'                  : 'on',   # Internal energy conversion: on | off (coupling_activity 1 | 0 on the storage class)
    'importer'              : 'on',   # Active resource import: on | off (the ActiveUptakeRule fires | not)
    'heredity'              : 'on',   # Information maintenance: on (HeredityAttachmentStep decides) | off (non-heritable control)
    'environment'           : 'vent',   # vent | generic (og-adv reject control: every driver at its floor from t = 0, no pore concentration)
    'h2_vent'               : 0.0108,   # Dissolved H2 of the vent fluid (M) [vg-data: calibrated, Phase-1 posterior draw]
    'delta_ph'              : 3.5,   # Vent - ocean pH contrast [vg-data: calibrated, Phase-1 posterior draw]
    'temperature'           : 322.15,   # Vent fluid temperature (K) [vg-data: calibrated, Phase-1 posterior draw]
    'dic'                   : 0.002,   # Ocean DIC / CO2 (M), never withdrawn [vg-data: extrapolated]
    'flow_velocity'         : 0.003,   # Vent flow through the reaction zone (m/s) [vg-data: Phase-1 posterior draw]
    'olivine_rate'          : 0.2,   # Serpentinization extent of the geology layer [vg-data: Phase-1 posterior draw]
    'ph_ocean'              : 6.0,   # Ocean-side pH [vg-data: extrapolated]
    'ablation_start'        : 3600.0,   # World time at which withdrawal begins (s)
    'ablation_duration'     : 6000.0,   # Duration of the linear withdrawal (s)
    'feed_floor'            : 0.01,   # Ocean floor of the feed fraction (H2, organics, N, P) [vg-data: hypothetical, swept]
    'gradient_floor'        : 0.0,   # Ocean floor of the gradient fraction
    'mineral_floor'         : -1.0,   # Ocean floor of the mineral access (residual catalytic mineral; < 0 = same as feed_floor) [vg-data: hypothetical]
    'tick_seconds'          : 60.0,   # World seconds per tick (every process runs at this interval)
    'store_seconds'         : 10.0,   # Interval of the storage class + its ledger, the cell's chemistry and the importer (s; the Phase-6 step)
    'heredity_clock_ratio'  : 0.016666666666666666,   # Lineage-time units of the heredity class per world second (one per tick) [vg-data: hypothetical]
    'coupling_activity'     : 1.0,   # Storage-class coupling activity when the pump is on [vg-data: hypothetical]
    'permeability_reference': 1e-06,   # H+ permeability of the fatty-acid reference membrane (m/s) [vg-data: extrapolated]
    'max_relative_leak'     : 100.0,   # Cap on a boundary class's leakiness relative to the fatty-acid vesicle [vg-data: hypothetical]
    'v_max_import'          : 1e-19,   # Importer capacity per substrate at full machinery (mol/s; ~6e4 molecules/s) [vg-data: hypothetical]
    'k_m_import'            : 1e-06,   # Importer half-saturation (M) [vg-data: hypothetical]
    'import_ratio_cap'      : 100.0,   # Inside / outside ratio at which the importer stops [vg-data: hypothetical]
    'import_setpoint'       : 0.3,   # The importer holds each substrate at this fraction of its in-vent level [vg-data: hypothetical]
    'import_budget_fraction': 0.3,   # Share of the lineage's work the importer may spend per tick [vg-data: hypothetical]
    'permeability_anchor'   : 'extrapolated',   # Charged-solute exchange permeabilities: extrapolated (Phase-8 classes) | measured (1000x tighter: the fatty-acid-vesicle charged-solute anchor, og-07-alt decision)
    'k_org'                 : 1e-05,   # Drive half-saturation on the cell's organic substrate (M) [vg-data: hypothetical]
    'k_h2'                  : 1e-05,   # Drive half-saturation on the cell's H2 (M; hydrogenase-like affinity) [vg-data: hypothetical]
    'function_reference'    : 0.25,   # Heredity-class mean function at which the lineage holds its full machinery [vg-data: hypothetical]
    'heredity_allocation'   : 0.05,   # Share of the lineage's work allocated to heredity [vg-data: hypothetical]
    'k_wash'                : 0.01,   # Pore feed / washout rate (1/s) [vg-data: inferred, Phase-8 canonical]
    'peclet'                : 8.0,   # Control Peclet of the multisolute transport class [Phase-8 operating point]
    'pore_flow_velocity'    : 1e-05,   # Flow of the single-tracer transport classes (m/s): accumulation Peclet ~1.8, inside og-03's resolvable range
    'feed_pyruvate'         : 0.0001,   # Organic feed the vent delivers at full support (M) [vg-data: inferred]
    'n_initial'             : 8,   # Founding individuals of the population
    'protocell_radius'      : 5e-07,   # Protocell radius (m): the cell's volume, membrane area and A/V [vg-data: inferred, Phase 6]
    'seed'                  : 1,   # Heredity-class RNG seed
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-08-alt-dx-importer-cost ===
STUDY = 'og-08-alt-dx-importer-cost'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Integrated_autonomy_world_og_08_alt_canonical_structure = 100
INTERVAL_Integrated_autonomy_world_og_08_alt_canonical_structure = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_integrated_autonomy_integrated_autonomy_world if 'ov_viva_genesis_composites_integrated_autonomy_integrated_autonomy_world' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_integrated_autonomy_integrated_autonomy_world = materialize_composite('viva_genesis.composites.integrated_autonomy.integrated_autonomy_world', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_integrated_autonomy_integrated_autonomy_world['state']}, core=core)
            _comp.run(STEPS_Integrated_autonomy_world_og_08_alt_canonical_structure)
        print('ran', 'Integrated autonomy world (og-08-alt canonical structure)', f'(viva_genesis.composites.integrated_autonomy.integrated_autonomy_world) for', STEPS_Integrated_autonomy_world_og_08_alt_canonical_structure, 'steps')
    except Exception as _e:
        print(f"Integrated autonomy world (og-08-alt canonical structure) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

## Study: Coupled Geochemical Reaction Interface (`og-27-coupled-geochemical-reaction-interface`)

**Question.** Can an evolving mineral–water interface sustainably supply carbon, H2, chemical gradients and catalysis to a prebiotic reaction network — and over what timescale, before mineral evolution (precipitation, passivation) chokes that supply?

**Objective.** Build a cell-resolved geochemical reaction interface that evolves its own mineral surfaces, and quantify the time- and space-resolved supply of reducing power, carbon, gradients and catalytic surface it hands to the downstream network (study 29), with closed mass/charge/energy ledgers.

**Hypothesis.** A cell-resolved coupling of aqueous speciation (PHREEQC/PhreeqcRM), reactive transport (FiPy) and multiphase mineral equilibrium (Reaktoro), with explicit precipitation/dissolution and surface-passivation dynamics, will show H2, inorganic-carbon and gradient delivery sustained over transport timescales but declining on the slower mineral-evolution timescale as reactive surfaces passivate — i.e. the interface is a depleting supply with a measurable crossover time, not a perpetual one.

**Claim.** An evolving mineral–water interface at a Lost City–type vent can be modeled as a coupled geochemical reactor whose precipitation, dissolution and surface passivation reshape — not merely sustain — its delivery of H2, inorganic carbon, pH/redox gradients and catalytic surface. Whether that delivery stays above the level a downstream reaction network needs, and for how long before mineral evolution throttles it, is the quantity this study measures; sustained reactive flux is a result to demonstrate, not an assumption.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- serpentinization-driven H2 + alkalinity generation at the vent wall
- mixing-controlled pH/redox/ionic-activity gradients across the interface
- mineral precipitation/dissolution that opens and closes reactive porosity
- surface passivation degrading catalytic accessibility over time

**Proposed models / methods**

- PHREEQC/PhreeqcRM cell-resolved aqueous speciation + mineral reactions  [build_status: wrapper-needed]
- FiPy finite-volume multispecies reactive transport                      [in-workspace: viva_genesis.transport]
- Reaktoro multiphase equilibrium cross-check                             [in-workspace: viva_genesis.geochemistry]
- FEniCSx Poisson/Nernst–Planck verification of electrochemical driving forces [exists: viva_fenics]
- PetDB/EarthChem rock & mineral constraints for boundary composition

**Key assumptions**

- The mineral surface is dynamic (it precipitates, dissolves and passivates); a static-surface model is the control it must beat.
- Supply is a sustained-flux property over mineral-evolution time, not a terminal-state concentration.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Coupled Geochemical Reaction Interface` | `viva_genesis.composites.og_27_coupled_geochemical_reaction_interface.og_27_coupled_geochemical_reaction_interface` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.og_27_coupled_geochemical_reaction_interface.og_27_coupled_geochemical_reaction_interface`**

Study 27: og-00's real serpentinizing vent made into an EVOLVING interface. geochemical_world (genuine PHREEQC speciation -> genuine FiPy transport) is augmented with MineralSurfaceDynamicsProcess: the reacted fluid's real carbonate saturation index drives precipitation/dissolution (a porosity delta that composes with FiPy's) and passivates the catalytic surface over time. Reports time- and space-resolved supply of H2/carbon/gradients/catalysis and the crossover time at which mineral evolution throttles it, with closed mass/charge/energy ledgers.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_og_27_coupled_geochemical_reaction_interface_og_27_coupled_geochemical_reaction_interface = {
    'flow_velocity': 0.001,   # Boundary water flux through the reaction zone (m/s)
    'olivine_rate' : 0.15,   # Olivine reaction extent per flow-saturated step
    'fluid_dic'    : 0.002,   # Inflowing-fluid DIC, Hadean-ocean proxy (molar)
    'nx'           : 40,   # FiPy finite-volume cells across the interface column
    'k_precip'     : 0.05,   # Precipitation rate coefficient (pore clogging, 1/s)
    'k_dissolve'   : 0.02,   # Dissolution rate coefficient (pore opening, 1/s)
    'k_passivate'  : 0.5,   # Catalytic passivation coefficient under precipitation
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature` | float | kelvin | [273.0, 673.0] |
| `H2` | float | molar | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `reactive_flux` | float | mol/second |  |
| `interface_potential` | float | volt |  |
| `catalyst_accessibility` | float | dimensionless |  |

**Conditions**

- `mass_charge_balance` (invariant): `abs(residual_C)+abs(residual_H)+abs(residual_O)+abs(residual_charge) < tol`
- `reproducibility` (invariant): `solver and db and seed and scenario recorded`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-27-coupled-geochemical-reaction-interface ===
STUDY = 'og-27-coupled-geochemical-reaction-interface'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Coupled_Geochemical_Reaction_Interface = 100
INTERVAL_Coupled_Geochemical_Reaction_Interface = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_27_coupled_geochemical_reaction_interface_og_27_coupled_geochemical_reaction_interface if 'ov_viva_genesis_composites_og_27_coupled_geochemical_reaction_interface_og_27_coupled_geochemical_reaction_interface' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_27_coupled_geochemical_reaction_interface_og_27_coupled_geochemical_reaction_interface = materialize_composite('viva_genesis.composites.og_27_coupled_geochemical_reaction_interface.og_27_coupled_geochemical_reaction_interface', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_27_coupled_geochemical_reaction_interface_og_27_coupled_geochemical_reaction_interface['state']}, core=core)
            _comp.run(STEPS_Coupled_Geochemical_Reaction_Interface)
        print('ran', 'Coupled Geochemical Reaction Interface', f'(viva_genesis.composites.og_27_coupled_geochemical_reaction_interface.og_27_coupled_geochemical_reaction_interface) for', STEPS_Coupled_Geochemical_Reaction_Interface, 'steps')
    except Exception as _e:
        print(f"Coupled Geochemical Reaction Interface not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**coupled-interface-evolution**


In [ ]:
# coupled-interface-evolution
show_viz(_render_one('local:OgCoupledInterfaceViz', {'composite': 'viva_genesis.composites.og_27_coupled_geochemical_reaction_interface.og_27_coupled_geochemical_reaction_interface', 'n_steps': 80, 'title': 'The interface evolves — supply vs self-throttling', 'caption': 'Top row — outlet temperature, pH and H2 flux: the conditions and reducing power the vent wall delivers. Bottom row — porosity closing, catalytic accessibility passivating, and carbonate saturation index driving precipitation. Plainly: the wall delivers chemistry at first, but as minerals grow and clog it that delivery declines — the interface depletes rather than lasting forever.'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_og_27_coupled_geochemical_reaction_interface_og_27_coupled_geochemical_reaction_interface if 'ov_viva_genesis_composites_og_27_coupled_geochemical_reaction_interface_og_27_coupled_geochemical_reaction_interface' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| sustained-reactive-flux-over-timescales _(primary)_ | kind=observable detail=time- and space-resolved H2 + inorganic-carbon flux and pH/redox gradient over transport timescales | op threshold detail delivery at or above the downstream-required level over the transport timescale |
| mineral-evolution-supply-decay _(primary)_ | kind=observable detail=catalytic supply / reactive surface vs. the slower mineral-evolution timescale; crossover time below threshold | op threshold detail crossover time quantified (depleting-supply behaviour characterised) |
| electrochemical-driving-force-verified _(secondary)_ | kind=observable detail=interface potential from FiPy transport vs. independent FEniCSx Nernst–Planck solve | op threshold detail non-trivial driving force, cross-solver agreement to tolerance |
| coupled-conservation _(secondary)_ | kind=observable detail=C/H/O/charge/energy residuals across the coupled solve | op threshold detail residuals within tolerance; no untracked feed |
| handoff-27-to-29-complete _(secondary)_ | kind=observable detail=presence + typing of every field in the 27→29 transfer bundle | op threshold detail bundle complete and typed (satisfies 27→29 handoff validator) |
| dic-crossover-sweep _(primary)_ | kind=observable detail=H2 flux / precipitation / accessibility / porosity vs fluid_dic across SI=0 ledger closure=None | op threshold detail the crossover is characterized with the ledger closing at every point (window_exists resolved either way) |


## Study: Prebiotic Reaction Atlas (`og-28-prebiotic-reaction-atlas`)

**Question.** Which measured or defensibly inferred reaction pathways connect C1–C3 products to functional chemistry, and with what confidence?

**Objective.** Build and version the shared prebiotic-reaction-atlas (schema on PR #69): import/curate candidate reactions from Rhea/ChEBI/ORD plus primary-paper extraction, run deterministic chemistry validation (ChEBI identity, mass/charge balance, eQuilibrator ΔG), assign confidence tiers and kinetic domains, and expose the 28→29 transfer bundle.

**Hypothesis.** The primary literature plus curated reaction databases, filtered through balance, thermodynamic feasibility and kinetic-accessibility checks, yield a connected network from C1–C3 species to key functional products in which the measured-tier core is sparse but real, the inferred tier extends it under explicit thermodynamic/kinetic assumptions, and the hypothetical tier is flagged — and keeping these three tiers separate is precisely what stops enzyme-catalyzed or synthetic-only reactions from leaking into prebiotic models.

**Claim.** A versioned, provenance-tracked atlas of candidate prebiotic reactions — each with canonical species identities, a mass- and charge-balanced equation, a confidence tier (measured / inferred / hypothetical) and a kinetic domain — can connect C1–C3 carbon products to functional downstream chemistry while guaranteeing that no unbalanced or unsourced reaction ever enters an accepted model. The atlas's worth is measured by its coverage and the defensibility of its tiers, not by any single simulation.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- separate reaction IDENTIFICATION (Rhea/ORD) from thermodynamic FEASIBILITY (eQuilibrator) from kinetic ACCESSIBILITY (rate form + domain) — never infer abiotic kinetics from enzyme catalysis
- confidence tiering: measured | inferred | hypothetical
- deterministic chemistry validation as a write-time gate into atlas.vN.jsonl

**Proposed models / methods**

- Rhea + ChEBI canonical identities & balanced reaction definitions
- ORD structured organic-reaction evidence; RDKit structure/reaction validation + precursor enumeration [to-build: viva_reaction_atlas]
- eQuilibrator component-contribution ΔG feasibility screen                [exists: viva_equilibrator]
- AI-assisted literature extraction with deterministic validators + expert review [to-build: viva_reaction_atlas.extract — every record gated]
- KIDA/NIST selective kinetic references (caution: mostly gas-phase/astrochemical, not aqueous-vent)

**Key assumptions**

- Reaction identification, thermodynamic feasibility and kinetic accessibility are three separate gates; a database reaction is not assumed prebiotically viable.
- Enzyme-catalysed kinetics (Rhea) do NOT license abiotic kinetic claims.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Prebiotic Reaction Atlas` | `viva_genesis.composites.og_28_prebiotic_reaction_atlas.og_28_prebiotic_reaction_atlas` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.og_28_prebiotic_reaction_atlas.og_28_prebiotic_reaction_atlas`**

Study 28: build the versioned Prebiotic Reaction Atlas. Runs the real extract->validate pipeline over the measured-tier carbon-fixation seed and reports coverage + the zero-leakage guarantee: accepted_balance_violations == 0 while an unbalanced candidate is held back in proposed. Produces the shared data asset consumed by studies 29/30/31.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_og_28_prebiotic_reaction_atlas_og_28_prebiotic_reaction_atlas = {
    'run_feasibility': False,   # Attach eQuilibrator ΔG feasibility (advisory; needs the real bridge)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `candidate_reactions` | integer | dimensionless | >= 0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `balanced_fraction` | float | dimensionless | [0.0, 1.0] |
| `provenance_complete_fraction` | float | dimensionless | [0.0, 1.0] |

**Conditions**

- `every_reaction_balanced` (invariant): `every accepted reaction mass- and charge-balanced (balanced_fraction == 1.0)`
- `every_reaction_sourced` (invariant): `every accepted reaction carries species identity + source provenance + confidence tier`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-28-prebiotic-reaction-atlas ===
STUDY = 'og-28-prebiotic-reaction-atlas'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Prebiotic_Reaction_Atlas = 100
INTERVAL_Prebiotic_Reaction_Atlas = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_28_prebiotic_reaction_atlas_og_28_prebiotic_reaction_atlas if 'ov_viva_genesis_composites_og_28_prebiotic_reaction_atlas_og_28_prebiotic_reaction_atlas' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_28_prebiotic_reaction_atlas_og_28_prebiotic_reaction_atlas = materialize_composite('viva_genesis.composites.og_28_prebiotic_reaction_atlas.og_28_prebiotic_reaction_atlas', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_28_prebiotic_reaction_atlas_og_28_prebiotic_reaction_atlas['state']}, core=core)
            _comp.run(STEPS_Prebiotic_Reaction_Atlas)
        print('ran', 'Prebiotic Reaction Atlas', f'(viva_genesis.composites.og_28_prebiotic_reaction_atlas.og_28_prebiotic_reaction_atlas) for', STEPS_Prebiotic_Reaction_Atlas, 'steps')
    except Exception as _e:
        print(f"Prebiotic Reaction Atlas not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**reaction-atlas-coverage**


In [ ]:
# reaction-atlas-coverage
show_viz(_render_one('local:OgReactionAtlasCoverageViz', {'composite': 'viva_genesis.composites.og_28_prebiotic_reaction_atlas.og_28_prebiotic_reaction_atlas', 'n_steps': 3, 'title': 'Reaction atlas — coverage, confidence, and zero leakage', 'caption': "Three bars from the atlas's real coverage emit — accepted reactions by confidence tier (measured / inferred / hypothetical), accepted versus proposed-and-held, and the zero-leakage gate (admitted-unbalanced = 0), with the count of C-species connected annotated. Plainly: how much defensible chemistry the atlas connects, how sure we are of each part, and proof that nothing unbalanced or unsourced got in."}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_og_28_prebiotic_reaction_atlas_og_28_prebiotic_reaction_atlas if 'ov_viva_genesis_composites_og_28_prebiotic_reaction_atlas_og_28_prebiotic_reaction_atlas' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| inferred-tier-has-thermo-and-rate-basis _(primary)_ | kind=observable detail=fraction of inferred-tier reactions carrying a ΔG feasibility flag + justified rate form/range | op threshold detail every inferred reaction has an explicit thermo + rate basis |
| atlas-write-gate-zero-leakage _(primary)_ | kind=observable detail=count of unbalanced / unsourced records admitted to atlas.vN.jsonl (target 0) | op threshold detail zero balance/provenance violations in the accepted atlas |
| handoff-28-to-29-complete _(secondary)_ | kind=observable detail=presence + typing of every field in the 28→29 transfer bundle | op threshold detail bundle complete and typed (satisfies 28→29 handoff validator) |


## Study: Endogenous Autocatalysis (`og-29-endogenous-autocatalysis`)

**Question.** Can vent-fed chemistry spontaneously form network amplification or self-maintaining cycles — genuine, balanced autocatalysis rather than accumulation driven by the input feed?

**Objective.** Assemble vent-fed reaction networks from the 27 and 28 handoff bundles, detect stoichiometric autocatalysis, simulate kinetics with dilution, and test any detected amplification against a matched null — reporting either genuine autocatalysis with its critical dilution, or a clean negative that names the gap.

**Hypothesis.** Given the 27 interface feeds and the 28 reaction set, stoichiometric autocatalysis detection plus kinetic simulation under dilution will reveal whether a self-amplifying cycle exists; if it does, its amplification and net carbon fixation exceed a matched null (identical input fluxes, autocatalytic topology removed) and persist above a critical dilution rate; if it does not, the analysis isolates the missing precursor or rate-limiting step.

**Claim.** Vent-fed chemistry, assembled from the study-28 atlas reactions under the study-27 interface feeds, either forms a genuinely balanced, mass-conserving autocatalytic cycle whose amplification beats a matched null with equal chemical input fluxes — or it does not, in which case the study names the specific missing precursor or rate-limiting mechanism. Spontaneous network amplification is a claim won against a null, never inferred from apparent growth.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- network amplification / self-maintaining cycles seeded from atlas reactions
- stoichiometric autocatalysis (a subnetwork that produces its own catalyst/seed)
- dilution as the selective pressure separating true autocatalysis from feed-driven accumulation

**Proposed models / methods**

- RDKit rule-generated reaction candidates + curated atlas alternatives [to-build: reaction-rule-generation / viva_reaction_atlas]
- stoichiometric autocatalysis detection on the reaction network (algebraic, pre-kinetic)
- COPASI/libRoadRunner kinetic simulation with dilution                 [in-workspace: viva_genesis.protometabolism]
- discrete/stochastic kinetics where expected molecule counts make continuum concentrations unreliable (evidence standard
- eQuilibrator ΔG screen so any detected cycle is thermodynamically allowed [exists: viva_equilibrator]
- matched-null construction + PyMC/SALib sensitivity & identifiability   [partial: viva_genesis.uq]

**Key assumptions**

- Autocatalysis must be stoichiometrically balanced — an apparent cycle fed by an untracked source is rejected, not counted.
- A matched non-concentrating / non-autocatalytic null is the control the favourable case must beat.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Endogenous Autocatalysis` | `viva_genesis.composites.og_29_endogenous_autocatalysis.og_29_endogenous_autocatalysis` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.og_29_endogenous_autocatalysis.og_29_endogenous_autocatalysis`**

Study 29: is vent-fed chemistry self-amplifying? Reuses open_chemical_network's real reaction-universe engine + autocatalysis detector, but feeds the network from the ENDOGENOUS 27->29 supply (H2 + inorganic carbon from the evolving geochemical interface) rather than a prescribed feed. matched_null swaps the degree-preserving null on the same supply for the Gate A comparison (chemical input fluxes held equal). Reports amplification (autocatalytic_gain), net carbon fixation, intermediate turnover and the critical dilution (k_wash) at which the loop washes out.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_og_29_endogenous_autocatalysis_og_29_endogenous_autocatalysis = {
    'supply_H2'   : 0.001,   # Endogenous H2 supply from the vent interface (molar) [27->29 handoff]
    'supply_CO2'  : 0.002,   # Endogenous inorganic-carbon supply (molar) [27->29 handoff]
    'k_wash'      : 0.05,   # CSTR washout / dilution rate (1/s) — the critical-dilution knob
    'max_carbons' : 4,   # Carbon cap of the generated reaction universe
    'matched_null': False,   # Gate A arm: run the degree-preserving null on the SAME supply
    'null_seed'   : 0,   # Null-network draw (matched_null only)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `feed_rate` | float | molar/second | >= 0.0 |
| `dilution_rate` | float | 1/second | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `autocatalytic_gain` | float | dimensionless |  |
| `net_carbon_fixation` | float | mol/second |  |
| `critical_dilution` | float | 1/second |  |

**Conditions**

- `balanced_autocatalysis` (invariant): `any detected autocatalytic cycle is mass- and charge-balanced (no phantom source)`
- `null_controls_fail` (invariant): `matched non-autocatalytic nulls show no amplification above washout`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-29-endogenous-autocatalysis ===
STUDY = 'og-29-endogenous-autocatalysis'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Endogenous_Autocatalysis = 100
INTERVAL_Endogenous_Autocatalysis = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_29_endogenous_autocatalysis_og_29_endogenous_autocatalysis if 'ov_viva_genesis_composites_og_29_endogenous_autocatalysis_og_29_endogenous_autocatalysis' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_29_endogenous_autocatalysis_og_29_endogenous_autocatalysis = materialize_composite('viva_genesis.composites.og_29_endogenous_autocatalysis.og_29_endogenous_autocatalysis', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_29_endogenous_autocatalysis_og_29_endogenous_autocatalysis['state']}, core=core)
            _comp.run(STEPS_Endogenous_Autocatalysis)
        print('ran', 'Endogenous Autocatalysis', f'(viva_genesis.composites.og_29_endogenous_autocatalysis.og_29_endogenous_autocatalysis) for', STEPS_Endogenous_Autocatalysis, 'steps')
    except Exception as _e:
        print(f"Endogenous Autocatalysis not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**endogenous-autocatalysis-gate-a**


In [ ]:
# endogenous-autocatalysis-gate-a
show_viz(_render_one('local:OgEndogenousAutocatalysisViz', {'composite': 'viva_genesis.composites.og_29_endogenous_autocatalysis.og_29_endogenous_autocatalysis', 'n_steps': 120, 'title': 'Autocatalysis under flow — and the Gate-A null test', 'caption': 'Net carbon flux and the autocatalytic gain of the vent-fed network under flow, then the Gate-A comparison against a degree-preserving matched null run at identical input fluxes. Plainly: is the network fixing and multiplying carbon, and does that amplification beat a look-alike control fed exactly the same — or was the apparent growth just the feed?'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_og_29_endogenous_autocatalysis_og_29_endogenous_autocatalysis if 'ov_viva_genesis_composites_og_29_endogenous_autocatalysis_og_29_endogenous_autocatalysis' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| receives-27-handoff-bundle _(secondary)_ | kind=observable detail=completeness + typing of the 27→29 bundle checked against 27's emitted bundle | op threshold detail bundle received complete and typed |
| receives-28-handoff-bundle _(secondary)_ | kind=observable detail=completeness + typing of the 28→29 bundle checked against 28's emitted bundle | op threshold detail bundle received complete and typed |
| stoichiometric-autocatalysis-detection _(primary)_ | kind=observable detail=detected autocatalytic cycle; amplification factor + net carbon fixation when positive | op threshold detail balanced autocatalysis detected, OR a clean no-detection result |
| autocatalysis-beats-matched-null _(primary)_ | kind=observable detail=amplification in the autocatalytic network vs. matched null with equal input fluxes | op threshold detail amplification margin over the matched null (else reported as feed-driven accumulation) |
| critical-dilution-threshold _(secondary)_ | kind=observable detail=autocatalytic gain vs. dilution rate; critical dilution at collapse | op threshold detail critical dilution rate reported (robustness envelope characterised) |
| carbon-conservation-ledger _(secondary)_ | kind=observable detail=C/mass/charge residuals across the kinetic run | op threshold detail residuals within tolerance; 27 feeds the only source |
| negative-result-isolates-gap _(primary)_ | kind=observable detail=on no-detection, the identified missing precursor / rate-limiting mechanism | op threshold detail balanced autocatalysis demonstrated OR a named missing precursor/mechanism |


## Study: Reductive-TCA Autocatalysis (branched v2) (`og-rtca-autocatalysis`)

**Question.** Does the canonical reductive-TCA cycle — the branched-C6 route that the linear-skeleton formose universe (og-29) cannot represent — turn over and self-amplify under vent CO2/H2 feeds (feasibility), and does its specific wiring self-amplify beyond a degree-preserving matched null (topology-specificity)?

**Objective.** Build the curated mechanistic rTCA network as a real libRoadRunner reaction universe (same engine, feeds, eQuilibrator thermodynamics and matched-null machinery as Phase 3), grade it by the functional autocatalysis discriminator across the ignition sweep against a degree-preserving matched null, and report feasibility + topology-specificity honestly with the governing mechanism.

**Hypothesis.** Curated with its real branched intermediates (citrate/isocitrate/cis-aconitate) and eQuilibrator thermodynamics, the rTCA cycle's citrate-lyase branch (citrate → OAA + acetyl, acetyl recycled to a second OAA) is autocatalytic by topology (every elementary step 1:1, no coefficient-2 reaction). A functional, coefficient-independent discriminator will reveal whether that topology actually self-amplifies under vent feeds; if it does not, the analysis names the rate-limiting/thermodynamic step rather than tuning a pass.

**Claim.** Under vent conditions (H2 as reductant) a seeded reductive-TCA cycle does not self-amplify - it washes out across the x1-x50 feed sweep. The cycle is genuinely autocatalytic in topology: its net turn, 1 oxaloacetate + 4 CO2 + 5 H2 -> 2 oxaloacetate + 3 H2O, doubles the carrier oxaloacetate with no explicit coefficient-2 doubling reaction, so a coefficient detector reads zero by construction and a coefficient-free functional discriminator is required to judge it. Under vent H2 it fails that test: the reductive carboxylations that close the cycle (succinate to alpha-ketoglutarate, +18 kJ/mol; acetyl to pyruvate, +31 kJ/mol) run uphill and too slow against washout - the same reason the biological reductive TCA cycle uses ferredoxin, not H2, as its reductant. In a labelled permissive counterfactual that lowers the carboxylation barrier, the same topology meets the self-amplification bar and is topology-specific (a degree-preserving null does not), so the vent negative is a redox-environment limitation, not a defect of the cycle's wiring. The counterfactual is explicitly not a vent claim.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Curated mechanistic rTCA (vent-fed)` | `viva_genesis.composites.og_rtca_autocatalysis.og_rtca_autocatalysis` | 0 | supply_H2=0.02, supply_CO2=0.04, k_wash=0.02, matched_null=False, carboxylation_drive_kj=0.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.og_rtca_autocatalysis.og_rtca_autocatalysis`**

Curated MECHANISTIC reductive-TCA cycle (9 elementary steps incl. the citrate lyase branch; branched C6 intermediates citrate/isocitrate/cis-aconitate with eQuilibrator dG'; acetate as the acetyl-CoA surrogate; fumarate lumped) on the real libRoadRunner engine under vent CO2/H2 feed + washout. Its autocatalysis is topological (no coefficient-2 reaction), so the coefficient-gated autocatalytic_gain reads 0 by construction; feasibility + topology-specificity are graded by the coefficient-independent functional discriminator. matched_null swaps the degree-preserving null; carboxylation_drive_kj is a labelled stronger-reductant counterfactual (0 = real vent H2) that separates topology from thermodynamics.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_og_rtca_autocatalysis_og_rtca_autocatalysis = {
    'supply_H2'             : 0.001,   # Vent H2 supply (molar) [27->rtca handoff]
    'supply_CO2'            : 0.002,   # Vent inorganic-carbon supply (molar) [27->rtca handoff]
    'k_wash'                : 0.02,   # CSTR washout / dilution rate (1/s) — the critical-dilution knob
    'matched_null'          : False,   # Topology-specificity arm: degree-preserving null on the same feed
    'null_seed'             : 0,   # Null-network draw (matched_null only)
    'carboxylation_drive_kj': 0.0,   # Labelled counterfactual: kJ/mol subtracted from each reductive carboxylation dG' (stronger reductant than H2). 0 = real vent.
}

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-rtca-autocatalysis ===
STUDY = 'og-rtca-autocatalysis'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Curated_mechanistic_rTCA_vent_fed = 100
INTERVAL_Curated_mechanistic_rTCA_vent_fed = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_rtca_autocatalysis_og_rtca_autocatalysis if 'ov_viva_genesis_composites_og_rtca_autocatalysis_og_rtca_autocatalysis' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_rtca_autocatalysis_og_rtca_autocatalysis = materialize_composite('viva_genesis.composites.og_rtca_autocatalysis.og_rtca_autocatalysis', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_rtca_autocatalysis_og_rtca_autocatalysis['state']}, core=core)
            _comp.run(STEPS_Curated_mechanistic_rTCA_vent_fed)
        print('ran', 'Curated mechanistic rTCA (vent-fed)', f'(viva_genesis.composites.og_rtca_autocatalysis.og_rtca_autocatalysis) for', STEPS_Curated_mechanistic_rTCA_vent_fed, 'steps')
    except Exception as _e:
        print(f"Curated mechanistic rTCA (vent-fed) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**rtca-discriminator**


In [ ]:
# rtca-discriminator
show_viz(_render_one('local:OgRtcaAutocatalysisViz', {'artifact_json': 'workspace/studies/og-rtca-autocatalysis/artifacts/rtca_discriminator.json', 'title': 'Reductive-TCA autocatalysis — refuted at vent, topology-capable beyond it', 'caption': 'Across the x1–x50 vent feed sweep the seeded reductive-TCA cycle shows zero self-amplification growth (it washes out under vent H2 — the honest negative); it ignites only in a labelled stronger-reductant + lower-dilution counterfactual, and topology-specifically (the matched null stays off). So the vent negative is a redox / residence-time limitation, not a limitation of the rTCA wiring.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| rtca-vent-h2-washout |  |  |
| rtca-topology-net-turn |  |  |
| rtca-permissive-limit-topology-specific |  |  |
| gate-a-a2-branched-resolved |  |  |


## Study: Spatially Integrated Protometabolism (`og-30-spatially-integrated-protometabolism`)

**Question.** Can the full chemical network operate productively inside a pore despite fractionation and washout — and does the pore structure actually improve persistence over a well-mixed system?

**Objective.** Run the 29 reaction network inside an explicit mineral-pore transport field (FiPy), with thermophoresis and surface adsorption, using discrete stochastic chemistry where molecule counts make a continuum unreliable; quantify co-localization, spatial reaction flux, residence and persistence against matched well-mixed and nonconcentrating controls.

**Hypothesis.** Reactive multispecies transport with thermophoretic accumulation and mineral-surface adsorption will co-localize intermediates and raise local productive flux and residence time enough to beat both a matched well-mixed control and a matched nonconcentrating control; if co-localization does not produce a flux margin over those controls, the pore confers no benefit and that is reported.

**Claim.** At the same vent-posterior base feed held equal (CO2 and H2 at 1e-3), a concentrating mineral pore ignites the reaction network - net carbon flux +4.5e-4, a connected network of 17 persistent species - while a matched well-mixed, nonconcentrating control does not (flux ~0, network not connected). The only difference is spatial concentration, so the pore itself is what makes the network productive. A porosity sweep shows the benefit is a bounded, somewhat brittle window rather than a broad swath: net carbon flux rises monotonically with confinement (no flux washout in C4), but the connectivity-defined ignition occupies a band around porosity ~0.80-0.90 (pore-local ~0.01-0.2 M) - below it enrichment is sub-threshold, above it pore-local feeds become physically implausible and the connectivity metric itself degrades. Spatial organization rescues ignition at feeds where a well-mixed system stays dead.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- thermophoretic / gradient-driven accumulation of intermediates in the pore
- mineral-surface adsorption concentrating and positioning reactants
- washout vs residence as the competition that determines persistence

**Proposed models / methods**

- FiPy finite-volume multispecies reactive transport                      [in-workspace: viva_genesis.transport]
- multispecies-reactive-transport cross-solver comparison                 [to-build: adds an independent solver check]
- mineral-surface adsorption / evolving catalytic accessibility           [to-build: mineral-surface-dynamics]
- discrete/stochastic chemistry where expected counts make continuum unreliable (evidence standard #7) [in-workspace/partial: continuum-stochastic handoff is to-build]
- PHREEQC speciation coupling for pore chemistry                          [wrapper-needed: viva_phreeqc]

**Key assumptions**

- Two adverse controls are required — a well-mixed control and a nonconcentrating control — both matched on feed, network and total material.
- Where molecule counts are low, continuum concentrations are unreliable and discrete stochastic chemistry is used instead.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Spatially Integrated Protometabolism (concentrating pore)` | `viva_genesis.composites.og_30_spatially_integrated_protometabolism.og_30_spatially_integrated_protometabolism` | 0 | concentrating=True, base_CO2=0.001, base_H2=0.001, porosity=0.88, k_wash=0.05, max_carbons=4 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.og_30_spatially_integrated_protometabolism.og_30_spatially_integrated_protometabolism`**

Study 30: the reaction network operating IN A PORE. The real FiPy thermophoretic trap concentrates the precursor feed (enrichment_factor); the enriched pore-local feed drives open_chemical_network's real reaction universe. concentrating=False runs the matched nonconcentrating control (feed at base, everything else equal) so the decision test — productive flux in the pore exceeds the control — is a direct comparison. Reports co-localization, spatial reaction flux, residence, persistence.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_og_30_spatially_integrated_protometabolism_og_30_spatially_integrated_protometabolism = {
    'concentrating': True,   # Pore (True) vs matched nonconcentrating control (False)
    'base_CO2'     : 0.001,   # Bulk precursor CO2 before pore concentration (molar)
    'base_H2'      : 0.001,   # Bulk precursor H2 before pore concentration (molar)
    'temperature'  : 333.15,   # Pore temperature (K)
    'flow_velocity': 0.0001,   # Convective flow through the pore (m/s); 0 = no-trap control
    'porosity'     : 0.3,   # Pore openness (0-1); ~1 = open-pore (wash-through) control
    'delta_T'      : 30.0,   # Thermal gradient across the pore (K) driving the Soret trap
    'k_wash'       : 0.05,   # CSTR washout / dilution rate (1/s)
    'max_carbons'  : 4,   # Carbon cap of the reaction universe
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `temperature_gradient` | float | kelvin/meter |  |
| `flow_velocity` | float | meter/second |  |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `productive_flux` | float | mol/second |  |
| `co_localization` | float | dimensionless |  |
| `residence_time` | float | second |  |

**Conditions**

- `species_mass_balance` (invariant): `per-species mass balance holds across transport + reaction + adsorption`
- `controls_matched` (invariant): `well-mixed and nonconcentrating controls share feed, network and total material`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-30-spatially-integrated-protometabolism ===
STUDY = 'og-30-spatially-integrated-protometabolism'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Spatially_Integrated_Protometabolism_concentrating_pore = 100
INTERVAL_Spatially_Integrated_Protometabolism_concentrating_pore = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_30_spatially_integrated_protometabolism_og_30_spatially_integrated_protometabolism if 'ov_viva_genesis_composites_og_30_spatially_integrated_protometabolism_og_30_spatially_integrated_protometabolism' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_30_spatially_integrated_protometabolism_og_30_spatially_integrated_protometabolism = materialize_composite('viva_genesis.composites.og_30_spatially_integrated_protometabolism.og_30_spatially_integrated_protometabolism', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_30_spatially_integrated_protometabolism_og_30_spatially_integrated_protometabolism['state']}, core=core)
            _comp.run(STEPS_Spatially_Integrated_Protometabolism_concentrating_pore)
        print('ran', 'Spatially Integrated Protometabolism (concentrating pore)', f'(viva_genesis.composites.og_30_spatially_integrated_protometabolism.og_30_spatially_integrated_protometabolism) for', STEPS_Spatially_Integrated_Protometabolism_concentrating_pore, 'steps')
    except Exception as _e:
        print(f"Spatially Integrated Protometabolism (concentrating pore) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**spatial-protometabolism-pore-vs-control**


In [ ]:
# spatial-protometabolism-pore-vs-control
show_viz(_render_one('local:OgSpatialProtometabolismViz', {'composite': 'viva_genesis.composites.og_30_spatially_integrated_protometabolism.og_30_spatially_integrated_protometabolism', 'n_steps': 120, 'title': 'Does the pore help? Enrichment, residence, and flux vs control', 'caption': 'Thermophoretic enrichment factor and residence time in the structured pore, then net productive carbon flux in the pore against a matched well-mixed control at identical feeds. Plainly: the pore gathers and holds the ingredients — but the real test is whether that makes more product than a stirred pot, shown in the third panel.'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_og_30_spatially_integrated_protometabolism_og_30_spatially_integrated_protometabolism if 'ov_viva_genesis_composites_og_30_spatially_integrated_protometabolism_og_30_spatially_integrated_protometabolism' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| receives-29-handoff-bundle _(secondary)_ | kind=observable detail=completeness + typing of the 29→30 bundle checked against 29's emitted bundle | op threshold detail bundle received complete and typed |
| colocalization-residence-measured _(primary)_ | kind=observable detail=co-localization + residence time as time/space-resolved fields | op threshold detail measurable co-localization and residence increase reported |
| productive-flux-beats-wellmixed-and-nonconcentrating _(primary)_ | kind=observable detail=pore productive flux + persistence vs. matched well-mixed and nonconcentrating controls | op threshold detail pore flux exceeds BOTH matched controls (else reported as no benefit) |
| spatial-flux-conservation-and-cross-solver _(secondary)_ | kind=observable detail=flux-divergence residual vs. source−sink; cross-solver agreement | op threshold detail conservation within tolerance; cross-solver agreement to tolerance |


## Study: Protometabolism to Amphiphiles (`og-31-protometabolism-to-amphiphiles`)

**Question.** Can actual metabolic products — not supplied amphiphiles — yield membrane-forming amphiphile mixtures under vent-local conditions?

**Objective.** Transform the study-30 precursor pool via modeled chain-extension/headgroup/hydrolysis chemistry, track chain-length and headgroup distributions, concentrations and losses, and test against measured critical aggregation and phase-stability data — with zero exogenous amphiphile input.

**Hypothesis.** Carbon-chain extension, headgroup generation and competing hydrolysis acting on the real product pool from study 30 will produce a chain-length and headgroup distribution whose concentration reaches the measured critical aggregation concentration and a stable phase window; if losses (hydrolysis, dilution) keep concentrations below that window, no assembly is claimed.

**Claim.** Under a reduced-coupling membrane model - retention reduces effective washout, finite access reduces effective feed crossing, explicitly NOT the MD-resolved permeability PMF (that is study 32's full Gate C test) - an endogenous membrane pays for itself on persistence but not on throughput. On the Gate-B pore-concentrated network, which is the sustained A1 carbon-fixing metabolism and NOT autocatalytic (A2 is refuted under vent conditions), a membrane at the Jordan-2019 vent-calibrated setting (retention 0.905, access 0.997; grounded, not tuned) raises the standing metabolite pool to about 2.9x the no-membrane ablation, while net carbon-fixation flux is about 0.6x - lower, because throughput is feed-limited, so reduced washout grows the pool rather than the rate. The benefit is windowed: it holds for membrane access down to about 0.45 and inverts below that (a too-tight membrane starves the feed and the retained pool falls below the open system), with the vent-calibrated access inside the window. This is the "benefits from it" half of Gate C, complementing og-31's Martini-MD cohesion (the "makes a membrane" half); the structurally-resolved coupling is study 32.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- carbon-chain extension from real metabolic precursors
- headgroup generation and competing hydrolysis / degradation losses
- concentration vs the measured critical aggregation concentration (CAC) and phase window

**Proposed models / methods**

- RDKit chain-extension / headgroup generation + precursor enumeration   [to-build: viva_reaction_atlas]
- OpenMM/LAMMPS composition-dependent assembly + permeability closure     [to-build: viva_md_membrane]
- PubChem / ORD amphiphile species identities and properties
- measured CAC / phase-stability benchmarks for independent validation (evidence standard

**Key assumptions**

- No exogenous amphiphile feed — every amphiphile must trace back to the network's carbon products.
- The assembly window is a measured critical-aggregation / phase-stability band from experiment, not a threshold tuned to pass.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Membrane benefit (reduced coupling, concentrating pore)` | `viva_genesis.composites.og_31_membrane_benefit.og_31_membrane_benefit` | 0 | membrane=True, retention=0.905, access=0.997, porosity=0.88, base_CO2=0.001, base_H2=0.001, max_carbons=4 |
| `Protometabolism to Amphiphiles (Martini-MD cohesion)` | `viva_genesis.composites.og_31_protometabolism_to_amphiphiles.og_31_protometabolism_to_amphiphiles` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.og_31_membrane_benefit.og_31_membrane_benefit`**

Partial Gate C: does an endogenous membrane pay for itself? Puts the sustained (A1, NOT autocatalytic) carbon-fixing network in the Gate-B concentrating pore and adds a REDUCED membrane coupling - retention reduces effective washout (k_wash*(1-retention)), finite access reduces effective feed crossing (feed*access) - with Jordan-2019-calibrated vent retention/access (0.905/0.997, grounded not tuned). membrane=False is the matched ablation (retention 0, access 1). Benefit = net carbon flux / persistence(membrane) > (ablation). NOT the MD-resolved PMF coupling (that is study 32 / viva-md-membrane = full Gate C).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_og_31_membrane_benefit_og_31_membrane_benefit = {
    'membrane'   : True,   # Endogenous membrane (True) vs matched no-membrane ablation (False)
    'retention'  : 0.905,   # Fraction of metabolites the membrane retains [Jordan-2019 vent value]
    'access'     : 0.997,   # Membrane feed access/permeability in [0,1]; lower = tighter = starves feed
    'base_CO2'   : 0.001,   # Bulk precursor CO2 (molar)
    'base_H2'    : 0.001,   # Bulk precursor H2 (molar)
    'porosity'   : 0.88,   # Gate-B pore porosity (0.88 = demo pore)
    'k_wash'     : 0.05,   # Base CSTR washout (1/s) before retention
    'max_carbons': 4,   # Carbon cap of the reaction universe
}

**The model — `viva_genesis.composites.og_31_protometabolism_to_amphiphiles.og_31_protometabolism_to_amphiphiles`**

Study 31: do amphiphiles assemble into a membrane? Drives real Martini 3 CG MD (viva-openmm) on a generated POPE bilayer; minimize+NVT relaxes the clashy initial placement into a bound bilayer so the potential energy settles negative — the membrane-cohesion / assembly signal. Reports PE/KE (cohesion), static area-per-lipid and lipid count. Charged-intermediate permeability (PMF) is the documented follow-up.


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_og_31_protometabolism_to_amphiphiles_og_31_protometabolism_to_amphiphiles = {
    'nx'              : 4,   # Lipids per leaflet in x
    'ny'              : 4,   # Lipids per leaflet in y
    'temperature'     : 310.0,   # Thermostat (K)
    'steps_per_update': 500,   # MD steps per composite tick
    'platform'        : 'OpenCL',   # OpenMM platform (OpenCL=Metal GPU; CPU/Reference also valid)
    'minimize'        : True,   # Energy-minimize the clashy initial placement first
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `carbon_product_supply` | float | molar/second | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `amphiphile_concentration` | float | molar | >= 0.0 |
| `mean_chain_length` | float | dimensionless |  |
| `hydrolysis_loss` | float | molar/second |  |

**Conditions**

- `no_exogenous_amphiphile` (invariant): `all amphiphiles derive from the network's carbon products (no untracked amphiphile feed)`
- `carbon_mass_balance` (invariant): `chain extension + headgroup + hydrolysis conserve carbon (balanced against supply)`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-31-protometabolism-to-amphiphiles ===
STUDY = 'og-31-protometabolism-to-amphiphiles'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Membrane_benefit_reduced_coupling_concentrating_pore = 100
INTERVAL_Membrane_benefit_reduced_coupling_concentrating_pore = 1.0
STEPS_Protometabolism_to_Amphiphiles_Martini_MD_cohesion = 100
INTERVAL_Protometabolism_to_Amphiphiles_Martini_MD_cohesion = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_31_membrane_benefit_og_31_membrane_benefit if 'ov_viva_genesis_composites_og_31_membrane_benefit_og_31_membrane_benefit' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_31_membrane_benefit_og_31_membrane_benefit = materialize_composite('viva_genesis.composites.og_31_membrane_benefit.og_31_membrane_benefit', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_31_membrane_benefit_og_31_membrane_benefit['state']}, core=core)
            _comp.run(STEPS_Membrane_benefit_reduced_coupling_concentrating_pore)
        print('ran', 'Membrane benefit (reduced coupling, concentrating pore)', f'(viva_genesis.composites.og_31_membrane_benefit.og_31_membrane_benefit) for', STEPS_Membrane_benefit_reduced_coupling_concentrating_pore, 'steps')
    except Exception as _e:
        print(f"Membrane benefit (reduced coupling, concentrating pore) not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_31_protometabolism_to_amphiphiles_og_31_protometabolism_to_amphiphiles if 'ov_viva_genesis_composites_og_31_protometabolism_to_amphiphiles_og_31_protometabolism_to_amphiphiles' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_31_protometabolism_to_amphiphiles_og_31_protometabolism_to_amphiphiles = materialize_composite('viva_genesis.composites.og_31_protometabolism_to_amphiphiles.og_31_protometabolism_to_amphiphiles', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_31_protometabolism_to_amphiphiles_og_31_protometabolism_to_amphiphiles['state']}, core=core)
            _comp.run(STEPS_Protometabolism_to_Amphiphiles_Martini_MD_cohesion)
        print('ran', 'Protometabolism to Amphiphiles (Martini-MD cohesion)', f'(viva_genesis.composites.og_31_protometabolism_to_amphiphiles.og_31_protometabolism_to_amphiphiles) for', STEPS_Protometabolism_to_Amphiphiles_Martini_MD_cohesion, 'steps')
    except Exception as _e:
        print(f"Protometabolism to Amphiphiles (Martini-MD cohesion) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**membrane-cohesion**


In [ ]:
# membrane-cohesion
show_viz(_render_one('local:OgMembraneCohesionViz', {'composite': 'viva_genesis.composites.og_31_protometabolism_to_amphiphiles.og_31_protometabolism_to_amphiphiles', 'n_steps': 8, 'nx': 4, 'ny': 4, 'platform': 'CPU', 'title': 'Bilayer cohesion and packing — a real Martini bilayer', 'caption': 'Two panels from the Martini NVT run — potential energy settling negative as the coarse-grained bilayer assembles and holds, and the equilibrium area per lipid. Plainly: the membrane snaps into a stable sheet instead of dispersing, and the lipids pack at the spacing real membranes show. Rendered from the real OpenMM run where the membrane extra is installed; otherwise the figure shows a note that the engine is unavailable.'}, RUNS_DB, STUDY_YAML, overrides=(ov_viva_genesis_composites_og_31_protometabolism_to_amphiphiles_og_31_protometabolism_to_amphiphiles if 'ov_viva_genesis_composites_og_31_protometabolism_to_amphiphiles_og_31_protometabolism_to_amphiphiles' in dir() else None)))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| receives-30-handoff-bundle _(secondary)_ | kind=observable detail=completeness + typing of the 30→31 bundle checked against 30's emitted bundle | op threshold detail bundle received complete and typed |
| no-untracked-amphiphile-feed _(secondary)_ | kind=observable detail=amphiphile mass ledger — fraction traceable to the study-30 pool (target 1.0); exogenous feed (target 0) | op threshold detail no untracked amphiphile feed; losses tracked |
| reaches-measured-assembly-window _(secondary)_ | kind=observable detail=endogenous amphiphile concentration + chain-length/headgroup distribution vs. measured CAC + phase window | op threshold detail concentration reaches the measured assembly window (else a reported shortfall) |
| handoff-31-to-32-complete _(secondary)_ | kind=observable detail=presence + typing of every field in the 31→32 transfer bundle | op threshold detail bundle complete and typed (satisfies 31→32 handoff validator) |
| membrane-benefit-persistence _(primary)_ | kind=observable detail=total_organic_carbon(membrane) vs total_organic_carbon(ablation), equal feed | op threshold detail membrane TOC exceeds the ablation TOC (else no persistence benefit) |
| membrane-benefit-access-window _(primary)_ | kind=observable detail=membrane TOC vs ablation across an access sweep; locate the inversion boundary | op threshold detail a bounded access window where membrane TOC exceeds ablation, with a located failure boundary |
| membrane-benefit-throughput _(secondary)_ | kind=observable detail=net_carbon_flux(membrane) vs net_carbon_flux(ablation) | op threshold detail reported either way (the physically-expected null is not a gate failure) |


## Study: First Self-Maintaining Protocell (`og-32-first-self-maintaining-protocell`)

**Question.** Can a vent-fed metabolism and an endogenous membrane be coupled so the membrane encloses and is maintained by the metabolism - and does the enclosed system persist and function better than the same metabolism with no membrane, at equal feeds?

**Objective.** Join the study-30 reactive transport and study-29 kinetics to the study-31 amphiphile pool in one Vivarium composite, with membrane permeability informed by short molecular dynamics of the amphiphile composition and gating the metabolite exchange; track membrane integrity (production vs degradation), standing metabolite pool and productive flux; and test both self-maintenance criteria against a matched no-endogenous-membrane ablation with input fluxes held equal - reporting a net cost honestly if that is the result.

**Hypothesis.** Coupling the study-29/30 reaction network to the study-31 amphiphile/membrane closure yields a compartment that retains metabolites against leakage, is replenished by metabolism-supplied amphiphiles faster than it degrades, and sustains higher productive flux or persistence than a membrane-less control at equal feeds - with the membrane's permeability informed by short molecular dynamics of its composition, so retention follows more directly from the chemistry than an imposed parameter does. If net amphiphile production falls below degradation, or the compartment fails to beat the matched null, no self-maintenance is claimed.

**Claim.** A composite coupling the endogenous vent-fed metabolism to an endogenous membrane - with the membrane's permeability informed by short molecular dynamics of its amphiphile composition, a tighter coupling than og-31's retention/access proxy - self-maintains in the measurable sense: at input fluxes held equal to a no-endogenous-membrane ablation, the membrane is rebuilt by metabolism-supplied amphiphiles at least as fast as it degrades (net production >= degradation) AND the enclosed system exceeds the ablation on persistence (standing metabolite pool ~2.0x). Throughput is not improved (0.79x, feed-limited) - so the benefit is retention, not faster chemistry, and the verdict is qualified, not an unconditional pass. The metabolism is the sustained A1 carbon-fixing network (not autocatalytic; A2 refuted), so the result is that a persisting network can make and keep a membrane that pays its way on persistence, within a located window (integrity holds up to a degradation rate k_deg <= ~0.01/s, and the persistence benefit tracks the MD composition).


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- metabolism-supplied amphiphile production vs membrane degradation (net integrity)
- composition-informed permeability from short molecular dynamics (tightening og-31's retention/access proxy)
- permeability-gated metabolite retention feeding back on the enclosed metabolism
- benefit (persistence and/or productive flux) over a matched membrane-less control at equal feeds

**Proposed models / methods**

- Short Martini 3 CG-MD of two amphiphile chain-length compositions -> bilayer thickness -> effective permeability via a published solubility-diffusion structure->permeability relation [viva-openmm / martini_openmm, run on the mini; Option B = MD-informed structural proxy. The PMF-resolved closure (Option A) is a budgeted follow-up - see discovery_implications]
- study-30 reactive transport + study-29 kinetics (real libRoadRunner reaction universe) coupled [in-workspace]
- Vivarium coupling composite: metabolism <-> membrane via permeability-gated metabolite retention
- composition / k_deg boundary sweep to locate the self-maintenance window
- measured permeability / CAC / phase-stability benchmarks for independent validation (evidence standard #6)

**Key assumptions**

- Input fluxes are held equal to the no-endogenous-membrane ablation - the Gate C benefit must be a margin over the matched null, not more feed.
- Membrane permeability is informed by short molecular dynamics of the amphiphile composition, not set as a free parameter (Option B MD-informed structural proxy; PMF-resolved A is a budgeted follow-up).
- Self-maintenance is the measurable criterion (net production >= degradation AND beats the ablation), not strict autopoietic closure.
- The enclosed metabolism is the sustained A1 carbon-fixing network, not autocatalytic (A2 refuted under vent conditions).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `First Self-Maintaining Protocell (MD-coupled, long di-C16)` | `viva_genesis.composites.og_32_first_self_maintaining_protocell.og_32_first_self_maintaining_protocell` | 0 | membrane=True, composition=long_di-C16, porosity=0.88, base_CO2=0.001, base_H2=0.001, max_carbons=4 |
| `No-endogenous-membrane ablation (permeability open, equal feeds)` | `viva_genesis.composites.og_32_first_self_maintaining_protocell.og_32_first_self_maintaining_protocell` | 0 | membrane=False, porosity=0.88, base_CO2=0.001, base_H2=0.001, max_carbons=4 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.og_32_first_self_maintaining_protocell.og_32_first_self_maintaining_protocell`**

Full Gate C (Option B, MD-INFORMED not PMF-resolved): couples the sustained A1 carbon-fixing network (Gate-B pore) to an endogenous membrane whose permeability is INFORMED BY short Martini CG-MD of the amphiphile composition (bilayer thickness: DLPC/di-C12 2.325 nm vs DPPC/di-C16 2.870 nm). A solubility-diffusion relation (proposed_input) maps thickness->permeability->retention (thicker long-chain membrane retains more); retention lowers effective washout (k_wash*(1-retention)) at feeds held EQUAL to the ablation. membrane=False is the matched no-endogenous-membrane control (permeability open, retention 0). Metabolism = sustained A1 network, NOT autocatalytic (A2 refuted). Full PMF-resolved permeability is Option A (study og-32a, budgeted follow-on).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_og_32_first_self_maintaining_protocell_og_32_first_self_maintaining_protocell = {
    'membrane'   : True,   # Endogenous membrane (True) vs matched no-membrane ablation (False)
    'composition': 'long_di-C16',   # MD amphiphile composition (long_di-C16 = aggregation-competent pool)
    'base_CO2'   : 0.001,   # Bulk precursor CO2 (molar) — equal across arms
    'base_H2'    : 0.001,   # Bulk precursor H2 (molar) — equal across arms
    'porosity'   : 0.88,   # Gate-B pore porosity (0.88 = demo pore)
    'k_wash'     : 0.05,   # Base CSTR washout (1/s) before retention
    'max_carbons': 4,   # Carbon cap of the reaction universe
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `external_co2_supply` | float | molar/second | >= 0.0 |
| `external_h2_supply` | float | molar/second | >= 0.0 |
| `amphiphile_precursor_supply` | float | molar/second | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `membrane_integrity` | float | molar/second |  |
| `metabolite_retention` | float | dimensionless | >= 0.0 |
| `standing_metabolite_pool` | float | molar | >= 0.0 |
| `productive_flux` | float | molar/second |  |
| `md_informed_permeability` | float | dimensionless | >= 0.0 |

**Conditions**

- `equal_input_fluxes_vs_ablation` (invariant): `external feed fluxes are held equal to the no-endogenous-membrane ablation (the Gate C benefit is the margin over this matched null)`
- `permeability_informed_by_md` (invariant): `membrane permeability is informed by short molecular dynamics of the amphiphile composition (MD-informed structural proxy, not PMF-resolved), a tighter coupling than og-31's retention/access proxy - not a free parameter`
- `metabolism_is_A1_not_autocatalytic` (invariant): `the enclosed metabolism is the sustained A1 carbon-fixing network; no autocatalytic self-amplification is assumed (A2 refuted under vent conditions)`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-32-first-self-maintaining-protocell ===
STUDY = 'og-32-first-self-maintaining-protocell'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_First_Self_Maintaining_Protocell_MD_coupled_long_di_C16 = 100
INTERVAL_First_Self_Maintaining_Protocell_MD_coupled_long_di_C16 = 1.0
STEPS_No_endogenous_membrane_ablation_permeability_open_equal_feeds = 100
INTERVAL_No_endogenous_membrane_ablation_permeability_open_equal_feeds = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_32_first_self_maintaining_protocell_og_32_first_self_maintaining_protocell if 'ov_viva_genesis_composites_og_32_first_self_maintaining_protocell_og_32_first_self_maintaining_protocell' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_32_first_self_maintaining_protocell_og_32_first_self_maintaining_protocell = materialize_composite('viva_genesis.composites.og_32_first_self_maintaining_protocell.og_32_first_self_maintaining_protocell', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_32_first_self_maintaining_protocell_og_32_first_self_maintaining_protocell['state']}, core=core)
            _comp.run(STEPS_First_Self_Maintaining_Protocell_MD_coupled_long_di_C16)
        print('ran', 'First Self-Maintaining Protocell (MD-coupled, long di-C16)', f'(viva_genesis.composites.og_32_first_self_maintaining_protocell.og_32_first_self_maintaining_protocell) for', STEPS_First_Self_Maintaining_Protocell_MD_coupled_long_di_C16, 'steps')
    except Exception as _e:
        print(f"First Self-Maintaining Protocell (MD-coupled, long di-C16) not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_32_first_self_maintaining_protocell_og_32_first_self_maintaining_protocell if 'ov_viva_genesis_composites_og_32_first_self_maintaining_protocell_og_32_first_self_maintaining_protocell' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_32_first_self_maintaining_protocell_og_32_first_self_maintaining_protocell = materialize_composite('viva_genesis.composites.og_32_first_self_maintaining_protocell.og_32_first_self_maintaining_protocell', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_32_first_self_maintaining_protocell_og_32_first_self_maintaining_protocell['state']}, core=core)
            _comp.run(STEPS_No_endogenous_membrane_ablation_permeability_open_equal_feeds)
        print('ran', 'No-endogenous-membrane ablation (permeability open, equal feeds)', f'(viva_genesis.composites.og_32_first_self_maintaining_protocell.og_32_first_self_maintaining_protocell) for', STEPS_No_endogenous_membrane_ablation_permeability_open_equal_feeds, 'steps')
    except Exception as _e:
        print(f"No-endogenous-membrane ablation (permeability open, equal feeds) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| permeability-md-informed _(primary)_ | kind=observable detail=MD thickness (long 2.870 nm vs short 2.325 nm) -> permeability -> retention | op threshold detail permeability/retention derived from MD thickness moving with composition None |
| self-maintenance-membrane-integrity _(primary)_ | kind=observable detail=net membrane_integrity = eta_amph*productive_flux - k_deg*M_req | op threshold detail net production >= degradation |
| beats-ablation-flux-or-persistence _(primary)_ | kind=observable detail=productive_flux + standing pool membrane vs ablation=None equal feed=None | op threshold detail membrane beats the ablation on flux or persistence |
| matched-null-equal-feeds _(primary)_ | kind=observable detail=external feed ledger - membrane vs ablation feeds held equal | op threshold detail input fluxes equal across arms |
| self-maintenance-windowed _(secondary)_ | kind=observable detail=k_deg boundary sweep + composition (short/long) sweep | op threshold detail a real upper k_deg boundary is bracketed |


## Study: Chemically Coupled Growth and Division (`og-33-chemically-coupled-growth-and-division`)

**Question.** Can a self-maintaining protocell grow and divide as a consequence of its own chemistry - membrane area and content increasing from metabolism until a physical instability splits it - rather than by an imposed division rule, and are the daughters viable?

**Objective.** Couple study-32's self-maintaining protocell to membrane growth from the metabolic amphiphile flux; grow membrane area until a measured area-doubling budding geometry triggers division; partition contents to daughters and test whether each daughter resumes the study-32 self-maintenance criterion - all against a matched null (growth decoupled from metabolism, or an imposed-division rule) at input fluxes held equal, reporting an imposed-rule outcome honestly if that is the result.

**Hypothesis.** Metabolism-supplied amphiphiles grow the membrane; as area outpaces volume the compartment reaches a shape instability and divides at a measured critical-geometry threshold, partitioning contents; division rate scales with metabolic flux and the daughters re-enter self-maintenance. If division needs an imposed event, or the daughters cannot resume self-maintenance, no emergent growth/division is claimed.

**Claim.** The protocell grows from its own metabolic amphiphile production and divides at a physically-grounded, measured area-doubling geometry - yielding viable daughters that resume self-maintenance - rather than by an imposed rule. Division rate scales with metabolic flux (coupled) while a growth-decoupled null is flux-independent and an imposed-division null divides on a timer; the metabolism-coupled arm beats both matched nulls at equal feeds, and daughters are viable only because they resume the study-32 self-maintenance criterion (integrity plus benefit over the no-membrane ablation). The verdict is qualified: the criterion is met on the measured-geometry first cut (Option B); full coarse-grained membrane mechanics (Option A, og-33a) is the budgeted follow-up. The metabolism is the sustained A1 carbon-fixing network (not autocatalytic), so growth is funded by carbon fixation, not self-amplification.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- membrane growth from the metabolic amphiphile flux (area increasing with metabolic flux)
- division at a MEASURED critical-geometry threshold (area-doubling budding instability), no imposed event
- content partitioning to daughters, each tested for resumed self-maintenance (viability)
- division rate scaling with metabolic flux vs a growth-decoupled / imposed-rule control at equal feeds

**Proposed models / methods**

- Membrane growth from the metabolic flux + division via a MEASURED area-doubling budding geometry  [Option B / first cut; grounded in hanczyc/budin growth-division geometries]
- Coarse-grained membrane mechanics for division (viva-md-membrane)  [to-build; the richer follow-on, tracked as og-33a in discovery_implications - NOT over-committed here]
- study-32 self-maintaining protocell (real A1 reaction universe + MD-informed membrane) coupled to the GrowthDivisionProcess  [in-workspace]
- division-rate vs flux slope + daughter-viability across a feed sweep
- measured growth-division geometry / critical-radius benchmarks for independent validation (evidence standard

**Key assumptions**

- Input fluxes are held equal to the matched null (growth-decoupled or imposed-division) - emergent growth/division must be a margin over it, not more feed.
- Division triggers at a MEASURED area-doubling budding geometry (hanczyc/budin), not a value tuned to force division (Option B; CG membrane mechanics is a follow-up og-33a).
- Daughters are viable only if they resume the study-32 self-maintenance criterion (production >= degradation), not merely form.
- The enclosed metabolism is the sustained A1 carbon-fixing network, not autocatalytic (A2 refuted under vent conditions).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Chemically coupled growth and division (metabolism-coupled, measured geometry)` | `viva_genesis.composites.og_33_chemically_coupled_growth_and_division.og_33_chemically_coupled_growth_and_division` | 0 | coupled=True, porosity=0.88, base_CO2=0.001, base_H2=0.001, max_carbons=4 |
| `Growth-decoupled null (equal feeds)` | `viva_genesis.composites.og_33_chemically_coupled_growth_and_division.og_33_chemically_coupled_growth_and_division` | 0 | coupled=False, porosity=0.88, base_CO2=0.001, base_H2=0.001, max_carbons=4 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.og_33_chemically_coupled_growth_and_division.og_33_chemically_coupled_growth_and_division`**

Gate D first half: couples study 32's self-maintaining protocell to amphiphile-driven membrane growth and division at a MEASURED critical-geometry (reduced-volume) threshold (hanczyc/budin; ⚑2 Option B, NOT CG membrane mechanics - that is og-33a). Membrane area grows from the metabolism's productive flux; at the measured critical reduced volume the compartment divides (geometry, not an imposed rule). coupled=False is the growth-decoupled null (fixed amphiphile supply); imposed_division=True is the imposed-rule null. Daughters viable only if they resume study-32 self-maintenance. Metabolism = sustained A1 network, NOT autocatalytic (A2 refuted).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_og_33_chemically_coupled_growth_and_division_og_33_chemically_coupled_growth_and_division = {
    'coupled'         : True,   # Growth driven by metabolic flux (True) vs growth-decoupled null (False)
    'imposed_division': False,   # Divide on a fixed timer (imposed-rule null) vs measured geometry
    'composition'     : 'long_di-C16',   # MD amphiphile composition for the study-32 membrane
    'base_CO2'        : 0.001,   # Bulk precursor CO2 (molar) — equal across arms
    'base_H2'         : 0.001,   # Bulk precursor H2 (molar) — equal across arms
    'porosity'        : 0.88,   # Gate-B pore porosity
    'k_wash'          : 0.05,   # Base CSTR washout (1/s)
    'max_carbons'     : 4,   # Carbon cap of the reaction universe
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `external_co2_supply` | float | molar/second | >= 0.0 |
| `external_h2_supply` | float | molar/second | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `membrane_area` | float | nanometer**2 | >= 0.0 |
| `reduced_volume` | float | dimensionless | >= 0.0 |
| `division_rate` | float | 1/second | >= 0.0 |
| `daughter_viable` | float | dimensionless | >= 0.0 |

**Conditions**

- `equal_input_fluxes_vs_null` (invariant): `external feed fluxes are held equal to the matched null (growth-decoupled or imposed-division control); emergent growth/division is the margin over it`
- `no_imposed_division` (invariant): `division follows from the area-doubling budding instability at a measured critical-geometry threshold, with no imposed division event`
- `division_geometry_measured` (invariant): `the critical-geometry threshold (area-doubling budding) is taken from measured growth-division geometries (hanczyc/budin), not tuned to force division`
- `daughters_resume_self_maintenance` (invariant): `a daughter is viable only if it resumes the study-32 self-maintenance criterion (production >= degradation)`
- `metabolism_is_A1_not_autocatalytic` (invariant): `the enclosed metabolism is the sustained A1 carbon-fixing network; growth is funded by carbon fixation, not autocatalytic self-amplification (A2 refuted)`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-33-chemically-coupled-growth-and-division ===
STUDY = 'og-33-chemically-coupled-growth-and-division'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Chemically_coupled_growth_and_division_metabolism_coupled_measured_geometry = 100
INTERVAL_Chemically_coupled_growth_and_division_metabolism_coupled_measured_geometry = 1.0
STEPS_Growth_decoupled_null_equal_feeds = 100
INTERVAL_Growth_decoupled_null_equal_feeds = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_33_chemically_coupled_growth_and_division_og_33_chemically_coupled_growth_and_division if 'ov_viva_genesis_composites_og_33_chemically_coupled_growth_and_division_og_33_chemically_coupled_growth_and_division' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_33_chemically_coupled_growth_and_division_og_33_chemically_coupled_growth_and_division = materialize_composite('viva_genesis.composites.og_33_chemically_coupled_growth_and_division.og_33_chemically_coupled_growth_and_division', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_33_chemically_coupled_growth_and_division_og_33_chemically_coupled_growth_and_division['state']}, core=core)
            _comp.run(STEPS_Chemically_coupled_growth_and_division_metabolism_coupled_measured_geometry)
        print('ran', 'Chemically coupled growth and division (metabolism-coupled, measured geometry)', f'(viva_genesis.composites.og_33_chemically_coupled_growth_and_division.og_33_chemically_coupled_growth_and_division) for', STEPS_Chemically_coupled_growth_and_division_metabolism_coupled_measured_geometry, 'steps')
    except Exception as _e:
        print(f"Chemically coupled growth and division (metabolism-coupled, measured geometry) not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_33_chemically_coupled_growth_and_division_og_33_chemically_coupled_growth_and_division if 'ov_viva_genesis_composites_og_33_chemically_coupled_growth_and_division_og_33_chemically_coupled_growth_and_division' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_33_chemically_coupled_growth_and_division_og_33_chemically_coupled_growth_and_division = materialize_composite('viva_genesis.composites.og_33_chemically_coupled_growth_and_division.og_33_chemically_coupled_growth_and_division', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_33_chemically_coupled_growth_and_division_og_33_chemically_coupled_growth_and_division['state']}, core=core)
            _comp.run(STEPS_Growth_decoupled_null_equal_feeds)
        print('ran', 'Growth-decoupled null (equal feeds)', f'(viva_genesis.composites.og_33_chemically_coupled_growth_and_division.og_33_chemically_coupled_growth_and_division) for', STEPS_Growth_decoupled_null_equal_feeds, 'steps')
    except Exception as _e:
        print(f"Growth-decoupled null (equal feeds) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| receives-32-handoff-bundle _(secondary)_ | kind=observable detail=og_33 composite wires the og_32 protocell + GrowthDivisionProcess | op threshold detail composite builds + runs on the study-32 protocell |
| growth-from-metabolism _(primary)_ | kind=observable detail=division_rate vs net carbon flux coupled vs growth-decoupled null=None | op threshold detail coupled division rate rises with flux (slope > 0) while decoupled is flat |
| division-at-measured-geometry _(primary)_ | kind=observable detail=divisions trigger at the measured area-doubling threshold not a timer=None | op threshold detail division triggers at the measured geometry no imposed event None |
| daughters-viable _(primary)_ | kind=observable detail=daughter_viable across the coupled feed sweep | op threshold detail daughters resume self-maintenance (viable) |
| beats-null-growth-decoupled-or-imposed _(primary)_ | kind=observable detail=coupled division rate vs flux-independent decoupled/imposed nulls equal feeds=None | op threshold detail metabolism-coupled growth/division exceeds the matched null |
| handoff-33-to-34-complete _(secondary)_ | kind=observable detail=division events + daughter compartments available to study 34 | op threshold detail dividing-protocell handoff available |


## Study: Chemical Heredity and Selection (`og-34-chemical-heredity-and-selection`)

**Question.** Does a chemical difference inherited by a daughter protocell cause reproducible differences in survival or reproduction - is there heritable variation selection can act on?

**Objective.** Run a multi-compartment population of study-33 dividing protocells under selection on an inherited network-composition trait whose fitness is grounded in the study-32/33 retention -> division-rate coupling; measure the parent-offspring fitness correlation and whether a population under selection shifts composition - against a matched no-inheritance control (daughters reset to a common composition) at input fluxes held equal, reporting no-selectable-difference honestly if that is the result. Instantiates the og-06 heredity rescope.

**Hypothesis.** Compartments carrying different inherited network compositions show reproducible, heritable fitness differences under the vent feed, so a population shifts composition under selection rather than drifting. If inherited composition produces no reproducible fitness difference above a no-inheritance control, or a population does not shift measurably, no selectable heredity is claimed.

**Claim.** A daughter-inherited chemical difference produces a reproducible fitness difference that selection acts on, shown against a matched no-inheritance control (daughters reset to a common composition) at equal feeds. Heredity is carried by inherited network composition (Option B), and fitness is grounded in the real study-32/33 coupling - composition sets membrane retention (study 32), which sets the standing pool and division rate (study 33) - so selection acts on a trait with a genuine mechanistic fitness consequence. Across seeds and generations, an inherited composition difference yields a parent-offspring fitness correlation far above the no-inheritance control, and the inheriting population shifts composition toward higher retention while the control drifts. The vasas-2010 evolvability caveat was TESTED (a declared, not tuned, heritable variance) and the population shifts rather than drifts - so compositional inheritance carries enough selectable variance here. Verdict qualified: compositional inheritance (Option B); template copying (Option A, og-34a) is the higher-fidelity follow-up. The metabolism is the sustained A1 carbon-fixing network (not autocatalytic), so selection acts on the inherited composition of a carbon-fixing network, not a self-amplifying one.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- inheritance via NETWORK COMPOSITION partitioned at division (compositional / GARD-style, not template copying in v1)
- fitness grounded in the study-32/33 coupling (composition -> retention -> standing pool -> division rate)
- parent-offspring fitness correlation vs a no-inheritance control (daughters reset to a common composition)
- selection as differential reproduction shifting a population's composition vs drift
- the vasas-2010 evolvability caveat (compositional inheritance may lack the variance selection needs) tested, not assumed

**Proposed models / methods**

- Wright-Fisher selection on an inherited network-composition trait; fitness grounded in study-32/33 retention -> division-rate coupling  [Option B / first cut]
- Template-copying inheritance  [the richer follow-on, tracked as og-34a in discovery_implications - NOT built here]
- study-33 division + study-32 self-maintenance set the per-compartment fitness  [in-workspace]
- dependency-free PERMUTATION test for parent-offspring correlation + selection-response significance  [PyMC noted as the richer stat, not installed]
- measured compositional-inheritance / GARD evolvability benchmarks for independent validation (evidence standard

**Key assumptions**

- Input fluxes are held equal to the no-inheritance control (daughters reset to a common composition) - any heritable fitness difference must be a margin over it, not more feed.
- Heredity is carried by inherited network composition (Option B); template copying is a tracked follow-up (og-34a).
- Fitness is grounded in the real study-32/33 coupling (composition -> retention -> division rate), not a toy function.
- The vasas-2010 evolvability caveat is tested with a declared (not tuned) heritable variance - a drift-only outcome would be reported.
- The enclosed metabolism is the sustained A1 carbon-fixing network, not autocatalytic (A2 refuted under vent conditions).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Chemical heredity and selection (inherited network composition)` | `viva_genesis.composites.og_34_chemical_heredity_and_selection.og_34_chemical_heredity_and_selection` | 0 | inheritance=True, porosity=0.88, base_CO2=0.001, base_H2=0.001, max_carbons=4 |
| `No-inheritance control (daughters reset to common composition, equal feeds)` | `viva_genesis.composites.og_34_chemical_heredity_and_selection.og_34_chemical_heredity_and_selection` | 0 | inheritance=False, porosity=0.88, base_CO2=0.001, base_H2=0.001, max_carbons=4 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.og_34_chemical_heredity_and_selection.og_34_chemical_heredity_and_selection`**

Gate D second half: selection on an inherited NETWORK COMPOSITION trait in a protocell population (⚑3 Option B; template copying = follow-on og-34a). Fitness is grounded in the real study-32/33 coupling (composition -> membrane retention [study-32 MD endpoints 0.333/0.661] -> standing pool -> division rate). inheritance=False is the matched no-inheritance null (daughters reset to the common composition). Reports parent-offspring fitness correlation + population composition shift; the vasas-2010 evolvability caveat is TESTED (does the population shift above the null?), not assumed. Metabolism = sustained A1 network, NOT autocatalytic (A2 refuted).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_og_34_chemical_heredity_and_selection_og_34_chemical_heredity_and_selection = {
    'inheritance': True,   # Inherit parent composition (True) vs no-inheritance null (False)
    'n_pop'      : 80,   # Population size
    'sigma'      : 0.03,   # Heritable mutation std (declared, not tuned)
    'seed'       : 7,   # RNG seed
    'base_CO2'   : 0.001,   # Vent feed (provenance for the fitness regime)
    'base_H2'    : 0.001,   # Vent feed (provenance for the fitness regime)
    'porosity'   : 0.88,   # Gate-B pore porosity (provenance)
    'max_carbons': 4,   # Carbon cap (provenance)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `external_co2_supply` | float | molar/second | >= 0.0 |
| `external_h2_supply` | float | molar/second | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `parent_offspring_fitness_correlation` | float | dimensionless |  |
| `population_composition_shift` | float | dimensionless |  |
| `heritable_fitness_variance` | float | dimensionless | >= 0.0 |
| `selection_response` | float | dimensionless |  |

**Conditions**

- `equal_input_fluxes_vs_no_inheritance` (invariant): `external feed fluxes are held equal to the no-inheritance control (daughters reset to a common composition); any heritable fitness difference is the margin over it`
- `inheritance_via_network_composition` (invariant): `heredity is carried by network composition partitioned at division (compositional / GARD-style), not template copying in this first cut`
- `fitness_grounded_in_real_coupling` (invariant): `fitness is the study-33 division rate set by study-32 membrane retention from the inherited composition - a real mechanistic consequence, not a toy fitness function`
- `evolvability_caveat_tested` (invariant): `the vasas-2010 evolvability caveat (compositional inheritance may lack selectable variance) is tested against the no-inheritance control with a declared (not tuned) heritable variance, not assumed`
- `metabolism_is_A1_not_autocatalytic` (invariant): `the enclosed metabolism is the sustained A1 carbon-fixing network; selection acts on inherited composition of a carbon-fixing network, not an autocatalytic one (A2 refuted)`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-34-chemical-heredity-and-selection ===
STUDY = 'og-34-chemical-heredity-and-selection'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Chemical_heredity_and_selection_inherited_network_composition = 100
INTERVAL_Chemical_heredity_and_selection_inherited_network_composition = 1.0
STEPS_No_inheritance_control_daughters_reset_to_common_composition_equal_feeds = 100
INTERVAL_No_inheritance_control_daughters_reset_to_common_composition_equal_feeds = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_34_chemical_heredity_and_selection_og_34_chemical_heredity_and_selection if 'ov_viva_genesis_composites_og_34_chemical_heredity_and_selection_og_34_chemical_heredity_and_selection' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_34_chemical_heredity_and_selection_og_34_chemical_heredity_and_selection = materialize_composite('viva_genesis.composites.og_34_chemical_heredity_and_selection.og_34_chemical_heredity_and_selection', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_34_chemical_heredity_and_selection_og_34_chemical_heredity_and_selection['state']}, core=core)
            _comp.run(STEPS_Chemical_heredity_and_selection_inherited_network_composition)
        print('ran', 'Chemical heredity and selection (inherited network composition)', f'(viva_genesis.composites.og_34_chemical_heredity_and_selection.og_34_chemical_heredity_and_selection) for', STEPS_Chemical_heredity_and_selection_inherited_network_composition, 'steps')
    except Exception as _e:
        print(f"Chemical heredity and selection (inherited network composition) not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_34_chemical_heredity_and_selection_og_34_chemical_heredity_and_selection if 'ov_viva_genesis_composites_og_34_chemical_heredity_and_selection_og_34_chemical_heredity_and_selection' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_34_chemical_heredity_and_selection_og_34_chemical_heredity_and_selection = materialize_composite('viva_genesis.composites.og_34_chemical_heredity_and_selection.og_34_chemical_heredity_and_selection', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_34_chemical_heredity_and_selection_og_34_chemical_heredity_and_selection['state']}, core=core)
            _comp.run(STEPS_No_inheritance_control_daughters_reset_to_common_composition_equal_feeds)
        print('ran', 'No-inheritance control (daughters reset to common composition, equal feeds)', f'(viva_genesis.composites.og_34_chemical_heredity_and_selection.og_34_chemical_heredity_and_selection) for', STEPS_No_inheritance_control_daughters_reset_to_common_composition_equal_feeds, 'steps')
    except Exception as _e:
        print(f"No-inheritance control (daughters reset to common composition, equal feeds) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| receives-33-handoff-bundle _(secondary)_ | kind=observable detail=population built on the study-33 dividing protocell with an inherited composition trait | op threshold detail population runs on the study-33 division substrate |
| inherited-difference-reproducible-fitness _(primary)_ | kind=observable detail=parent-offspring fitness correlation inheritance vs no-inheritance null=None permutation p=None | op threshold detail correlation above the null (perm p < 0.05) |
| population-shifts-under-selection _(primary)_ | kind=observable detail=population composition shift over generations inheritance vs null=None permutation p=None | op threshold detail the population shifts above the null (perm p < 0.05) |
| matched-null-no-inheritance _(primary)_ | kind=observable detail=null shows neither parent-offspring correlation nor directional shift | op threshold detail null correlation ~ 0 and null drift ~ 0 |
| handoff-34-to-35-36-37-complete _(secondary)_ | kind=observable detail=heritable population + fitness-composition map available | op threshold detail heritable-population handoff available |


## Study: Chemical Energy Coupling (`og-35-chemical-energy-coupling`)

**Question.** Does coupling a specified chemical energy carrier (thioester / acyl-phosphate cycle) to the protocell's metabolism improve net useful work and persistence over using the environmental gradient alone - with a full energy ledger and no hidden inflow?

**Objective.** Couple a specified thioester carrier cycle (tied to og-12) to the protocell energy supply under a declining geochemical drive; account every energy term (activation, use, hydrolysis, stored) in a closing ledger with no hidden inflow; and test net useful work and persistence - including through a gradient dip - against a matched gradient-only control at input fluxes held equal, reporting a net cost / wash honestly if that is the result.

**Hypothesis.** A carrier cycle that stores and transfers free energy lets the protocell do endergonic work when the gradient dips, raising persistence/robustness versus gradient-only operation - provided its synthesis and hydrolysis/leak costs are outweighed. If the carrier's costs exceed its benefit, or the energy ledger does not close without a hidden inflow, no net energy benefit is claimed.

**Claim.** Coupling a specified thioester carrier cycle to the protocell's energy supply pays for itself on PERSISTENCE through a gradient dip, not on total throughput. At input fluxes held equal to a matched gradient-only control and with a full free-energy ledger that closes (no hidden inflow), the carrier does useful work through the dip (where gradient-only, lacking a store, starves to near zero) - but its total useful work is not greater than gradient-only, because it time-shifts banked free energy into the dip and loses some to hydrolysis, and it gates activation at the bond rather than capturing the above-bond surplus (true even at zero hydrolysis). So the honest result is dip-resilience, not more work overall - verdict qualified - and the alternative carrier / microkinetic resolution (og-35a) is the follow-up. The metabolism is the sustained A1 carbon-fixing network (not autocatalytic), so the carrier funds endergonic work, not self-amplification.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- a specified thioester carrier cycle storing and transferring free energy
- endergonic work funded by the carrier when the environmental gradient dips (buffering)
- full energy ledger (activation / use / hydrolysis / stored) with no hidden inflow
- net useful work + persistence vs a gradient-only control at equal feeds (benefit minus carrier cost)

**Proposed models / methods**

- og-12 thioester carrier chemistry (ThioesterCarrierCycleProcess) + declining GeochemicalDrive, lumped kinetics  [Option B / first cut; one specified carrier]
- Alternative carrier (acyl-phosphate) and/or microkinetic enzyme-free resolution  [the richer follow-on, tracked as og-35a]
- matched gradient-only control (GradientOnlyWorkProcess): same activation throughput + drive + load, no store  [in-workspace]
- full energy-ledger accounting (activation / use / hydrolysis / stored), closure checked; k_hydrolyze cost/benefit window
- measured carrier thermodynamics (thioester hydrolysis dG') for independent validation (evidence standard

**Key assumptions**

- Drive + load are held equal to the gradient-only control - any energy benefit must be a margin over it, not more input.
- The energy ledger (activation / use / hydrolysis / stored) must close with no hidden inflow (budget_residual ~ 0).
- Carrier chemistry is one specified thioester cycle first (Option B); the alternative carrier / microkinetic resolution is a tracked follow-up (og-35a).
- The enclosed metabolism is the sustained A1 carbon-fixing network, not autocatalytic (A2 refuted under vent conditions).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Chemical energy coupling (thioester carrier)` | `viva_genesis.composites.og_35_chemical_energy_coupling.og_35_chemical_energy_coupling` | 0 | carrier=True, g_drive=50.0, decline_rate=0.4, load=30.0, bond_energy_kjmol=31.5 |
| `Gradient-only null (no carrier store, equal drive/load)` | `viva_genesis.composites.og_35_chemical_energy_coupling.og_35_chemical_energy_coupling` | 0 | carrier=False, g_drive=50.0, decline_rate=0.4, load=30.0, bond_energy_kjmol=31.5 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.og_35_chemical_energy_coupling.og_35_chemical_energy_coupling`**

Gate E energy coupling: does a specified thioester carrier cycle do endergonic work when the geochemical gradient DIPS, beating a matched gradient-only control, with a FULL closing energy ledger and no hidden inflow? carrier=True charges/discharges a stored carrier (buffered); carrier=False is the gradient-only null (direct drive->work, no store). Both see the same declining drive + load. Carrier chemistry = one specified thioester (Option B; acyl-phosphate / microkinetic = og-35a). Metabolism = sustained A1 network, NOT autocatalytic (A2 refuted).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_og_35_chemical_energy_coupling_og_35_chemical_energy_coupling = {
    'carrier'          : True,   # Thioester carrier cycle (True) vs matched gradient-only null (False)
    'g_drive'          : 50.0,   # Initial geochemical drive free energy (kJ/mol)
    'decline_rate'     : 0.4,   # Drive decline per step (kJ/mol) — the gradient dip
    'g_floor'          : 2.0,   # Drive floor (kJ/mol)
    'load'             : 30.0,   # Endergonic load to lift (kJ/mol)
    'bond_energy_kjmol': 31.5,   # Carrier high-energy bond (thioester ~31.5; proposed_input)
    'k_hydrolyze'      : 0.02,   # Carrier hydrolysis leak rate (1/s)
    'base_CO2'         : 0.001,   # Vent feed (provenance for the metabolic regime)
    'base_H2'          : 0.001,   # Vent feed (provenance)
    'porosity'         : 0.88,   # Gate-B pore porosity (provenance)
    'max_carbons'      : 4,   # Carbon cap (provenance)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `external_co2_supply` | float | molar/second | >= 0.0 |
| `external_h2_supply` | float | molar/second | >= 0.0 |
| `gradient_available` | float | kilojoule/mole | >= 0.0 |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `net_useful_work` | float | kilojoule/mole |  |
| `carrier_pool` | float | molar | >= 0.0 |
| `energy_ledger_residual` | float | kilojoule/mole |  |
| `dip_useful_work` | float | kilojoule/mole | >= 0.0 |

**Conditions**

- `equal_input_fluxes_vs_gradient_only` (invariant): `drive + load are held equal to the gradient-only (no-store) control; any benefit is the margin over it`
- `energy_ledger_closes` (invariant): `the full energy ledger (activation + use + hydrolysis + stored) closes to machine precision; energy_ledger_residual ~ 0`
- `no_hidden_energy_inflow` (invariant): `there is no energy inflow outside the accounted drive and carrier terms (budget_residual ~ 0 witnesses this)`
- `metabolism_is_A1_not_autocatalytic` (invariant): `the enclosed metabolism is the sustained A1 carbon-fixing network; the carrier funds endergonic work, not autocatalytic self-amplification (A2 refuted)`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-35-chemical-energy-coupling ===
STUDY = 'og-35-chemical-energy-coupling'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Chemical_energy_coupling_thioester_carrier = 100
INTERVAL_Chemical_energy_coupling_thioester_carrier = 1.0
STEPS_Gradient_only_null_no_carrier_store_equal_drive_load = 100
INTERVAL_Gradient_only_null_no_carrier_store_equal_drive_load = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_35_chemical_energy_coupling_og_35_chemical_energy_coupling if 'ov_viva_genesis_composites_og_35_chemical_energy_coupling_og_35_chemical_energy_coupling' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_35_chemical_energy_coupling_og_35_chemical_energy_coupling = materialize_composite('viva_genesis.composites.og_35_chemical_energy_coupling.og_35_chemical_energy_coupling', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_35_chemical_energy_coupling_og_35_chemical_energy_coupling['state']}, core=core)
            _comp.run(STEPS_Chemical_energy_coupling_thioester_carrier)
        print('ran', 'Chemical energy coupling (thioester carrier)', f'(viva_genesis.composites.og_35_chemical_energy_coupling.og_35_chemical_energy_coupling) for', STEPS_Chemical_energy_coupling_thioester_carrier, 'steps')
    except Exception as _e:
        print(f"Chemical energy coupling (thioester carrier) not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_35_chemical_energy_coupling_og_35_chemical_energy_coupling if 'ov_viva_genesis_composites_og_35_chemical_energy_coupling_og_35_chemical_energy_coupling' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_35_chemical_energy_coupling_og_35_chemical_energy_coupling = materialize_composite('viva_genesis.composites.og_35_chemical_energy_coupling.og_35_chemical_energy_coupling', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_35_chemical_energy_coupling_og_35_chemical_energy_coupling['state']}, core=core)
            _comp.run(STEPS_Gradient_only_null_no_carrier_store_equal_drive_load)
        print('ran', 'Gradient-only null (no carrier store, equal drive/load)', f'(viva_genesis.composites.og_35_chemical_energy_coupling.og_35_chemical_energy_coupling) for', STEPS_Gradient_only_null_no_carrier_store_equal_drive_load, 'steps')
    except Exception as _e:
        print(f"Gradient-only null (no carrier store, equal drive/load) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| receives-34-handoff-bundle _(secondary)_ | kind=observable detail=composite runs on the merged thioester carrier + declining drive | op threshold detail composite builds + runs |
| carrier-pays-on-dip-persistence _(primary)_ | kind=observable detail=useful work during the dip carrier vs gradient-only=None equal drive/load=None | op threshold detail carrier dip-work > gradient-only dip-work |
| carrier-throughput-not-improved _(primary)_ | kind=observable detail=total useful work carrier vs gradient-only=None | op threshold detail total useful work not improved (carrier <= gradient-only) - reported not tuned None |
| energy-ledger-closes _(primary)_ | kind=observable detail=max |budget_residual| in both arms | op threshold detail ledger closes (residual ~ 0) no hidden inflow None |
| matched-null-equal-feeds _(primary)_ | kind=observable detail=drive + load held equal across arms | op threshold detail inputs equal across arms |
| handoff-35-to-37-complete _(secondary)_ | kind=observable detail=energy-coupling readouts available to study 37 | op threshold detail handoff available |


## Study: Stochastic Protocell Ecology (`og-36-stochastic-protocell-ecology`)

**Question.** In a heterogeneous protocell population under honest stochastic low-count dynamics, do cross-feeding and community-level function develop that single lineages do not have at equal feed - and do stochastic dynamics materially change the outcome versus the deterministic mean-field?

**Objective.** Run a cross-feeding protocell community (shared limiting feed held equal across arms) as an exact Gillespie SSA and a matched mean-field ODE; measure cumulative productivity against the BEST single-lineage control at equal feed, test whether the margin is causally cross-feeding with a knockout, and compare stochastic vs deterministic outcomes across a system-size sweep - reporting a null / productivity-not-robustness result honestly if that is what the matched comparison shows.

**Hypothesis.** Heterogeneous protocell populations over a shared feed develop metabolic division of labour / cross-feeding and greater robustness than any single lineage, with stochastic dynamics materially changing outcomes at low counts. If heterogeneity confers no community benefit, or the benefit is not causally cross-feeding, that is reported honestly rather than smoothed.

**Claim.** A stochastic cross-feeding protocell community out-produces the best single lineage at equal feed - a real, cross-feeding-caused community function - but that benefit is productivity, not robustness. At the shared feed budget held equal to a matched best-single-lineage control, a ring of 4 cross-feeding lineages does ~1.9x the best single lineage's cumulative productivity (margin positive in every seed), and knocking out cross-feeding (same feed) collapses the margin to ~2% - so the benefit is causally cross-feeding, not mere diversity, with coexistence stabilised by depletable byproduct pools (negative frequency dependence), not assumed. BUT under low-count demographic noise the community is ~2x more variable than the rock-stable monoculture and loses a lineage in ~12% of runs; a system-size sweep shows this divergence from the deterministic mean-field GROWING as counts fall (to ~31% extinction / 20% margin erosion at half scale). So the honest result is the counterpoint to the required spine: cross-feeding genuinely RAISES productivity (the campaign's first throughput win) but PAYS in robustness - verdict qualified - and the GillesPy2/COBRApy genome-scale resolution (og-36a) is the follow-up. The metabolism is the sustained A1 cross-feeding community, not autocatalytic (A2 refuted).


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- {'cross-feeding ring': "each lineage's byproduct is substrate for its downstream neighbour"}
- coexistence stabilised by depletable byproduct pools (negative frequency dependence), not assumed
- community-level productivity margin over the best single lineage at equal shared feed (division of labour + recycling)
- honest low-count demographic stochasticity (extinction breaks the ring) vs the deterministic mean-field

**Proposed models / methods**

- dependency-free numpy Gillespie SSA (exact) + matched mean-field Euler ODE; explicit integer counts (shared feed, K byproduct pools, K cell counts)  [Option B / first cut]
- GillesPy2 / SSA tau-leaping + COBRApy genome-scale metabolic benchmark  [the richer follow-on, tracked as og-36a]
- matched best-single-lineage control: each lineage as a monoculture on the full shared feed, the best taken  [in-workspace]
- cross-feeding knockout (mu_cross gated to 0) at equal feed  [in-workspace causal ablation]
- system-size (Omega) sweep: scale the extensive counts to show the stochastic<->deterministic divergence grow as counts fall

**Key assumptions**

- The shared feed budget feed_F is held equal across arms - any community benefit must be a margin over the best single lineage, not more input.
- Coexistence is produced by depletable byproduct pools (negative frequency dependence), not assumed; the knockout excludes down.
- The SSA is exact on integer counts; a lineage at zero is extinct (no cell influx) and breaks the ring - the mean-field never truly reaches zero.
- Model is a dependency-free numpy SSA first (Option B); the GillesPy2 / COBRApy genome-scale resolution is a tracked follow-up (og-36a).
- The cross-feeding community is the sustained A1 network, not autocatalytic (A2 refuted under vent conditions).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Cross-feeding community (stochastic, SSA)` | `viva_genesis.composites.og_36_stochastic_protocell_ecology.og_36_stochastic_protocell_ecology` | 0 | n_lineages=4, n_seeded=4, deterministic=False, feed_F=7.0, mu_cross=0.5 |
| `Best-single-lineage null (monoculture, full shared feed)` | `viva_genesis.composites.og_36_stochastic_protocell_ecology.og_36_stochastic_protocell_ecology` | 0 | n_lineages=4, n_seeded=1, deterministic=False, feed_F=7.0 |
| `Cross-feeding knockout (community, cross-feeding off, equal feed)` | `viva_genesis.composites.og_36_stochastic_protocell_ecology.og_36_stochastic_protocell_ecology` | 0 | n_lineages=4, n_seeded=4, knockout=True, deterministic=False, feed_F=7.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.og_36_stochastic_protocell_ecology.og_36_stochastic_protocell_ecology`**

Gate E ecology: does a cross-feeding protocell community out-produce the best single lineage at EQUAL feed, is the margin causally due to cross-feeding (knockout), and do low-count stochastic dynamics materially change it versus the deterministic mean-field? One StochasticProtocellEcologyProcess drives all arms: community (all seeded, cross-feeding on), monoculture (best single lineage, full feed, no partner), knockout (all seeded, cross-feeding off). deterministic=False is the exact Gillespie SSA (extinction breaks the ring); deterministic=True the matched mean-field. Option B (dependency-free SSA); GillesPy2/COBRApy = og-36a. Metabolism = sustained A1 community, NOT autocatalytic (A2 refuted).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_og_36_stochastic_protocell_ecology_og_36_stochastic_protocell_ecology = {
    'n_lineages'   : 4,   # Number of protocell lineages on the ring
    'n_seeded'     : 4,   # Lineages alive at t0 (set 1 for a monoculture arm — the best single lineage)
    'knockout'     : False,   # Cross-feeding OFF (b_cross gated to 0) — the causal ablation
    'deterministic': False,   # Mean-field Euler ODE (True) vs exact Gillespie SSA (False)
    'feed_F'       : 16.0,   # Shared limiting feed influx, held EQUAL across arms (NOT split by lineage count)
    'feed_Ks'      : 6.0,   # Monod half-saturation on the shared feed S
    'cross_Kb'     : 6.0,   # Monod half-saturation on the byproduct B
    'mu'           : 0.55,   # Birth coefficient on the shared feed
    'mu_cross'     : 0.5,   # Cross-feeding (byproduct recycling) birth coefficient
    'k_feed_wash'  : 0.25,   # Shared-feed washout rate
    'k_byprod_wash': 0.3,   # Byproduct washout rate (what a monoculture loses)
    'd0'           : 0.11,   # Cell death/washout rate
    'eff_spread'   : 0.3,   # Heterogeneity of per-lineage feed efficiency (0 = identical)
    'n0'           : 5,   # Initial cells per seeded lineage
    'S0'           : 20.0,   # Initial shared feed
    'mono_lineage' : -1,   # Monoculture arm: which lineage to seed (-1 = most efficient); lets the sims take the true best single lineage
    'seed'         : 1,   # RNG seed (reproducible SSA trajectory / efficiency assignment)
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `shared_feed_flux` | float | dimensionless | >= 0.0 |
| `n_lineages` | integer | dimensionless | >= 1 |
| `cross_feeding_on` | boolean | dimensionless |  |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `cumulative_productivity` | float | dimensionless | >= 0.0 |
| `productivity_margin_over_best_single` | float | dimensionless |  |
| `n_alive` | float | dimensionless | >= 0.0 |
| `extinction_fraction` | float | dimensionless | [0.0, 1.0] |

**Conditions**

- `equal_shared_feed_across_arms` (invariant): `the shared feed flux feed_F is held equal across community / best-single / knockout arms; any benefit is the margin over the control`
- `coexistence_stabilised_not_assumed` (invariant): `deterministic coexistence of all K lineages is produced by depletable byproduct pools (negative frequency dependence); the knockout excludes down`
- `honest_low_count_stochasticity` (invariant): `the SSA is exact on integer counts; a lineage reaching zero is extinct (no cell influx) and breaks the ring - the mean-field never truly reaches zero`
- `metabolism_is_A1_not_autocatalytic` (invariant): `the cross-feeding community is the sustained A1 network; it is division of labour, not autocatalytic self-amplification (A2 refuted)`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-36-stochastic-protocell-ecology ===
STUDY = 'og-36-stochastic-protocell-ecology'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Cross_feeding_community_stochastic_SSA = 100
INTERVAL_Cross_feeding_community_stochastic_SSA = 1.0
STEPS_Best_single_lineage_null_monoculture_full_shared_feed = 100
INTERVAL_Best_single_lineage_null_monoculture_full_shared_feed = 1.0
STEPS_Cross_feeding_knockout_community_cross_feeding_off_equal_feed = 100
INTERVAL_Cross_feeding_knockout_community_cross_feeding_off_equal_feed = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_36_stochastic_protocell_ecology_og_36_stochastic_protocell_ecology if 'ov_viva_genesis_composites_og_36_stochastic_protocell_ecology_og_36_stochastic_protocell_ecology' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_36_stochastic_protocell_ecology_og_36_stochastic_protocell_ecology = materialize_composite('viva_genesis.composites.og_36_stochastic_protocell_ecology.og_36_stochastic_protocell_ecology', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_36_stochastic_protocell_ecology_og_36_stochastic_protocell_ecology['state']}, core=core)
            _comp.run(STEPS_Cross_feeding_community_stochastic_SSA)
        print('ran', 'Cross-feeding community (stochastic, SSA)', f'(viva_genesis.composites.og_36_stochastic_protocell_ecology.og_36_stochastic_protocell_ecology) for', STEPS_Cross_feeding_community_stochastic_SSA, 'steps')
    except Exception as _e:
        print(f"Cross-feeding community (stochastic, SSA) not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_36_stochastic_protocell_ecology_og_36_stochastic_protocell_ecology if 'ov_viva_genesis_composites_og_36_stochastic_protocell_ecology_og_36_stochastic_protocell_ecology' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_36_stochastic_protocell_ecology_og_36_stochastic_protocell_ecology = materialize_composite('viva_genesis.composites.og_36_stochastic_protocell_ecology.og_36_stochastic_protocell_ecology', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_36_stochastic_protocell_ecology_og_36_stochastic_protocell_ecology['state']}, core=core)
            _comp.run(STEPS_Best_single_lineage_null_monoculture_full_shared_feed)
        print('ran', 'Best-single-lineage null (monoculture, full shared feed)', f'(viva_genesis.composites.og_36_stochastic_protocell_ecology.og_36_stochastic_protocell_ecology) for', STEPS_Best_single_lineage_null_monoculture_full_shared_feed, 'steps')
    except Exception as _e:
        print(f"Best-single-lineage null (monoculture, full shared feed) not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_36_stochastic_protocell_ecology_og_36_stochastic_protocell_ecology if 'ov_viva_genesis_composites_og_36_stochastic_protocell_ecology_og_36_stochastic_protocell_ecology' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_36_stochastic_protocell_ecology_og_36_stochastic_protocell_ecology = materialize_composite('viva_genesis.composites.og_36_stochastic_protocell_ecology.og_36_stochastic_protocell_ecology', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_36_stochastic_protocell_ecology_og_36_stochastic_protocell_ecology['state']}, core=core)
            _comp.run(STEPS_Cross_feeding_knockout_community_cross_feeding_off_equal_feed)
        print('ran', 'Cross-feeding knockout (community, cross-feeding off, equal feed)', f'(viva_genesis.composites.og_36_stochastic_protocell_ecology.og_36_stochastic_protocell_ecology) for', STEPS_Cross_feeding_knockout_community_cross_feeding_off_equal_feed, 'steps')
    except Exception as _e:
        print(f"Cross-feeding knockout (community, cross-feeding off, equal feed) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| receives-handoff-bundle _(secondary)_ | kind=observable detail=composite runs the SSA cross-feeding community + controls | op threshold detail composite builds + runs |
| community-out-produces-best-single _(primary)_ | kind=observable detail=community vs best-single-lineage cumulative productivity at equal feed | op threshold detail community productivity > best single lineage; margin positive in most seeds |
| margin-causally-cross-feeding _(primary)_ | kind=observable detail=knockout margin as a fraction of the community margin | op threshold detail knockout recovers < 15% of the community margin |
| coexistence-stabilized-not-assumed _(primary)_ | kind=observable detail=deterministic n_alive community vs knockout=None | op threshold detail community n_alive == K; knockout n_alive < K |
| stochastic-materially-changes-low-count _(primary)_ | kind=observable detail=SSA extinction fraction + stochastic/deterministic margin gap vs Omega | op threshold detail nominal extinction fraction > 0; divergence grows as counts fall |
| community-not-more-robust-than-monoculture _(primary)_ | kind=observable detail=community CV vs best-single CV; community_more_robust flag | op threshold detail community CV > best-single CV (community_more_robust False) - reported honestly |
| matched-null-equal-feeds _(primary)_ | kind=observable detail=feed_F held equal across arms | op threshold detail feed budget equal across arms |
| handoff-36-to-37-complete _(secondary)_ | kind=observable detail=ecology readouts available to study 37 | op threshold detail handoff available |


## Study: Progressive Independence (`og-37-progressive-independence`)

**Question.** As each environmental support is withdrawn one at a time, how independent can a protocell lineage become — which supports can it internalize or replace via its innovations, and where does it still fail — against a matched per-support ablation with input fluxes held equal?

**Objective.** Reuse og-08's validated co-limited off-gradient budget and withdraw each support channel one at a time (others held full); for each, determine whether the lineage persists via its innovation set (internalizable) or fails, attribute the rescue to a specific innovation with a matched ablation, show the all-at-once run as og-08's attribution confound, report per-channel budget margins + a pump-affordability sensitivity, and state plainly the lumped-feed and coarse-abstraction limits — reporting the honest profile, not an escape it does not demonstrate.

**Hypothesis.** The coupled protocell (studies 32/34 + 35 energy + 36 ecology) can internalize a subset of supports (energy via an internal pump, resources via import) but remains dependent on others, giving a measurable, attributed independence profile rather than an all-or-nothing escape. If no support is internalizable, or the apparent independence is not causally attributable to a specific innovation, that is reported honestly.

**Claim.** A protocell lineage achieves GRADED, attributed independence — a per-support map of which environmental support each innovation internalizes — not an all-or-nothing escape, and not a free one. Reusing og-08's validated co-limited budget and withdrawing each support ONE AT A TIME (where og-08 relaxed them jointly), each of the three channels is internalizable with the full innovation set and is causally attributed to a specific innovation by a matched ablation (gradient<-pump, mineral<-importer, feed<-importer; switching that innovation off collapses the lineage) — an attribution og-08's all-at-once relaxation cannot make. Heredity is the irreducible substrate (every channel collapses without it), and the gradient (pH/pmf drive) is budget-sensitive: its internalization requires a reasonably efficient internal pump (catalyst_efficiency >= ~0.7) and flips to a hard dependency below that, grounded in study-35's persistence-not-throughput limited surplus. Because the feed is LUMPED (H2+CO2+organics) and the innovations are coarse budget abstractions, this is an affordability/attribution profile, not a demonstrated escape — verdict qualified — and the resolved per-feedstock + temperature-Arrhenius + combinatorial model (og-37a) is the follow-up.


### Design — originally proposed

_The mechanisms and methods proposed when this study was designed; the model **actually built** (and the tools it uses) is shown below. These may differ as the design was realized._

**Proposed mechanisms**

- per-support one-at-a-time withdrawal of geological support (vs og-08's joint all-at-once relaxation)
- internalization via the og-08 innovations — internal pump (energy; the study-35 carrier), active importer (resource), heredity (holds machinery)
- matched per-support ablation (rescuing innovation off -> collapse) for causal attribution
- a graded, attributed independence profile with named irreducible dependencies (heredity substrate; budget-sensitive gradient)

**Proposed models / methods**

- REUSE og-08 autonomy budget (OffGradientPersistenceProcess + EnvironmentalRelaxationProcess), co-limited Liebig-minimum, in a new og_37 composite with per-channel withdrawal  [Option B / first cut, no invented terms]
- per-feedstock resolution (split feed into H2 / CO2 / organics) + temperature as an Arrhenius rate modifier + combinatorial multi-withdrawal  [the richer follow-on, tracked as og-37a]
- matched per-support ablation (rescuing innovation off) + the all-at-once confound run  [in-workspace causal controls]
- pump-affordability sensitivity sweep (catalyst_efficiency) locating the gradient internalization boundary

**Key assumptions**

- Each support is withdrawn one-at-a-time with the others held full; the matched ablation sees the same withdrawal, so any independence is a margin over the ablation, not more input.
- The budget is og-08's validated co-limited (Liebig-minimum) model reused verbatim — no parallel ledger, no invented terms.
- A support is internalized by an innovation only if the lineage persists WITH it and collapses WITHOUT it (matched ablation).
- Feed is LUMPED (H2+CO2+organics); innovations are coarse budget abstractions — an affordability/attribution profile, not a demonstrated escape. Per-feedstock + temperature-Arrhenius = og-37a.
- The metabolism is the sustained A1 carbon-fixing network, not autocatalytic (A2 refuted under vent conditions).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `Progressive independence (all-at-once, full innovation set — the og-08 reference)` | `viva_genesis.composites.og_37_progressive_independence.og_37_progressive_independence` | 0 | withdraw=all, has_pump=1.0, has_importer=1.0, has_heredity=1.0, catalyst_efficiency=1.0 |
| `One-at-a-time withdrawal (per-channel; the study's novelty)` | `viva_genesis.composites.og_37_progressive_independence.og_37_progressive_independence` | 0 | withdraw=gradient, has_pump=1.0, has_importer=1.0, has_heredity=1.0, catalyst_efficiency=1.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**The model — `viva_genesis.composites.og_37_progressive_independence.og_37_progressive_independence`**

Gate E progressive independence: withdraw each environmental support ONE AT A TIME and ask which the lineage can internalize (persist via an innovation) vs which are hard dependencies. Reuses the validated og-08 off-gradient budget (OffGradientPersistenceProcess + EnvironmentalRelaxationProcess) verbatim — no invented terms — but relaxes a single channel at a time (others held full) instead of og-08's joint all-at-once relaxation, so a matched per-support ablation (rescuing innovation off -> fail) attributes each support to its internalizing innovation (gradient->pump, mineral/feed->importer, all gated by heredity). Channels: gradient (pH/pmf drive), mineral_access (mineral catalysis), feed (H2+CO2+organics LUMPED — per-feedstock split + temperature-Arrhenius = og-37a). Coarse budget abstraction = an affordability/attribution profile, NOT a demonstrated escape; heredity = irreducible substrate. Terminal (consumes 34/35/36, extends nothing).


In [ ]:
# === Parameters — edit any value, then re-run from the Run cell below ===
# Each entry is the composite's current default; the comment explains it.
ov_viva_genesis_composites_og_37_progressive_independence_og_37_progressive_independence = {
    'withdraw'           : 'all',   # Which support to withdraw one-at-a-time: gradient | mineral | feed | all (og-08 confound) | none (baseline)
    'has_pump'           : 1.0,   # Internal energy-conversion innovation (study-35 carrier); 1 present, 0 knockout
    'has_importer'       : 1.0,   # Active resource-import innovation; 1 present, 0 knockout
    'has_heredity'       : 1.0,   # Information-maintenance innovation (holds machinery); 1 present, 0 knockout
    'catalyst_efficiency': 1.0,   # Internal-pump catalytic efficiency (dimensionless) — the pump-affordability knob
    'decline_rate'       : 0.02,   # Support relaxation per step (1/s); the withdrawn channel reaches 0 partway through the run
}

### Typed contract (declared interface)

_The bigraph-schema typed ports + conservation predicates this study's composite must satisfy (audited in the workbench §Contract panel). Declared design — independent of whether a run exists yet._

**Inputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `withdrawn_support` | string | dimensionless |  |
| `innovation_set` | string | dimensionless |  |

**Outputs**

| port | type | units | bounds |
| --- | --- | --- | --- |
| `internalizable` | boolean | dimensionless |  |
| `budget_margin` | float | dimensionless |  |
| `survivors` | float | dimensionless | [0.0, 1.0] |
| `affordability_boundary` | float | dimensionless |  |

**Conditions**

- `equal_input_fluxes_per_support` (invariant): `each support is withdrawn one-at-a-time with the others held full; the matched ablation sees the same withdrawal — any independence is a margin over the ablation, not more input`
- `reuses_validated_og08_budget` (invariant): `the budget is og-08's validated co-limited (Liebig-minimum) model reused verbatim; no parallel ledger, no invented terms`
- `attribution_requires_causal_ablation` (invariant): `a support counts as internalized by an innovation only if it persists WITH the innovation and collapses WITHOUT it (matched ablation)`
- `metabolism_is_A1_not_autocatalytic` (invariant): `the lineage's metabolism is the sustained A1 carbon-fixing network; independence is budget persistence, not autocatalytic self-amplification (A2 refuted)`


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: og-37-progressive-independence ===
STUDY = 'og-37-progressive-independence'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_Progressive_independence_all_at_once_full_innovation_set_the_og_08_reference = 100
INTERVAL_Progressive_independence_all_at_once_full_innovation_set_the_og_08_reference = 1.0
STEPS_One_at_a_time_withdrawal_per_channel_the_study_s_novelty = 100
INTERVAL_One_at_a_time_withdrawal_per_channel_the_study_s_novelty = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_37_progressive_independence_og_37_progressive_independence if 'ov_viva_genesis_composites_og_37_progressive_independence_og_37_progressive_independence' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_37_progressive_independence_og_37_progressive_independence = materialize_composite('viva_genesis.composites.og_37_progressive_independence.og_37_progressive_independence', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_37_progressive_independence_og_37_progressive_independence['state']}, core=core)
            _comp.run(STEPS_Progressive_independence_all_at_once_full_innovation_set_the_og_08_reference)
        print('ran', 'Progressive independence (all-at-once, full innovation set — the og-08 reference)', f'(viva_genesis.composites.og_37_progressive_independence.og_37_progressive_independence) for', STEPS_Progressive_independence_all_at_once_full_innovation_set_the_og_08_reference, 'steps')
    except Exception as _e:
        print(f"Progressive independence (all-at-once, full innovation set — the og-08 reference) not runnable yet: {type(_e).__name__}: {_e}")
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_genesis_composites_og_37_progressive_independence_og_37_progressive_independence if 'ov_viva_genesis_composites_og_37_progressive_independence_og_37_progressive_independence' in dir() else None  # edits from the panel cell, if any
            spec_viva_genesis_composites_og_37_progressive_independence_og_37_progressive_independence = materialize_composite('viva_genesis.composites.og_37_progressive_independence.og_37_progressive_independence', _ov)
            _comp = Composite({'state': spec_viva_genesis_composites_og_37_progressive_independence_og_37_progressive_independence['state']}, core=core)
            _comp.run(STEPS_One_at_a_time_withdrawal_per_channel_the_study_s_novelty)
        print('ran', "One-at-a-time withdrawal (per-channel; the study's novelty)", f'(viva_genesis.composites.og_37_progressive_independence.og_37_progressive_independence) for', STEPS_One_at_a_time_withdrawal_per_channel_the_study_s_novelty, 'steps')
    except Exception as _e:
        print(f"One-at-a-time withdrawal (per-channel; the study's novelty) not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| receives-handoff-bundle _(secondary)_ | kind=observable detail=og_37 composite builds + runs the reused og-08 processes | op threshold detail composite builds + runs |
| channels-internalizable-with-full-set _(primary)_ | kind=observable detail=final survivors per channel at full withdrawal full innovation set=None | op threshold detail survivors >= 0.5 for each channel |
| attribution-causal-via-ablation _(primary)_ | kind=observable detail=survivors with the rescuing innovation off vs on per channel=None | op threshold detail full-set survivors >= 0.5 AND ablated survivors < 0.5 for each channel |
| heredity-irreducible-substrate _(primary)_ | kind=observable detail=survivors with heredity off per channel=None | op threshold detail survivors < 0.5 for every channel with heredity off |
| all-at-once-is-the-confound _(primary)_ | kind=observable detail=all-at-once survivors full set vs each innovation knockout=None | op threshold detail escapes with full set AND fails with each single innovation removed |
| gradient-budget-sensitive _(primary)_ | kind=observable detail=catalyst_efficiency sweep of the gradient-channel internalization | op threshold detail the gradient verdict flips within the swept efficiency range (a located boundary) |
| terminal-emits-nothing _(secondary)_ | kind=observable detail=handoff.emits is empty | op threshold detail emits empty |


## Study: Amphiphile assembly & composition (membrane-MD M1) (`mem-m1-amphiphile-assembly`)

**Question.** Which amphiphile structures and mixtures - fatty acids C8-C14, fatty alcohols, monoacylglycerols, and their mixtures - form micelles, bilayers or vesicles under vent-relevant conditions?

**Objective.** Self-assemble a defined set of prebiotically-plausible compositions in coarse-grained MD and classify the assembly state, with unbiased self-assembly rather than a preassembled start.

**Hypothesis.** Assembly state is set by chain length, headgroup, mixture composition and the carboxylate protonation fraction; a feasible bilayer/vesicle window exists for long-enough chains and favorable mixtures, bounded below by the critical aggregation concentration and above by insolubility.


### Parameters


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: mem-m1-amphiphile-assembly ===
STUDY = 'mem-m1-amphiphile-assembly'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No baseline composite declared for this study — nothing to build.")

### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| souza2021martini3 | | | |
| marrink2003vesicle | | | |
| janke2014oleic | | | |
| jordan2019promotion | | | |


## Study: Permeability & selective retention (membrane-MD M2) (`mem-m2-permeability-retention`)

**Question.** Which membranes admit substrates while retaining products - which compositions are selectively permeable in the direction the metabolism needs?

**Objective.** Compute per-solute-class permeabilities with uncertainty via PMF + diffusivity and build a composition->transport lookup the functional-window study (M5) consumes.

**Hypothesis.** Permeability is graded by solute class and set by composition; a window exists where small substrates (water, neutral organics) cross while larger / charged products (organic acids, charged metabolites) are retained - but stability and selective retention may trade off (the core feasibility tension).


### Parameters


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: mem-m2-permeability-retention ===
STUDY = 'mem-m2-permeability-retention'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No baseline composite declared for this study — nothing to build.")

### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| souza2021martini3 | | | |
| grunewald2020titratable | | | |
| marrink2007martini | | | |


## Study: Stability across environmental gradients (membrane-MD M3) (`mem-m3-stability-gradients`)

**Question.** How do ionic strength, mono- and divalent ions, temperature, and carboxylate protonation fraction change membrane integrity?

**Objective.** Map the stability envelope of candidate compositions across the vent-relevant gradients, one factor at a time then a small factorial.

**Hypothesis.** Vent-relevant ionic strength, divalent cations (Mg2+/Ca2+), temperature and pH-set protonation define a stability envelope; integrity degrades (defects, desorption, rupture) outside it, and that envelope may not coincide with the functional (selective-permeability) window.


### Parameters


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: mem-m3-stability-gradients ===
STUDY = 'mem-m3-stability-gradients'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No baseline composite declared for this study — nothing to build.")

### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| janke2014oleic | | | |
| grunewald2020titratable | | | |
| souza2021martini3 | | | |


## Study: Growth, shape change & division propensity (membrane-MD M4) (`mem-m4-growth-division-propensity`)

**Question.** Can supplied amphiphiles insert into, reshape, and expand a vesicle - and what is its propensity to divide?

**Objective.** Measure amphiphile insertion/exchange, area growth, curvature change and the budding/division barrier for candidate vesicles - a division PROPENSITY, kept separate from heritable division.

**Hypothesis.** Supplied amphiphiles insert and expand membrane area; area growth drives curvature/shape change and lowers the budding barrier - a propensity to divide, set by composition and osmotic conditions.


### Parameters


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: mem-m4-growth-division-propensity ===
STUDY = 'mem-m4-growth-division-propensity'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No baseline composite declared for this study — nothing to build.")

### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| hanczyc2003experimental | | | |
| budin2012concentration | | | |
| marrink2003vesicle | | | |


## Study: Functional membrane window (membrane-MD M5) (`mem-m5-functional-membrane-window`)

**Question.** Which measured compositions sustain a modeled internal reaction network - where do assembly, stability, selective permeability and growth-requirement constraints jointly hold?

**Objective.** Insert the M2-M4 measured properties into a coupled compartment-reaction model (reusing og-32's coupling and the A1 network), scan compositions, and rank them on joint feasibility.

**Hypothesis.** The feasible region is an intersection, likely narrow: the most stable composition is not the most functional, so the compositions that sustain the internal network are a bounded compromise - and the region may be empty (a meaningful mechanistic boundary).


### Parameters


### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: mem-m5-functional-membrane-window ===
STUDY = 'mem-m5-functional-membrane-window'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No baseline composite declared for this study — nothing to build.")

### Literature anchors

| expectation | model observable | source | status |
| --- | --- | --- | --- |
| souza2021martini3 | | | |
| sojo2014bioenergetic | | | |
| jordan2019promotion | | | |


## Open decisions
- Read og-08's {pump, importer, heredity} as a HYPOTHESIS GENERATOR, not a result: the integrated world (og-08-alt-robust-innovation-set) finds the required set structure-dependent and the off-gradient verdict decided by the ocean floor, the membrane leakiness and the inheritance class's budget sensitivity. Adopt this framing for the campaign's endpoint statement, and supply the ocean's H2 / organic background (the floor) as an expert prior?
- Agent motion model (ecology stage): DEFERRED. The proto_ecology component points agents at fixed cells of the og-03 enrichment field (interleaved or segregated along the pore); no motility, chemotaxis or advective drift of agents is modelled. Which motion model (none / advected with the pore flow / Brownian between cells / thermophoretic like the solutes) should the ecology follow-up adopt?
- Membrane permeability of dicarboxylates (ecology stage): the community function (the reductive C4 branch crossing from the carboxylating to the hydrogenating lineage) emerges only above a diacid permeability of ~3e-9 m/s at 100 s residence, while the measured passage of charged solutes through fatty-acid vesicles (Mansy 2008, proposed_inputs, pending) is ~1e-11-1e-10 m/s. Is a leakier boundary (mixed / shorter-chain amphiphiles, the thioester_aminothiol boundary class) or a longer pore residence the defensible route, or should the exchanged intermediate be a neutral species?
- Heredity attachment thresholds: what precursor budget (copies per individual the pool must cover, attach_min_copies), energy margin (attach_energy_margin) and fidelity floor are defensible for attaching heredity to a lineage? The build uses hypothetical values (10 copies, margin 1.0) and Eigen's error threshold Q >= 1/sigma as the fidelity floor, and SWEEPS the budget x fidelity space instead of asserting a point.
- Rule-set scope for the reaction universe: keep C1-C3 feedstock / C/H/O-only chemistry (as built: products capped at C4, 413 reactions) or include N/P-bearing reactions from the start, folding the og-res-N / og-res-P resource chemistries into the universe?
- Publication milestone: close the first paper at the pre-genetic arc (og-00 -> og-03) or extend through the early-genetic arc (og-04 protocells -> og-06 heredity)?
- Actually RUN the og-03 -> og-02 localization->ignition coupling (trap enrichment feeds the rTCA network as a real swept phase map), resolving the demoted phantom finding and giving og-03 its own Decision phase map?
- Reframe the campaign's headline thesis from 'cells could originate at alkaline vents' to the enrichment plan's 'quantitative MAP OF THE BOTTLENECKS between geology and cells' (dependencies, failure regimes, discriminating experiments)?
- Adopt the Full Modeling Plan architecture (emergence-not-presupposition + structural rewrites + canonical schemas + model substitution) as the campaign's build target, in the order — canonical schemas + geochemical_world first, then the electrochemical interface, then C-N-P/energetics as cross-cutting layers, then localization, boundary, growth/division rewrites, competing heredity, ecology, autonomy?
- Adopt the three-quantity standard (feasibility / robustness / discrimination) and evidence tiers as the reporting requirement for every stage — i.e. a stage's verdict may not advance until it carries a TransitionReport whose decision-discipline audit is clean — and re-label the fourteen earlier studies (every member study except the already- tier-labelled og-cal) with their tier language?
- Proto-energetics: replace og-05's population-mean internalization readout (mean active-pump fraction of the surviving population) with the per-individual autonomy internal / (internal + external) power that the energy_coupling face defines — i.e. re-point og-05's gates (reduced-gradient-dependence, final_internalization) at autonomy?
- Attach the MARHYS 4.0 hydrothermal-fluid compilation (PANGAEA) as a workspace dataset so the vent-calibration conditioning set becomes the ultramafic-vent population rather than Lost City alone — and which Hadean-ocean prior (DIC, pH, T) should replace the modern-seawater proxy the audit flags?
